# B7 · Du notebook à la production — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b7_production/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B7.Q1 à B7.Q11, B7.R1 à B7.R3, B7.1 à B7.4 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B7.9 | Du notebook au paquet : data.py, model.py, train.py, pyproject.toml et installation éditable | 📦 | ★★★ | 60 |
|  | B7.10 | Une interface en ligne de commande : argparse, logging et codes de sortie | 📦 | ★★ | 30 |
|  | B7.11 | Configuration : dataclass + YAML, surcharges en ligne de commande, validation | 📦 | ★★ | 30 |
|  | B7.12 | Prédire : même graine, puis num_workers = 4, puis GPU — résultats identiques ? | 🔮 | ★ | 10 |
|  | B7.13 | Mesurer le non-déterminisme : graines fixées ou non, algorithmes déterministes, cinq exécutions | 🔬 | ★★ | 30 |
|  | B7.14 | Suivi d'expériences maison : identifiant de run, configuration, métriques et artefacts en JSONL | 📦 | ★★ | 30 |
|  | B7.15 | Tests pytest pour le ML : formes, overfitting d'un mini-batch, prétraitement, schéma des données | 📦 | ★★★ | 45 |
| B | B7.16 | Le modèle qui marche dans le notebook et pas en production : cinq écarts à retrouver | 🐛 | ★★ | 30 |
|  | B7.17 | L'artefact du modèle : safetensors, configuration, normalisation, classes, version, empreinte des données | 📦 | ★★ | 30 |
|  | B7.18 | Une API de prédiction avec FastAPI (repli : http.server) : /health, /predict, validation des entrées | 📦 | ★★★ | 45 |
|  | B7.19 | Prédire la latence p95 de l'API, requête par requête puis par batches de 32 | 🔮 | ★ | 10 |
|  | B7.20 | Accélérer l'inférence sur CPU : batches, threads, inference_mode, bfloat16, quantification | 🔬 | ★★ | 30 |
|  | B7.21 | Exporter avec torch.export et vérifier l'équivalence numérique (ONNX en option) | 📦 | ★★ | 25 |
| C | B7.22 | Surveiller la dérive : PSI et test KS sur des images assombries ou bruitées, quand alerter ? | 📈 | ★★ | 25 |
|  | B7.23 | Intégration continue : un workflow GitHub Actions qui installe, teste et lance un entraînement fumée | 🛠️ | ★★ | 25 |
|  | B7.24 | Lire et compléter un Dockerfile : image de base, dépendances CPU, utilisateur non root, variables | 📦 | ★★ | 20 |
|  | B7.25 | Défi : un dépôt prêt pour la production, de git clone à une prédiction servie en moins de 10 min de CPU | 🏆 | ★★★★ | 180 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

## Objectifs et rappel express

- Restructurer un notebook en paquet Python installable, avec une ligne de commande, du logging et une configuration validée.
- Rendre un entraînement reproductible et traçable (graines, déterminisme, versions, empreinte des données, journal des runs), et rapporter un score sur plusieurs graines avec son intervalle.
- Écrire des tests utiles à un projet de ML et les automatiser dans une intégration continue.
- Empaqueter un modèle avec son prétraitement, le servir par une API HTTP et mesurer sa latence.
- Exporter et accélérer l'inférence sur CPU en vérifiant l'équivalence numérique ; détecter une dérive des données et décider d'une action.

**Rappel express.** Intervalle sur $n$ graines : $\bar a \pm t_{0{,}975;\,n-1}\,\frac{s}{\sqrt n}$ ($s$ divisé par $n - 1$). Percentile de `np.percentile` : position $h = \frac{q}{100}(n - 1)$ dans les mesures triées, interpolation linéaire. Débit $= \frac{\text{taille du batch}}{\text{latence du batch}}$ ; loi de Little $L = \lambda W$. Taille des poids $= N \times$ octets par paramètre (4, 2, 1). $\operatorname{PSI} = \sum_k (p_k - q_k)\ln\frac{p_k}{q_k} = D_{\text{KL}}(p\,\|\,q) + D_{\text{KL}}(q\,\|\,p)$ ; $D = \sup_x |F_{\text{réf}}(x) - F_{\text{prod}}(x)|$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B7.1 à B7.4)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres ensemble, entre guillemets, dans n'importe quel ordre (`"EF"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », l'estimation 🧮 B7.5, la réflexion (🗣️ B7.6, ⚖️ B7.7, 📄 B7.8) et l'entretien se corrigent avec `05_solutions.md`.

In [ ]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np

MB, MIB = 1e6, 2 ** 20                  # 1 Mo = 10**6 bytes, 1 Mio = 2**20 bytes (fiche, 207.8)

# quiz Q1: the cells x = 2 ; x = x * 10 ; print(x + 1), run in the order 1, 2, 3, 2, 3
Q1A = 2 * 10 * 10 + 1
# quiz Q3: basicConfig(level=WARNING), then 1 debug, 3 info, 1 warning and 1 error messages
LEVELS_Q3 = ["DEBUG"] + ["INFO"] * 3 + ["WARNING", "ERROR"]
Q3C = sum(level in ("WARNING", "ERROR", "CRITICAL") for level in LEVELS_Q3)
# quiz Q7: the ML Test Score is the minimum of the four section sums
SECTIONS_Q7 = {"data": 3.0, "model": 2.5, "infrastructure": 4.0, "monitoring": 1.0}
Q7E = min(SECTIONS_Q7.values())
# quiz Q9: 900 requests of 10 ms and 100 of 200 ms ; Little's law ; a throughput
LAT_Q9 = np.array([10.0] * 900 + [200.0] * 100)
Q9A, Q9B = float(LAT_Q9.mean()), float(np.percentile(LAT_Q9, 95))
Q9D = 60 * 0.150                         # L = lambda W, W in seconds
Q9E = 64 / 0.4
# recall R1: a JSONL log of 7 runs, 2 runs added (append, never rewrite)
R1D = 7 + 2

# ✏️ B7.1: ConvNeXt-Tiny (torchvision) and the VGG of the MP5 solution
N_B71, LIN_W_B71, LIN_B_B71 = 28_589_128, 26_646_528, 34_120
F32_B71 = 4 * N_B71 / MB
F32_MIB_B71 = 4 * N_B71 / MIB
BF16_B71 = 2 * N_B71 / MB
INT8_B71 = N_B71 / MB
DYNQ_B71 = (LIN_W_B71 + 4 * (N_B71 - LIN_W_B71)) / MB          # dense matrices in int8, everything else float32
RATIO_B71 = F32_B71 / DYNQ_B71
PEAK_B71 = 2 * F32_MIB_B71                                     # random weights + the copy read by torch.load
VGG_PARAMS_B71, VGG_LINEAR_B71, VGG_FILE_B71 = 72_954, 650, 296_464
VGG_BN_CHANNELS_B71 = (16, 16, 32, 32, 64, 64)
VGG_FLOAT_BUFFERS_B71 = 2 * sum(VGG_BN_CHANNELS_B71)            # running_mean and running_var
VGG_BYTES_B71 = 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71) + 8 * len(VGG_BN_CHANNELS_B71)
HEADER_B71 = VGG_FILE_B71 - VGG_BYTES_B71
LIN_SHARE_B71 = 100 * VGG_LINEAR_B71 / VGG_PARAMS_B71

# ✏️ B7.2: five seeds, Student t with n - 1 = 4 degrees of freedom (the table of 207.4)
A_B72 = np.array([64.2, 65.0, 63.6, 64.8, 64.4])
B_B72 = np.array([64.9, 65.3, 64.5, 65.6, 64.7])
T4_B72 = 2.776


def interval_b72(values, t=T4_B72, ddof=1):
    """Mean, standard deviation, standard error and half-width of the Student interval."""
    m, s = float(np.mean(values)), float(np.std(values, ddof=ddof))
    se = s / math.sqrt(len(values))
    return m, s, se, t * se


MEAN_A_B72, S_A_B72, SE_A_B72, HW_A_B72 = interval_b72(A_B72)
LOW_A_B72 = MEAN_A_B72 - HW_A_B72
MEAN_B_B72, _, _, HW_B_B72 = interval_b72(B_B72)
OVERLAP_B72 = bool(MEAN_A_B72 + HW_A_B72 >= MEAN_B_B72 - HW_B_B72)
MEAN_D_B72, S_D_B72, SE_D_B72, HW_D_B72 = interval_b72(B_B72 - A_B72)
ZERO_IN_D_B72 = bool(MEAN_D_B72 - HW_D_B72 <= 0 <= MEAN_D_B72 + HW_D_B72)

# ✏️ B7.3: twenty latencies of batch 1 (ms), np.percentile's default (method="linear")
LAT_B73 = np.array([21.3, 19.8, 20.4, 22.1, 19.6, 20.9, 20.1, 35.7, 20.6, 19.9,
                    21.0, 20.2, 23.4, 19.7, 20.5, 47.9, 20.8, 20.3, 21.6, 20.0])
MEAN_B73 = float(LAT_B73.mean())
P50_B73, P95_B73 = (float(v) for v in np.percentile(LAT_B73, [50, 95]))
ABOVE_B73 = int((LAT_B73 > MEAN_B73).sum())
P50_32_B73, P95_32_B73 = 118.0, 131.0                          # batch 32, given by the statement
THR1_B73 = 1000 / P50_B73
THR32_B73 = 32 * 1000 / P50_32_B73
RATIO_B73 = THR32_B73 / THR1_B73
PER_IMAGE_B73 = P50_32_B73 / 32
SLO_B73 = bool(P95_32_B73 <= 100)
PAGE_B73 = 1 - 0.95 ** 8

# ✏️ B7.4: PSI on four intervals (the quartiles of the reference), natural log, epsilon added to every share
Q_B74 = np.full(4, 0.25)
COUNTS_B74 = {1: [38, 52, 54, 56], 2: [96, 62, 30, 12], 3: [120, 56, 24, 0]}
P_B74 = {week: np.array(c) / 200 for week, c in COUNTS_B74.items()}


def psi_terms_b74(p, q=Q_B74, eps=0.0, log=np.log):
    """The four terms (p_k - q_k) log(p_k / q_k), eps added to every share of p and q (without renormalizing)."""
    p, q = np.asarray(p, float) + eps, np.asarray(q, float) + eps
    return (p - q) * log(p / q)


C1_W1_B74 = float(psi_terms_b74(P_B74[1])[0])
PSI_W1_B74 = float(psi_terms_b74(P_B74[1]).sum())
PSI_W2_B74 = float(psi_terms_b74(P_B74[2]).sum())
TOP_W2_B74 = "ABCD"[int(np.argmax(psi_terms_b74(P_B74[2])))]
KL_PQ_W2_B74 = float(np.sum(P_B74[2] * np.log(P_B74[2] / Q_B74)))
KL_QP_W2_B74 = float(np.sum(Q_B74 * np.log(Q_B74 / P_B74[2])))
PSI_W3_B74 = float(psi_terms_b74(P_B74[3], eps=1e-4).sum())
PSI_W3_EPS3_B74 = float(psi_terms_b74(P_B74[3], eps=1e-3).sum())
# a classic mistake for i): epsilon added to the production shares only (or to the empty share only: same at 3 decimals)
PSI_W3_EPS3_PART_B74 = float(((P_B74[3] + 1e-3 - Q_B74) * np.log((P_B74[3] + 1e-3) / Q_B74)).sum())

# The same answers from the libraries (no download): NumPy, SciPy, torchvision on the `meta` device, safetensors,
# PyYAML, logging, argparse, subprocess, DataLoader, torch.export, torch.load and scikit-learn
import io
import json
import logging
import subprocess
import sys
import tempfile
import warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import yaml
from scipy import stats

# quiz Q1: run the three cells in the order 1, 2, 3, 2, 3 and keep what the last one prints
cells, namespace, printed = {1: "x = 2", 2: "x = x * 10", 3: "shown = x + 1"}, {}, None
for k in (1, 2, 3, 2, 3):
    exec(cells[k], namespace)
assert namespace["shown"] == Q1A

# quiz Q3: a WARNING level lets through WARNING and ERROR only ; an exception swallowed by print ends with code 0
records = []
logger = logging.getLogger("wb_chB7_q3")
logger.propagate = False
handler = logging.Handler(level=logging.WARNING)
handler.emit = records.append
logger.addHandler(handler)
logger.setLevel(logging.DEBUG)
for level in LEVELS_Q3:
    logger.log(getattr(logging, level), "message")
logger.removeHandler(handler)
assert len(records) == Q3C
swallow = "try:\n    raise RuntimeError('epoch 2: loss is nan')\nexcept Exception as err:\n    print(err)\n"
assert subprocess.run([sys.executable, "-c", swallow], capture_output=True).returncode == 0
unknown = "import argparse; p = argparse.ArgumentParser(); p.add_argument('--lr'); p.parse_args(['--lerning-rate', '1'])"
assert subprocess.run([sys.executable, "-c", unknown], capture_output=True).returncode == 2

# quiz Q4: PyYAML 6.0 reads 5.0e-4 as a float, 5e-4 and "5.0e-4" as strings ; a dataclass does not check types
assert yaml.safe_load("lr: 5.0e-4")["lr"] == 0.0005
assert yaml.safe_load("lr: 5e-4")["lr"] == "5e-4" and yaml.safe_load('lr: "5.0e-4"')["lr"] == "5.0e-4"


@dataclass(frozen=True)
class FrozenOnly:
    lr: float = 0.05


assert FrozenOnly(lr="1e-3").lr == "1e-3"                       # accepted: no check of the annotation

# quiz Q5: float addition is not associative ; without a generator, an extra draw moves the DataLoader's order
assert ((0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3)) is False
import torch
from torch.utils.data import DataLoader


def first_order(extra_draw):
    torch.manual_seed(0)
    if extra_draw:
        torch.rand(1)
    return next(iter(DataLoader(list(range(10)), batch_size=10, shuffle=True))).tolist()


assert first_order(False) != first_order(True)

# quiz Q7, Q9 and recall R1
assert Q7E == 1.0 and Q9A == 29.0 and Q9B == 200.0 and math.isclose(Q9D, 9.0) and math.isclose(Q9E, 160.0)
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "runs.jsonl"
    for seed in list(range(7)) + [10, 11]:                     # 7 runs, then 2 runs again with other seeds
        with path.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"seed": seed}) + "\n")
    assert len(path.read_text(encoding="utf-8").splitlines()) == R1D
assert datetime.strptime("20261104-161731", "%Y%m%d-%H%M%S").strftime("%H:%M:%S") == "16:17:31"

# quiz Q8: load_state_dict refuses tensors of another shape ; safetensors metadata must be strings
narrow, wide = torch.nn.Linear(12, 3), torch.nn.Linear(16, 3)
try:
    wide.load_state_dict(narrow.state_dict())
except RuntimeError as err:
    assert "size mismatch" in str(err)
else:
    raise AssertionError("load_state_dict accepted tensors of another shape")
from safetensors.torch import save as save_safetensors

try:
    save_safetensors({"w": torch.zeros(2)}, metadata={"version": 1})
except TypeError:
    pass
else:
    raise AssertionError("safetensors accepted a metadata value that is not a string")

# quiz Q10: an export without dynamic_shapes refuses another batch size ; a data-dependent if does not export
small = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(12, 3)).eval()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    exported = torch.export.export(small, (torch.randn(8, 3, 2, 2),)).module()
    try:
        exported(torch.randn(3, 3, 2, 2))
    except Exception as err:                                    # « Guard failed: input.size()[0] == 8 »
        assert "8" in str(err)
    else:
        raise AssertionError("the exported model accepted a batch of 3")


class DataDependent(torch.nn.Module):
    def forward(self, x):
        if x.mean() > 0:
            return 2 * x
        return x - 1


import contextlib

torch_loggers = ("torch", "torch.fx.experimental.symbolic_shapes", "torch._dynamo", "torch.export")
saved_levels = {name: logging.getLogger(name).level for name in torch_loggers}
try:                                                            # torch.export prints the trace that failed
    for name in torch_loggers:
        logging.getLogger(name).setLevel(logging.CRITICAL)
    with contextlib.redirect_stderr(io.StringIO()), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        torch.export.export(DataDependent(), (torch.randn(4, 3),))
except Exception as err:
    assert "data-dependent" in str(err), str(err)[:200]
else:
    raise AssertionError("a data-dependent if was exported")
finally:
    for name, level in saved_levels.items():
        logging.getLogger(name).setLevel(level)

# recall R2: weights_only=True refuses an arbitrary Python object


class Trainer:
    pass


buffer = io.BytesIO()
torch.save({"trainer": Trainer()}, buffer)
buffer.seek(0)
try:
    torch.load(buffer, weights_only=True)
except Exception:
    pass
else:
    raise AssertionError("weights_only=True loaded a Python object")

# recall R3: the warning of scikit-learn when a model was written by another version
from sklearn.exceptions import InconsistentVersionWarning

assert issubclass(InconsistentVersionWarning, UserWarning)

# ✏️ B7.1: ConvNeXt-Tiny counted on the `meta` device (no memory, no download) ; the MP5 VGG and its safetensors
from torchvision.models import convnext_tiny

with torch.device("meta"):
    convnext = convnext_tiny(weights=None)
linear = [m for m in convnext.modules() if isinstance(m, torch.nn.Linear)]
assert sum(p.numel() for p in convnext.parameters()) == N_B71 and not list(convnext.buffers())
assert sum(m.weight.numel() for m in linear) == LIN_W_B71 and sum(m.bias.numel() for m in linear) == LIN_B_B71


def vgg_mp5(widths=(16, 32, 64)):
    """The VGG of the MP5 solution (cifarkit.models.vgg with downsample="pool" and batchnorm)."""
    layers, c = [], 3
    for i, w in enumerate(widths):
        for _ in range(2):
            layers += [torch.nn.Conv2d(c, w, 3, padding=1, bias=False), torch.nn.BatchNorm2d(w), torch.nn.ReLU()]
            c = w
        if i < len(widths) - 1:
            layers.append(torch.nn.MaxPool2d(2))
    layers += [torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten(), torch.nn.Linear(c, 10)]
    return torch.nn.Sequential(*layers)


vgg = vgg_mp5()
state = vgg.state_dict()
assert sum(p.numel() for p in vgg.parameters()) == VGG_PARAMS_B71
assert sum(t.numel() * t.element_size() for t in state.values()) == VGG_BYTES_B71
assert len(save_safetensors(state)) == VGG_FILE_B71
assert sum(p.numel() for p in vgg[-1].parameters()) == VGG_LINEAR_B71

# ✏️ B7.2: the Student intervals of SciPy (exact quantile 2.7764 instead of the 2.776 of the table)
assert math.isclose(stats.t.ppf(0.975, 4), T4_B72, abs_tol=5e-4)
ci_a = stats.ttest_1samp(A_B72, 0).confidence_interval()
ci_d = stats.ttest_rel(B_B72, A_B72).confidence_interval()
assert round(float(ci_a.low), 2) == round(LOW_A_B72, 2)
assert round(float(ci_d.high - ci_d.low) / 2, 2) == round(HW_D_B72, 2) and (float(ci_d.low) > 0) == (not ZERO_IN_D_B72)

# ✏️ B7.3: the other conventions of np.percentile, for the record
assert np.percentile(LAT_B73, 95, method="inverted_cdf") == 35.7 and np.percentile(LAT_B73, 50, method="inverted_cdf") == 20.5

# ✏️ B7.4: PSI = KL(p || q) + KL(q || p), with scipy.stats.entropy
for week in (1, 2):
    kl = stats.entropy(P_B74[week], Q_B74) + stats.entropy(Q_B74, P_B74[week])
    assert math.isclose(kl, float(psi_terms_b74(P_B74[week]).sum()))
assert math.isclose(stats.entropy(Q_B74, P_B74[2]), KL_QP_W2_B74)
with np.errstate(divide="ignore", invalid="ignore"):
    assert np.isinf(psi_terms_b74(P_B74[3]).sum())
print("✅ the paper answers agree with NumPy, SciPy, torchvision, safetensors, PyYAML, logging, subprocess and torch")

**B7.Q1 — Les pièges du notebook : état caché et ordre d'exécution**

In [ ]:
wb.record("B7.Q1a", Q1A, mistakes={"le noyau garde l'ordre où tu as exécuté les cellules, pas celui où elles sont écrites : refais la suite 1, 2, 3, 2, 3": 21,
          "compte les exécutions de la cellule 2 dans la suite 1, 2, 3, 2, 3": 2001,
          "relis la cellule 3 : elle affiche x + 1": 200})
wb.record("B7.Q1b", "A", mistakes={"« Redémarrer » vide le noyau : d'où viendrait encore scale ?": "B", "Jupyter n'exécute que les cellules du fichier : relis « Ce que cache un notebook » (§207.1)": "C"})
wb.record("B7.Q1c", False, mistakes={"un .ipynb range code, sorties et images en JSON : relis « Ce que cache un notebook » (§207.1)": True})
wb.record("B7.Q1d", "C", mistakes={"un module déjà importé est-il relu par le noyau ? Relis la fin de « pyproject.toml et installation éditable »": "A", "le lien éditable existe déjà : est-ce l'installation ou le noyau qui garde l'ancienne version ?": "B"})
wb.record("B7.Q1e", True, mistakes={"supprimer une cellule touche-t-il au noyau ? Relis « Ce que cache un notebook » (§207.1)": False})

**B7.Q2 — src/, tests/, configs/, scripts/ : où va quoi ?**

In [ ]:
wb.record("B7.Q2a", "A", mistakes={"un fichier de tests vérifie le code ; l'API, elle, importe cette fonction": "B", "configs/ ne contient que des réglages, pas du code": "C", "scripts/ range ce qu'on lance à la main ; cette fonction est importée par l'entraînement et l'API": "D"})
wb.record("B7.Q2b", "D", mistakes={"src/cifarprod/ ne contient que le code qu'importent l'entraînement et le service : ce script, on le lance": "A", "ce script ne vérifie pas le code, il prépare les données": "B", "configs/ ne contient que des réglages, pas du code": "C"})
wb.record("B7.Q2c", "C", mistakes={"un fichier YAML de réglages n'est pas du code importable": "A", "ce fichier décrit un entraînement ; il ne teste rien lui-même": "B", "scripts/ range du code à lancer ; un fichier YAML est un réglage": "D"})
wb.record("B7.Q2d", "B", mistakes={"src/cifarprod/ ne contient que le code qui sert en production": "A", "configs/ ne contient que des réglages": "C", "où pytest cherche-t-il les tests d'un dépôt ? Relis l'arborescence de 207.1": "D"})
wb.record("B7.Q2e", False, mistakes={"relis le paragraphe qui suit l'arborescence (§207.1) : les données et les sorties des runs vont-elles dans git ?": True})
wb.record("B7.Q2f", "B", mistakes={"relis l'avantage du layout src/ (§207.1) : le dossier courant suffit-il pour importer cifarprod ?": "A", "pytest installe-t-il quoi que ce soit ? Relis le layout src/ (§207.1)": "C"})
wb.record("B7.Q2g", True, mistakes={"relis [project.scripts] (§207.1) : que crée cette ligne à l'installation ?": False})

**B7.Q3 — print ou logging : niveaux, destinations et messages utiles**

In [ ]:
wb.record("B7.Q3a", "B", mistakes={"DEBUG sert au détail pour chercher un bug : l'écriture de l'artefact en est-elle un ?": "A", "WARNING signale l'inattendu : écrire l'artefact à la fin d'un run réussi l'est-il ?": "C", "ERROR dit qu'une opération a échoué : relis les niveaux (§207.2)": "D"})
wb.record("B7.Q3b", "C", mistakes={"un entraînement dix fois plus lent que prévu : un simple détail de débogage ?": "A", "est-ce une étape normale, ou quelque chose d'inattendu qui n'empêche pas de continuer ?": "B", "le programme continue : une opération a-t-elle échoué ?": "D"})
wb.record("B7.Q3c", Q3C, mistakes={"un niveau WARNING laisse-t-il passer les messages INFO ? Relis les niveaux (§207.2)": 5,
          "un message ERROR est plus grave qu'un WARNING : passe-t-il le niveau choisi ?": 1,
          "un niveau WARNING laisse-t-il passer DEBUG et INFO ?": 6})
wb.record("B7.Q3d", False, mistakes={"relis « Le logging plutôt que print » (§207.2) : où configure-t-on les handlers, et combien de fois ?": True})
wb.record("B7.Q3e", "C", mistakes={"quel fait vérifiable ce message donne-t-il ? Relis la fin de « Le logging plutôt que print »": "A", "un message utile donne des valeurs : lequel en donne ?": "B"})
wb.record("B7.Q3f", 0, mistakes={"le except affiche l'erreur, puis le script va au bout : comment finit-il ? Relis « Les codes de sortie » (§207.2)": 1,
          "2 est le code d'argparse devant une mauvaise option : relis « Les codes de sortie »": 2})

**B7.Q4 — Ce qui va dans la configuration, et ce qui ne doit jamais y aller**

In [ ]:
wb.record("B7.Q4a", "ACD", mistakes={"il manque au moins une lettre : relis 207.3 et « Variables d'environnement et secrets » (§207.9)": {'C', 'AD', 'A', 'AC', 'D', 'CD'}, "au moins une des lettres choisies n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)": {'BCD', 'AB', 'ABC', 'BD', 'ABD', 'B', 'BC'}})
wb.record("B7.Q4b", "A", mistakes={"relis le rappel outil sur PyYAML (§207.3) : que faut-il pour qu'un nombre en notation scientifique soit lu comme un float ?": "B", "des guillemets donnent-ils un nombre ou une chaîne en YAML ?": "C"})
wb.record("B7.Q4c", 2, mistakes={"relis main de la fiche (§207.2) : que rend-il quand load_config lève une ValueError ?": 0,
          "1 est le code d'une erreur générale : relis le except de main (§207.2)": 1})
wb.record("B7.Q4d", True, mistakes={"relis « Variables d'environnement et secrets » (§207.9) : que garde une image de chacune de ses couches ?": False})
wb.record("B7.Q4e", "C", mistakes={"un commit suivant efface-t-il le précédent de l'historique ?": "A", "un dépôt privé se clone, se partage et peut devenir public : relis 207.9": "B"})
wb.record("B7.Q4f", False, mistakes={"relis le commentaire de __post_init__ dans TrainConfig (§207.3) : une dataclass vérifie-t-elle les types ?": True})

**B7.Q5 — Même graine, résultats différents : les sources de non-déterminisme**

In [ ]:
wb.record("B7.Q5a", "C", mistakes={"l'initialisation de nn.Linear tire dans le générateur de PyTorch : torch.manual_seed le fixe-t-il ?": "A", "le dropout tire dans le générateur de PyTorch : torch.manual_seed le fixe-t-il ?": "B"})
wb.record("B7.Q5b", True, mistakes={"relis le paragraphe qui suit seed_everything (§207.4) : l'addition des flottants est-elle associative ?": False})
wb.record("B7.Q5c", "C", mistakes={"c'est le comportement sans warn_only : relis l'encadré 🕰️ de 207.4": "A", "PyTorch ne remplace pas une opération par une autre : relis l'encadré 🕰️ de 207.4": "B"})
wb.record("B7.Q5d", False, mistakes={"calcule les deux sommes en Python : l'addition des flottants est-elle associative ?": True})
wb.record("B7.Q5e", "B", mistakes={"sans generator, où le DataLoader tire-t-il son ordre ? Relis le paragraphe qui suit seed_everything (§207.4)": "A", "un tirage de plus n'est pas une erreur : que change-t-il au générateur global ?": "C"})

**B7.Q6 — requirements.txt, fichier de verrouillage, pyproject.toml : qui fige quoi ?**

In [ ]:
wb.record("B7.Q6a", "C", mistakes={"une borne >= dit-elle une version exacte ? Relis « pyproject.toml et installation éditable » (§207.1)": "A", "Colab installe ses propres versions : que dit une borne inférieure ?": "B"})
wb.record("B7.Q6b", "A", mistakes={"pyproject.toml donne des bornes : relis « Les versions figées et l'empreinte des données » (§207.4)": "B", "le cache de pip garde des fichiers téléchargés, pas ce qu'un projet a installé : relis « Les versions figées et l'empreinte des données » (§207.4)": "C"})
wb.record("B7.Q6c", False, mistakes={"relis « Les versions figées » (§207.4) : une bibliothèque qu'on publie garde-t-elle des versions exactes ou des bornes ?": True})
wb.record("B7.Q6d", "A", mistakes={"le run a tourné avec du code modifié : ce code est-il tout entier dans a1b2c3d ?": "B", "un commit enregistré reste valide : que manque-t-il au run ?": "C"})
wb.record("B7.Q6e", True, mistakes={"un fichier de verrouillage fige-t-il des versions exactes ou des bornes ? Relis « Les versions figées » (§207.4)": False})

**B7.Q7 — Que tester dans un projet de machine learning ?**

In [ ]:
wb.record("B7.Q7a", "B", mistakes={"la forme de la sortie dépend-elle de zero_grad ? Relis « Quatre familles de tests » (§207.6)": "A", "le schéma des données ne voit pas la boucle d'entraînement : relis « Quatre familles de tests »": "C"})
wb.record("B7.Q7b", "C", mistakes={"l'entraînement fumée vérifie que tout va au bout, pas le contenu des données": "A", "apprendre 16 images par cœur dit-il quelque chose du recouvrement des jeux ?": "B"})
wb.record("B7.Q7c", "A", mistakes={"relis la puce « bout en bout » (§207.6) : l'entraînement fumée juge-t-il l'accuracy ?": "B", "l'entraînement fumée lance un entraînement, pas l'API": "C"})
wb.record("B7.Q7d", False, mistakes={"relis la puce « bout en bout » (§207.6) : que vérifie l'entraînement fumée, et que ne juge-t-il pas ?": True})
wb.record("B7.Q7e", Q7E, decimals=1, mistakes={"relis la grille de Breck et coll. (§207.6) : le score résume-t-il les quatre sommes par leur moyenne ?": float(np.mean(list(SECTIONS_Q7.values()))),
          "relis la grille de Breck et coll. (§207.6) : le score résume-t-il les quatre sommes par leur total ?": float(sum(SECTIONS_Q7.values())),
          "relis la grille de Breck et coll. (§207.6) : le score résume-t-il les quatre sommes par la plus grande ?": 4.0})

**B7.Q8 — Artefact de modèle : les poids seuls ne suffisent pas**

In [ ]:
wb.record("B7.Q8a", "AB", mistakes={"il manque au moins une lettre : relis « L'artefact du modèle » (§207.7)": {'B', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis « L'artefact du modèle » (§207.7)": {'ABCD', 'ABC', 'ABD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)": {'C', 'AD', 'BCD', 'ACD', 'BD', 'AC', 'D', 'BC', 'CD'}})
wb.record("B7.Q8b", "B", mistakes={"load_state_dict vérifie-t-il les noms et les formes des tenseurs ? Relis §23.9.2 du ch. 23": "A", "PyTorch modifie-t-il les tenseurs qu'on lui donne ? Relis §23.9.2 du ch. 23": "C"})
wb.record("B7.Q8c", "C", mistakes={"relis le paragraphe qui suit le code de « L'artefact du modèle » : quels types prennent les métadonnées de safetensors ?": "A", "safetensors ignore-t-il ce qu'il ne sait pas écrire ? Relis « L'artefact du modèle » (§207.7)": "B"})
wb.record("B7.Q8d", True, mistakes={"relis « Les versions figées et l'empreinte des données » (§207.4) : que devient l'empreinte quand un seul pixel change ?": False})
wb.record("B7.Q8e", "C", mistakes={"la version change-t-elle le calcul ? Relis la traçabilité dans « L'artefact du modèle » (§207.7)": "A", "le format des pixels est une information à part de l'artefact : relis la liste de « L'artefact du modèle »": "B"})

**B7.Q9 — Latence p50, p95 et débit : ce que ressent l'utilisateur**

In [ ]:
wb.record("B7.Q9a", round(Q9A), mistakes={"relis la question : la moyenne pèse chaque requête, et 900 requêtes prennent 10 ms": 10,
          "c'est la moyenne des deux durées, sans tenir compte des effectifs": 105,
          "relis la question : on demande la moyenne": 200})
wb.record("B7.Q9b", round(Q9B), mistakes={"relis la question : on demande le p95, pas la moyenne": 29,
          "compte les requêtes de 10 ms : laissent-elles 95 % des mesures en dessous de cette valeur ?": 10,
          "c'est la moyenne des deux durées : le p95 se lit dans les mesures triées": 105})
wb.record("B7.Q9c", True, mistakes={"imagine 1 000 requêtes de 29 ms chacune : compare leur p95 à celui de l'énoncé, à moyenne égale": False})
wb.record("B7.Q9d", round(Q9D), mistakes={"convertis 150 ms en secondes avant de multiplier": 9000,
          "relis la loi de Little (§207.7) : L multiplie le débit par le temps de réponse": 400})
wb.record("B7.Q9e", round(Q9E), mistakes={"le débit divise la taille du batch par sa durée": 26})
wb.record("B7.Q9f", "A", mistakes={"une image d'un batch attend-elle les autres ? Relis la fin de « Latence, percentiles et débit » (§207.7)": "B", "relis la fin de « Latence, percentiles et débit » : que gagne-t-on à grouper, même sur CPU ?": "C"})

**B7.Q10 — torch.export, ONNX, TorchScript : lequel choisir aujourd'hui ?**

In [ ]:
wb.record("B7.Q10a", "B", mistakes={"relis l'encadré 🕰️ de 207.8 : que dit PyTorch 2.11 de torch.jit.script ?": "A", "un objet sauvé par pickle se recharge-t-il sans le code de sa classe ?": "C"})
wb.record("B7.Q10b", "A", mistakes={"torch.compile accélère un modèle dans un programme Python : produit-il un fichier ?": "B", "un state_dict se recharge-t-il sans PyTorch ni le code du modèle ?": "C"})
wb.record("B7.Q10c", False, mistakes={"relis l'encadré 🕰️ de 207.8 : que produit torch.compile ?": True})
wb.record("B7.Q10d", "B", mistakes={"relis le paragraphe sur dynamic_shapes (§207.8) : que devient la taille de batch d'un export sans dynamic_shapes ?": "A", "un modèle exporté complète-t-il une entrée ? Relis le paragraphe sur les gardes (§207.8)": "C"})
wb.record("B7.Q10e", False, mistakes={"relis l'encadré 🕰️ de 207.8 : sur quoi s'appuie torch.onnx.export par défaut depuis la 2.9 ?": True})
wb.record("B7.Q10f", "A", mistakes={"un graphe figé peut-il contenir un if Python sur une valeur des données ? Relis le début de « torch.export et ONNX »": "B", "garder la seule branche de l'exemple, c'était le défaut de torch.jit.trace : relis le début de « torch.export et ONNX »": "C"})

**B7.Q11 — Écart entraînement-production et dérive des données : vrai ou faux**

In [ ]:
wb.record("B7.Q11a", True, mistakes={"relis « Pourquoi un modèle se dégrade sans changer » (§207.10)": False})
wb.record("B7.Q11b", False, mistakes={"relis la définition du training-serving skew (§207.7) : d'où vient l'écart ?": True})
wb.record("B7.Q11c", True, mistakes={"relis la fin de « Le test de Kolmogorov-Smirnov » (§207.10) : que devient une p-valeur avec beaucoup de données ?": False})
wb.record("B7.Q11d", True, mistakes={"relis ce qu'on surveille sur les sorties (§207.10) : faut-il attendre les labels ?": False})
wb.record("B7.Q11e", "A", mistakes={"les deux chaînes de préparation ont-elles changé, ou les images reçues ? Relis 207.7 et 207.10": "B", "la relation entre une image et sa classe a-t-elle changé ? Relis « Pourquoi un modèle se dégrade sans changer » (§207.10)": "C"})
wb.record("B7.Q11f", False, mistakes={"relis la fin de « Les seuils » (§207.10) : quand fixe-t-on les bornes des intervalles ?": True})

**B7.R1 — Ce qu'une ligne de journal d'expériences doit contenir**

In [ ]:
wb.record("B7.R1a", "ABCD", mistakes={"il manque au moins une lettre : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.4": {'C', 'AD', 'BCD', 'AB', 'A', 'ABC', 'ACD', 'CD', 'BD', 'ABD', 'AC', 'B', 'BC', 'D'}})
wb.record("B7.R1b", "A", mistakes={"relis la fin du paragraphe qui suit log_run (§207.5) : que fausse l'effacement d'un run raté ?": "B", "remplacer la ligne, c'est réécrire le passé : relis la docstring de log_run (§207.5)": "C"})
wb.record("B7.R1c", False, mistakes={"relis le paragraphe qui suit log_run (§207.5) : un champ ajouté gêne-t-il les anciennes lignes ?": True})
wb.record("B7.R1d", R1D, mistakes={"un journal JSONL s'écrit en ajoutant des lignes : relance-t-on un run en réécrivant la sienne ?": 7,
          "le journal garde-t-il les sept premiers runs ? Relis la docstring de log_run (§207.5)": 2})
wb.record("B7.R1e", "C", mistakes={"relis new_run_id (§207.5) : de quoi l'identifiant est-il fait ?": "A", "relis new_run_id (§207.5) : le commit y figure-t-il ?": "B"})

**B7.R2 — Checkpoint complet, state_dict et weights_only**

In [ ]:
wb.record("B7.R2a", "ABD", mistakes={"il manque au moins une lettre : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)": {'AD', 'AB', 'A', 'BD', 'B', 'D'}, "au moins une des lettres choisies n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)": {'C', 'BCD', 'ABC', 'ACD', 'AC', 'BC', 'CD'}})
wb.record("B7.R2b", "B", mistakes={"l'optimiseur sert-il à prédire ? Relis §23.9.2 du ch. 23 (« Les poids seuls »)": "A", "l'état de l'optimiseur contient-il les poids ? Relis §23.9.2 du ch. 23": "C"})
wb.record("B7.R2c", True, mistakes={"relis l'encadré 🕰️ du ch. 23 sur les formats de sauvegarde : que refuse weights_only=True ?": False})
wb.record("B7.R2d", False, mistakes={"relis §23.9.2 du ch. 23 : que faut-il faire avant load_state_dict ?": True})
wb.record("B7.R2e", "A", mistakes={"relis §23.9.1 du ch. 23 : que perd-on sans l'état de l'optimiseur ?": "B", "le load_state_dict du modèle se soucie-t-il de l'optimiseur ?": "C"})

**B7.R3 — Sauvegarder un Pipeline et figer les versions**

In [ ]:
wb.record("B7.R3a", "A", mistakes={"relis la première règle de « sauvegarder un modèle et ses versions » (ch. 15)": "B", "des coefficients sans leur prétraitement servent-ils à prédire ? Relis la première règle du ch. 15": "C"})
wb.record("B7.R3b", False, mistakes={"relis la deuxième règle du ch. 15 : que garantit la documentation d'une version à l'autre ?": True})
wb.record("B7.R3c", "B", mistakes={"un FutureWarning annonce un changement d'API : relis la deuxième règle du ch. 15": "A", "relis la deuxième règle du ch. 15 : scikit-learn prévient-il, depuis la 1.3 ?": "C"})
wb.record("B7.R3d", True, mistakes={"relis la troisième règle du ch. 15 : que fait le chargement d'un pickle ?": False})
wb.record("B7.R3e", "ABC", mistakes={"il manque au moins une lettre : relis la deuxième règle du ch. 15 : que note-t-on à côté du fichier, et sous quelle forme ?": {'C', 'AB', 'A', 'AC', 'B', 'BC'}, "au moins une des lettres choisies n'a pas sa place : relis la deuxième règle du ch. 15 : que note-t-on à côté du fichier, et sous quelle forme ?": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis la deuxième règle du ch. 15 : que note-t-on à côté du fichier, et sous quelle forme ?": {'AD', 'ACD', 'BD', 'ABD', 'D', 'BCD', 'CD'}})
wb.record("B7.R3f", "C", mistakes={"skops.io vérifie les types avant de charger, mais en Python : relis la troisième règle du ch. 15": "A", "joblib demande Python et scikit-learn : relis la troisième règle du ch. 15": "B"})

**Ex B7.1 — Taille d'un artefact : float32, bfloat16, int8, et la mémoire au chargement**

In [ ]:
wb.record("B7.1a", F32_B71, decimals=1, mistakes={"un float32 occupe 4 octets : tu as compté 4 bits": N_B71 * 4 / 8 / MB,
          "relis la convention de l'énoncé : 1 Mo = 10⁶ octets": F32_MIB_B71,
          "un float32 occupe 4 octets par paramètre, pas 1": N_B71 / MB,
          "tu as compté en bits : un octet en fait 8": 32 * N_B71 / MB})
wb.record("B7.1b", F32_MIB_B71, decimals=1, mistakes={"relis la question : en Mio, avec 1 Mio = 2²⁰ octets": F32_B71,
          "1 Mio = 2²⁰ = 1 048 576 octets : relis la convention de l'énoncé": 4 * N_B71 / 1_024_000})
wb.record("B7.1c", BF16_B71, decimals=1, mistakes={"un bfloat16 occupe 2 octets par paramètre, pas 4": F32_B71,
          "relis l'unité : en Mo, 1 Mo = 10⁶ octets": 2 * N_B71 / MIB,
          "le bfloat16 a 16 bits : combien d'octets ?": N_B71 / MB})
wb.record("B7.1d", INT8_B71, decimals=1, mistakes={"un int8 occupe 1 octet, pas 2": BF16_B71,
          "relis l'unité : en Mo, 1 Mo = 10⁶ octets": N_B71 / MIB})
wb.record("B7.1e", DYNQ_B71, decimals=1, mistakes={"seules les matrices des couches denses passent en int8 : relis l'énoncé": INT8_B71,
          "les biais des couches denses restent en float32 : relis l'énoncé": (LIN_W_B71 + LIN_B_B71 + 4 * (N_B71 - LIN_W_B71 - LIN_B_B71)) / MB,
          "n'oublie pas les paramètres hors des matrices denses, en float32": LIN_W_B71 / MB,
          "relis l'unité : en Mo, 1 Mo = 10⁶ octets": DYNQ_B71 * MB / MIB})
wb.record("B7.1f", RATIO_B71, decimals=1, mistakes={"relis la question : e) ne passe pas tous les paramètres en int8": 4.0,
          "vérifie le sens du rapport : a) divisé par e)": 1 / RATIO_B71})
wb.record("B7.1g", PEAK_B71, decimals=1, mistakes={"relis l'énoncé : combien de copies des poids sont en mémoire au moment de load_state_dict ?": F32_MIB_B71,
          "relis l'unité : en Mio": 2 * F32_B71,
          "relis l'énoncé : deux copies des poids, pas trois": 3 * F32_MIB_B71})
wb.record("B7.1h", VGG_BYTES_B71, mistakes={"le state_dict contient aussi les buffers des batchnorms": 4 * VGG_PARAMS_B71,
          "relis l'énoncé : num_batches_tracked est un int64, de 8 octets": 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71 + 6),
          "n'oublie pas les six compteurs num_batches_tracked": 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71),
          "relis l'énoncé : chaque batchnorm a deux buffers d'une valeur par canal": 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71 // 2) + 48})
wb.record("B7.1i", HEADER_B71, mistakes={"les buffers des batchnorms sont aussi dans le fichier : retire-les avec les paramètres": VGG_FILE_B71 - 4 * VGG_PARAMS_B71,
          "relis la question : on demande l'en-tête, ce qui reste une fois les tenseurs retirés": VGG_BYTES_B71})
wb.record("B7.1j", LIN_SHARE_B71, decimals=1, mistakes={"relis la question : en pourcentage": VGG_LINEAR_B71 / VGG_PARAMS_B71,
          "vérifie le sens du rapport : les paramètres de la couche dense divisés par tous les paramètres": 100 * VGG_PARAMS_B71 / VGG_LINEAR_B71})

**Ex B7.2 — Cinq graines, un intervalle : la dispersion d'une accuracy rapportée**

In [ ]:
wb.record("B7.2a", MEAN_A_B72, decimals=1, mistakes={"divise la somme par le nombre de graines": float(A_B72.sum() / 4)})
wb.record("B7.2b", S_A_B72, decimals=3, mistakes={"l'écart-type corrigé divise par n − 1 : relis le rappel maths de 207.4": float(np.std(A_B72)),
          "c'est la variance : prends sa racine": float(np.var(A_B72, ddof=1)),
          "c'est la somme des carrés des écarts : divise-la, puis prends la racine": float(np.sum((A_B72 - A_B72.mean()) ** 2))})
wb.record("B7.2c", SE_A_B72, decimals=3, mistakes={"l'erreur type divise s par √n, avec n = 5 graines": S_A_B72 / 2,
          "l'erreur type divise s par √n, pas par n": S_A_B72 / 5,
          "pars de l'écart-type corrigé, divisé par n − 1 : relis b)": interval_b72(A_B72, ddof=0)[2]})
wb.record("B7.2d", HW_A_B72, decimals=2, mistakes={"avec cinq valeurs, on prend le quantile de Student : relis le rappel maths de 207.4": 1.96 * SE_A_B72,
          "le quantile de Student se lit à n − 1 = 4 degrés de liberté : relis la table de 207.4": 2.571 * SE_A_B72,
          "la demi-largeur multiplie t par l'erreur type, pas par l'écart-type": T4_B72 * S_A_B72,
          "pars de l'écart-type corrigé, divisé par n − 1": interval_b72(A_B72, ddof=0)[3]})
wb.record("B7.2e", LOW_A_B72, decimals=2, mistakes={"la borne basse retire la demi-largeur à la moyenne": MEAN_A_B72 + HW_A_B72,
          "avec cinq valeurs, on prend le quantile de Student : relis le rappel maths de 207.4": MEAN_A_B72 - 1.96 * SE_A_B72,
          "le quantile de Student se lit à n − 1 = 4 degrés de liberté : relis la table de 207.4": MEAN_A_B72 - 2.571 * SE_A_B72})
wb.record("B7.2f", OVERLAP_B72, mistakes={"calcule l'intervalle de B comme celui de A, puis compare leurs bornes": not OVERLAP_B72})
wb.record("B7.2g", MEAN_D_B72, decimals=1, mistakes={"vérifie le sens de la différence : B − A": -MEAN_D_B72})
wb.record("B7.2h", HW_D_B72, decimals=2, mistakes={"avec cinq différences, on prend le quantile de Student : relis le rappel maths de 207.4": 1.96 * SE_D_B72,
          "la demi-largeur multiplie t par l'erreur type des différences, pas par leur écart-type": T4_B72 * S_D_B72,
          "le quantile de Student se lit à n − 1 = 4 degrés de liberté : relis la table de 207.4": 2.571 * SE_D_B72})
wb.record("B7.2i", ZERO_IN_D_B72, mistakes={"calcule les deux bornes de l'intervalle de la différence avec g) et h)": not ZERO_IN_D_B72})

**Ex B7.3 — Latence et débit : p50 et p95 sur vingt mesures, batch 1 contre batch 32**

In [ ]:
wb.record("B7.3a", MEAN_B73, decimals=2, mistakes={"relis la question : la moyenne des vingt mesures": P50_B73,
          "divise la somme par le nombre de mesures, 20": float(LAT_B73.sum() / 19)})
wb.record("B7.3b", P50_B73, decimals=2, mistakes={"c'est la plus petite valeur qui laisse la moitié des mesures en dessous (method=\"inverted_cdf\") : relis la convention de l'énoncé": 20.5,
          "relis la position : h = 0,5 × (n − 1), avec n = 20 mesures": 20.6,
          "relis la question : on demande le p50, la médiane": MEAN_B73})
wb.record("B7.3c", P95_B73, decimals=2, mistakes={"c'est la convention « la plus petite valeur qui laisse 95 % des mesures en dessous » (inverted_cdf) : relis celle de l'énoncé": 35.7,
          "c'est la moyenne des deux plus grandes mesures : relis la position h de l'énoncé": 41.8,
          "c'est la plus grande mesure : relis la position h de l'énoncé": 47.9,
          "c'est une autre convention (position q × (n + 1)) : relis celle de l'énoncé": float(np.percentile(LAT_B73, 95, method="weibull"))})
wb.record("B7.3d", ABOVE_B73, mistakes={"compte les mesures au-dessus de la moyenne, pas de la médiane": 10,
          "relis la question : au-dessus de la moyenne": 20 - ABOVE_B73})
wb.record("B7.3e", round(THR1_B73), mistakes={"relis la question : le débit se calcule ici avec le p50, pas la moyenne": round(1000 / MEAN_B73),
          "relis la question : le débit se calcule ici avec le p50, pas le p95": round(1000 / P95_B73)},
          fractional="le débit se donne ici en images par seconde, arrondi à l'entier")
wb.record("B7.3f", round(THR32_B73), mistakes={"un batch de 32 images : le débit divise sa taille par sa durée": round(1000 / P50_32_B73),
          "relis la question : le débit se calcule avec le p50 du batch, pas son p95": round(32_000 / P95_32_B73)},
          fractional="le débit se donne ici en images par seconde, arrondi à l'entier")
wb.record("B7.3g", RATIO_B73, decimals=1, mistakes={"vérifie le sens du rapport : f) divisé par e)": 1 / RATIO_B73,
          "tu as divisé les débits arrondis à l'entier : garde les valeurs non arrondies de e) et f)": round(THR32_B73) / round(THR1_B73),
          "le débit ne croît pas comme la taille du batch : calcule-le avec les mesures": 32.0})
wb.record("B7.3h", PER_IMAGE_B73, decimals=2, mistakes={"relis la question : le p50 du batch divisé par 32": P50_32_B73,
          "relis la question : on part du p50 du batch, pas de son p95": P95_32_B73 / 32})
wb.record("B7.3i", SLO_B73, mistakes={"chaque image attend la fin de son batch : compare le p95 d'un batch de 32 à l'objectif": not SLO_B73})
wb.record("B7.3j", PAGE_B73, decimals=2, mistakes={"relis la question : au moins une des huit dépasse le p95": 0.95 ** 8,
          "les probabilités ne s'additionnent pas ici : passe par l'événement contraire": 8 * 0.05,
          "relis la question : huit prédictions, pas une seule": 0.05})

**Ex B7.4 — Indice de stabilité de population (PSI) à la main sur quatre intervalles**

In [ ]:
wb.record("B7.4a", C1_W1_B74, decimals=4, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[1], log=np.log10)[0]),
          "vérifie le signe : p − q et ln(p/q) ont toujours le même signe": -C1_W1_B74,
          "relis la formule : la contribution multiplie ln(p/q) par p − q": float(np.log(P_B74[1][0] / 0.25))})
wb.record("B7.4b", PSI_W1_B74, decimals=3, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[1], log=np.log10).sum()),
          "il manque une partie du PSI : relis le lien avec la divergence KL (§207.10)": float(np.sum(P_B74[1] * np.log(P_B74[1] / Q_B74)))})
wb.record("B7.4c", PSI_W2_B74, decimals=3, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[2], log=np.log10).sum()),
          "il manque une partie du PSI : relis le lien avec la divergence KL (§207.10)": KL_PQ_W2_B74,
          "relis la question : on demande le PSI, la somme des quatre contributions (p − q) ln(p/q)": KL_QP_W2_B74})
wb.record("B7.4d", "D", mistakes={"calcule la contribution de I₁ en semaine 2 et compare-la aux trois autres": "A", "calcule la contribution de I₂ en semaine 2 et compare-la aux trois autres": "B", "calcule la contribution de I₃ en semaine 2 et compare-la aux trois autres": "C"})
wb.record("B7.4e", KL_QP_W2_B74, decimals=3, mistakes={"relis la question : q en premier, Σ q ln(q/p)": KL_PQ_W2_B74,
          "relis la question : on demande une seule des deux divergences": PSI_W2_B74,
          "la divergence KL utilise le logarithme naturel ln": float(np.sum(Q_B74 * np.log10(Q_B74 / P_B74[2])))})
wb.record("B7.4f", "B", mistakes={"compare le PSI de la semaine 2 au second seuil, 0,25": "A", "compare le PSI de la semaine 1 au premier seuil, 0,1": "C"})
wb.record("B7.4g", True, mistakes={"calcule ln(0 / 0,25) : que vaut-il pour NumPy, et que devient son produit par (0 − 0,25) ?": False})
wb.record("B7.4h", PSI_W3_B74, decimals=2, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[3], eps=1e-4, log=np.log10).sum()),
          "relis l'énoncé : ε vaut 10⁻⁴ ici": PSI_W3_EPS3_B74})
wb.record("B7.4i", PSI_W3_EPS3_B74, decimals=3, mistakes={"relis la question : ε = 10⁻³": PSI_W3_B74,
          "relis l'énoncé : ε s'ajoute à toutes les parts, celles de la production et celles de la référence": PSI_W3_EPS3_PART_B74,
          "le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[3], eps=1e-3, log=np.log10).sum())})

## Partie A · Le projet : du notebook au paquet, ligne de commande, configuration, reproductibilité, suivi et tests

Au MP5, `cifarkit` rangeait déjà ton code dans un dossier que le notebook importait. Ici, ce code devient un **projet** : un paquet installable (B7.9), piloté par une ligne de commande (B7.10) et une configuration validée (B7.11), dont tu mesures la reproductibilité (B7.12, B7.13), qui journalise ses runs (B7.14) et que des tests protègent (B7.15). Le modèle est volontairement petit (le CNN de 207.6, 2 000 images, une epoch : quelques secondes) : on juge ici la mécanique du projet, pas l'accuracy.

La cellule suivante crée ton dossier de projet, `PROJECT` (`mon_travail/b7_production/cifar-prod/` ; le notebook de solutions travaille dans un dossier temporaire), y écrit `data/cifar10_2000.npz` (200 images par classe pour l'entraînement, 50 pour la validation, un sous-ensemble figé de CIFAR-10, `uint8` `(N, 32, 32, 3)`, avec les tableaux `x_train`, `y_train`, `x_val`, `y_val`) et charge `wbp` (`wb.prod`, l'infrastructure du chapitre : écrire un fichier, lancer une commande, installer dans un environnement neuf). Toutes les commandes partent du dossier du projet, avec des chemins absolus.

In [ ]:
# Tools of part A: the folder of your project, the data of the chapter, the commands; nothing here is your package
import atexit
import contextlib
import io
import json
import os
import re
import shutil
import sys
import tempfile
import time
import warnings
from pathlib import Path

import numpy as np
import torch
import yaml
from wb import prod as wbp
from wb.nbtools import StopCheck, checked, fr, refuses, run_pytest_group, same, verdict, written

PROJECT = wbp.project_dir("solution")             # the folder of your project
DATA_FILE = wbp.write_subset(PROJECT / "data")  # the 2 000 + 500 images of the chapter (.npz, 7.7 MB, not for git)
SUBSET = wbp.cifar_subset()                     # the same arrays, for the checks
CLASSES = wbp.CLASSES
GITIGNORE = "data/\nruns/\n__pycache__/\n*.egg-info/\n.pytest_cache/\n.wb_written.json\n"   # the last: what the notebook wrote
DEFAULTS = {"seed": 0, "epochs": 1, "lr": 0.05, "batch_size": 128, "n_train": 2000, "arch": "cnn", "widths": [16, 32],
            "data": "data/cifar10_2000.npz", "deterministic": True, "num_workers": 0}   # TrainConfig().to_dict()


SCRATCH_ROOT = Path(tempfile.mkdtemp(prefix="wb_b7_session_"))       # removed when the kernel stops
atexit.register(shutil.rmtree, SCRATCH_ROOT, ignore_errors=True)
LATER_FILES = {"tracking": ("TRACKING_PY", "B7.14"), "artifact": ("ARTIFACT_PY", "B7.17")}


def scratch(prefix):
    """A new temporary folder (outside the project, removed when the kernel stops): the checks write their test files
    there."""
    return Path(tempfile.mkdtemp(prefix=f"{prefix}_", dir=SCRATCH_ROOT))


def later_files_pending(ex_id):
    """⏳ (True) when the cli.py of your project imports a module of a later exercise (tracking.py, artifact.py) that
    the project does not have yet: on a new project, « Tout exécuter » reaches this cell before the cell that
    defines that module."""
    cli_file = PROJECT / "src" / "cifarprod" / "cli.py"
    text = cli_file.read_text(encoding="utf-8", errors="replace") if cli_file.is_file() else ""
    absent = [f"{module}.py (la cellule de {name}, Ex {ex})" for module, (name, ex) in LATER_FILES.items()
              if re.search(rf"^\s*(from\s+(\.|cifarprod\.)\s*{module}\b|from\s+(\.|cifarprod)\s+import\s.*\b{module}\b"
                           rf"|import\s+cifarprod\.{module}\b)", text, re.M)
              and not (PROJECT / "src" / "cifarprod" / f"{module}.py").is_file()]
    if absent:
        print(f"⏳ Ex {ex_id} : ton cli.py importe {' et '.join(absent)}, que ton projet n'a pas encore (sur un projet "
              "neuf, « Tout exécuter » arrive ici avant cette cellule) : exécute-la, puis réexécute celle-ci.")
    return bool(absent)


def run_line(ex_id, result, ok, success, failure):
    """One verdict about a command of your package: what was run, then the end of stderr when it went wrong (⏳ when
    the command stopped on a NotImplementedError of your skeleton)."""
    shown = result.command().replace(f"python -m {wbp.PACKAGE}", "cifarprod")
    pending = re.findall(r"NotImplementedError: (.+)", result.stderr)
    if not ok and pending:
        print(f"⏳ Ex {ex_id} : {shown} → pas encore fait ({pending[-1].strip()}).")
        return False
    if result.timed_out:
        ok, failure = False, f"délai dépassé ({result.seconds:.0f} s) : {failure}"
    if result.leftover:
        ok, failure = False, (f"{failure} ; et elle laisse un processus en vie après sa fin (arrêté ici) : arrête ce "
                              "qu'elle lance dans un finally")
    seen = " ; ".join(part for part in [
        f"stderr : {result.tail(2)[:400]}" if result.stderr.strip() else "rien sur stderr",
        f"stdout : {result.stdout.strip()[-300:]!r}" if result.stdout.strip() else ""] if part)
    verdict(ex_id, ok, f"{shown} → {success}", f"{shown} → {failure} (code {result.returncode} ; {seen})")
    return ok


print(wbp.project_line(PROJECT, "solution"))
print(f"ℹ️ données : data/{DATA_FILE.name} ({len(SUBSET['x_train'])} images d'entraînement, {len(SUBSET['x_val'])} de "
      f"validation, 200 et 50 par classe ; empreinte {wbp.SUBSET_FINGERPRINT[:12]}…)")

### Ex B7.9 — Du notebook au paquet : data.py, model.py, train.py, pyproject.toml et installation éditable 📦 ★★★ ⏱️ 60 min
**Objectif :** transformer le code d'un notebook en paquet Python installable, avec un layout `src/`.  
**Prérequis :** MP5 (`cifarkit`) · ch. 23 · fiche §207.1 (« Une structure de dépôt », « pyproject.toml et installation éditable »), §207.4 (`seed_everything`, `array_fingerprint`), §207.6 · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

> **Mode d'emploi du paquet** : chaque fichier de ton paquet est une chaîne de la cellule à compléter (`DATA_PY = r'''...'''`) ; la cellule de vérification l'**écrit** dans ton projet (`mon_travail/b7_production/cifar-prod/`, à côté de ce notebook), puis l'importe ou le lance dans un autre processus. Pour corriger un fichier, corrige sa chaîne et réexécute les deux cellules. Si tu préfères un éditeur, modifie le fichier sur le disque : la cellule le voit et ne l'écrase plus (elle le dit).

**Le contrat.** Ton projet doit avoir cette forme (le `.gitignore` et le fichier de données sont fournis) :

```text
cifar-prod/
├── pyproject.toml        .gitignore        data/cifar10_2000.npz
└── src/cifarprod/
    ├── __init__.py       (__version__ = "0.1.0", fourni)
    ├── data.py           model.py           train.py
```

- `pyproject.toml` : le backend de construction de la fiche (`setuptools`, déjà installé : un autre, comme `hatchling`, n'est pas dans l'environnement du workbook) ; le paquet `cifarprod`, version `0.1.0`, Python ≥ 3.11 ; les dépendances avec des bornes basses (`numpy`, `torch`, `pyyaml`, `safetensors`) ; deux extras, `dev` (`pytest` ; `httpx` et `requests`, pour les tests de l'API de B7.18 et la démo de B7.25 : `TestClient` exige `httpx`) et `serve` (`fastapi`, `uvicorn`, `pydantic`) ; le point d'entrée `cifarprod = "cifarprod.cli:main"` (le fichier `cli.py` viendra en B7.10) ; et, pour B7.15, une table `[tool.pytest.ini_options]` avec `testpaths = ["tests"]` et le marqueur `slow` déclaré dans `markers`.
- `data.py` : `CLASSES` (les 10 noms, dans l'ordre des sorties) ; `load_npz(path, part="train")` rend `(X, y)`, `X` `uint8` `(N, 32, 32, 3)` et `y` `int64` ; elle **vérifie le schéma** : `ValueError` pour des images qui ne sont pas `uint8` `(N, 32, 32, 3)`, des labels qui ne sont pas $N$ entiers de 0 à 9, ou une autre partie que `"train"` et `"val"` ; `FileNotFoundError` pour un fichier absent ; `fingerprint(*arrays)`, l'empreinte de 207.4 (le type, la forme, puis les octets de chaque tableau) ; `channel_stats(images)` rend `(mean, std)`, trois floats Python chacun, la moyenne et l'écart-type (divisé par $n$) de chaque canal dans $[0, 1]$ ; `preprocess(images, mean, std)` rend un tenseur `float32` `(N, 3, 32, 32)`, chaque canal valant $(\text{pixel}/255 - \text{mean}_c)/\text{std}_c$, accepte aussi **une** image `(32, 32, 3)` (rendue `(1, 3, 32, 32)`) et refuse un tableau qui n'est pas `uint8` (`ValueError`) ; `make_loader(X, y, mean, std, batch_size=128, shuffle=False, seed=0, num_workers=0)` rend un `DataLoader` de mini-batches `(preprocess(X), y)`, `float32` et `int64`, dont l'ordre mélangé vient d'un `torch.Generator().manual_seed(seed)` **à lui** (il ne dépend pas du générateur global) ; `synthetic(n, seed=0)` rend $n$ images `uint8` tirées par `np.random.default_rng(seed).integers(0, 256, (n, 32, 32, 3), dtype=np.uint8)` et les labels `np.arange(n) % 10` (pour les tests et l'intégration continue, qui n'ont pas CIFAR-10).
- `model.py` : `build_model(arch="cnn", widths=(12, 24), n_classes=10)` rend un `nn.Sequential` des couches de la docstring, **dans cet ordre** (`"cnn"`, le CNN de la fiche et de `cifarkit.small_cnn` ; `"vgg"`, celui de `cifarkit.vgg` avec batchnorm et max pooling) : les noms de ses poids (`0.weight`, `1.weight`…) sont alors ceux de `cifarkit`, et le modèle livré du MP5 s'y recharge ; `ValueError` pour une autre architecture ou des largeurs vides ; `n_parameters(model)`.
- `train.py` : `seed_everything(seed, deterministic=True)` (207.4) ; `evaluate(model, loader)` rend `(loss moyenne, accuracy)` en mode `eval()`, sans graphe, puis rend au modèle son mode d'avant ; `train(X, y, X_val, y_val, *, arch="cnn", widths=(16, 32), epochs=1, lr=0.05, batch_size=128, seed=0, deterministic=True, num_workers=0)` appelle d'abord `seed_everything(seed, deterministic)`, apprend la normalisation sur `X` **seulement**, entraîne avec SGD (`momentum=0.9`) et la cross-entropy sur les mini-batches de `make_loader(..., shuffle=True, seed=seed)`, et rend `{"model": le modèle en mode eval(), "mean", "std", "history": {"train_loss", "val_loss", "val_acc"} (un float par epoch), "seconds"}`. **Aucun `print`** : une ligne INFO par epoch et une ligne DEBUG par mini-batch avec `log = logging.getLogger(__name__)` (207.2), que B7.10 affichera.

La cellule de vérification écrit tes fichiers, importe ton paquet **neuf** depuis `src/` et compare chaque fonction : `load_npz` aux tableaux du fichier et quatre fichiers mal formés, `fingerprint` à `hashlib`, `channel_stats` (sur 2 000 images, puis sur 4, où un écart-type divisé par $n - 1$ se voit) et `preprocess` à NumPy, `make_loader` avec et sans mélange (le générateur global changé entre deux appels), `build_model` à l'architecture du contrat (mêmes tenseurs et mêmes formes, quatre réglages), puis lance `train` deux fois avec la même graine (mêmes poids) et une fois avec une autre, deux fois avec un learning rate presque nul (l'initialisation doit dépendre de la graine), un petit VGG (sa batchnorm doit voir chaque mini-batch en mode `train()`), `evaluate` sur un modèle en mode `train()` (qu'il doit lui rendre), et l'entraînement fumée des réglages par défaut sur les 2 000 images (moins d'une minute, une accuracy de validation au-dessus du hasard). Enfin, elle fait `pip install -e .` de ton projet dans un **environnement virtuel neuf** (qui voit les bibliothèques du notebook, mais pas ton dossier `src/`) et vérifie, depuis un autre dossier, que `import cifarprod` lit ton dossier `src/` (une installation éditable, pas une copie), la version, le point d'entrée, et que ton paquet n'importe pas `wb` (l'outillage du workbook n'est pas une de ses dépendances). Ton environnement à toi n'est jamais modifié : pour avoir la commande `cifarprod` dans un terminal, lance toi-même `pip install -e .` dans le dossier du projet.

Dans tes notes : pourquoi le layout `src/` empêche-t-il `import cifarprod` depuis la racine du projet tant que le paquet n'est pas installé, et qu'est-ce que cela protège ? Pourquoi `train` apprend-il la normalisation sur `X` seulement, et que deviendrait l'évaluation si on la calculait sur l'entraînement et la validation ensemble (ch. 12) ? Combien de secondes prend l'entraînement fumée, et que juge-t-il (et que ne juge-t-il pas) ?

In [ ]:
# Ex B7.9: the files of the package, as strings (the reference package of the chapter, solutions/b7_cifarprod)

# pyproject.toml
PYPROJECT_TOML = r'''[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "cifarprod"
version = "0.1.0"
description = "A CIFAR-10 classifier, from the notebook to production (bonus chapter B7 of the workbook)."
requires-python = ">=3.11"
dependencies = ["numpy>=2.0", "torch>=2.5", "pyyaml>=6.0", "safetensors>=0.4"]   # loose bounds: what the code needs

[project.optional-dependencies]
serve = ["fastapi>=0.110", "uvicorn>=0.30", "pydantic>=2.6"]
dev = ["pytest>=8.0", "httpx>=0.27", "requests>=2.31"]   # httpx: TestClient of FastAPI; requests: the tests of the API and the demo

[project.scripts]
cifarprod = "cifarprod.cli:main"

[tool.pytest.ini_options]
testpaths = ["tests"]
markers = ["slow: a long test (an end-to-end training); skipped with -m 'not slow'"]
'''

# src/cifarprod/__init__.py
INIT_PY = r'''"""cifarprod: a CIFAR-10 classifier as an installable package (data, model, training, configuration, command line,
run journal), from the notebook of the MP5 to production (bonus chapter B7)."""

__version__ = "0.1.0"
'''

# src/cifarprod/data.py
DATA_PY = r'''"""Data of cifarprod: the CIFAR-10 subset file, its schema, its fingerprint, the normalization and the loaders.

    X, y = load_npz("data/cifar10_2000.npz", "train")     # uint8 (N, 32, 32, 3), int64 labels 0-9
    mean, std = channel_stats(X)                           # learned on the training images only
    x = preprocess(X[:8], mean, std)                       # float32 tensor (8, 3, 32, 32)

The images are uint8 RGB arrays (N, 32, 32, 3), as CIFAR-10 is stored; the model receives float32 tensors (N, 3, 32,
32), each channel scaled to [0, 1] then normalized with the mean and the standard deviation of the training images.
One function, preprocess, does it for the training AND for the service.
"""

from __future__ import annotations

import hashlib
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

CLASSES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
PARTS = ("train", "val")


def check_images(X: np.ndarray, y: np.ndarray) -> None:
    """The schema of the data: X uint8 of shape (N, 32, 32, 3), y integers of shape (N,) in 0..9. ValueError if not."""
    if not isinstance(X, np.ndarray) or X.dtype != np.uint8:
        raise ValueError(f"expected uint8 images, got {getattr(X, 'dtype', type(X).__name__)}")
    if X.ndim != 4 or X.shape[1:] != (32, 32, 3):
        raise ValueError(f"expected images of shape (N, 32, 32, 3), got {X.shape}")
    y = np.asarray(y)
    if y.ndim != 1 or len(y) != len(X):
        raise ValueError(f"expected {len(X)} labels of shape ({len(X)},), got shape {y.shape}")
    if not np.issubdtype(y.dtype, np.integer):
        raise ValueError(f"expected integer labels, got {y.dtype}")
    if len(y) and (y.min() < 0 or y.max() >= len(CLASSES)):
        raise ValueError(f"expected labels in 0..{len(CLASSES) - 1}, got {int(y.min())}..{int(y.max())}")


def load_npz(path: str | Path, part: str = "train") -> tuple[np.ndarray, np.ndarray]:
    """(X, y) of one part ("train" or "val") of a .npz file with the arrays x_train, y_train, x_val, y_val: X uint8
    (N, 32, 32, 3), y int64 (N,). The schema is checked (check_images): ValueError for a wrong dtype, shape or label,
    or for another part; FileNotFoundError if the file does not exist."""
    if part not in PARTS:
        raise ValueError(f"part must be one of {PARTS}, got {part!r}")
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"no data file {path}")
    with np.load(path) as archive:
        if f"x_{part}" not in archive or f"y_{part}" not in archive:
            raise ValueError(f"{path} has no arrays x_{part} and y_{part}")
        X, y = archive[f"x_{part}"], archive[f"y_{part}"]
    check_images(X, y)
    return np.ascontiguousarray(X), y.astype(np.int64)


def fingerprint(*arrays: np.ndarray) -> str:
    """SHA-256 of the dtype, the shape and the bytes of each array (in this order): another value if one pixel, one
    label or the dtype changes."""
    h = hashlib.sha256()
    for a in arrays:
        a = np.ascontiguousarray(a)
        h.update(f"{a.dtype}{a.shape}".encode())
        h.update(a.tobytes())
    return h.hexdigest()


def channel_stats(images: np.ndarray) -> tuple[tuple[float, float, float], tuple[float, float, float]]:
    """(mean, std): three Python floats each, the mean and the standard deviation (divided by n) of the pixels of each
    channel on the scale [0, 1], over all the uint8 images (N, H, W, 3) given (the training images only)."""
    x = np.asarray(images, dtype=np.float64) / 255.0
    if x.ndim != 4 or x.shape[-1] != 3:
        raise ValueError(f"expected images of shape (N, H, W, 3), got {x.shape}")
    mean, std = x.mean(axis=(0, 1, 2)), x.std(axis=(0, 1, 2))
    return tuple(float(m) for m in mean), tuple(float(s) for s in std)


def preprocess(images: np.ndarray, mean, std) -> torch.Tensor:
    """The input of the model: uint8 RGB images (N, 32, 32, 3), or one image (32, 32, 3), as a float32 tensor (N, 3,
    32, 32) (N = 1 for one image), each channel c equal to (pixel / 255 - mean[c]) / std[c]. ValueError for an array
    that is not uint8 (a float image would not be scaled the same way) or not 32 x 32 x 3."""
    images = np.asarray(images)
    if images.dtype != np.uint8:
        raise ValueError(f"preprocess expects uint8 images (0..255), got {images.dtype}")
    if images.ndim == 3:
        images = images[None]
    if images.ndim != 4 or images.shape[1:] != (32, 32, 3):
        raise ValueError(f"expected images of shape (N, 32, 32, 3) or (32, 32, 3), got {images.shape}")
    x = torch.from_numpy(np.ascontiguousarray(images)).permute(0, 3, 1, 2).to(torch.float32) / 255.0
    m = torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)
    s = torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1)
    return ((x - m) / s).contiguous()


def make_loader(X: np.ndarray, y: np.ndarray, mean, std, batch_size: int = 128, shuffle: bool = False,
                seed: int = 0, num_workers: int = 0) -> DataLoader:
    """A DataLoader of (preprocess(X), y) mini-batches: float32 (B, 3, 32, 32) and int64 (B,). When shuffle, the order
    comes from its own torch.Generator().manual_seed(seed), so that it depends on nothing drawn before."""
    dataset = TensorDataset(preprocess(X, mean, std), torch.as_tensor(np.asarray(y), dtype=torch.int64))
    generator = torch.Generator().manual_seed(int(seed)) if shuffle else None
    return DataLoader(dataset, batch_size=int(batch_size), shuffle=shuffle, generator=generator,
                      num_workers=int(num_workers))


def synthetic(n: int, seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """(X, y): n random uint8 images (n, 32, 32, 3) drawn by np.random.default_rng(seed), and the labels i % 10, for
    the tests and the continuous integration, which have no CIFAR-10."""
    rng = np.random.default_rng(seed)
    X = rng.integers(0, 256, size=(n, 32, 32, 3), dtype=np.uint8)
    return X, np.arange(n, dtype=np.int64) % len(CLASSES)
'''

# src/cifarprod/model.py
MODEL_PY = r'''"""The models of cifarprod: the CNNs of the MP5 (cifarkit), rebuilt from a few settings, so that an artifact can
rebuild them.

    model = build_model("vgg", widths=(16, 32))       # float32 (N, 3, 32, 32) -> logits (N, 10)
    model = build_model(widths=(12, 24))              # the defaults: the CNN of the fiche (18 322 parameters)

- "cnn": for each width w, Conv2d(c, w, 3, padding=1), ReLU, MaxPool2d(2); then Flatten and Linear (no global pooling);
- "vgg": for each width w, two Conv2d(c, w, 3, padding=1, bias=False), each followed by BatchNorm2d(w) and ReLU; a
  MaxPool2d(2) after every stage but the last; then AdaptiveAvgPool2d(1), Flatten, Linear(widths[-1], n_classes).
"""

from __future__ import annotations

import torch.nn as nn

ARCHS = ("cnn", "vgg")


def build_model(arch: str = "cnn", widths=(12, 24), n_classes: int = 10) -> nn.Sequential:
    """The network ``arch`` ("cnn" or "vgg") with one stage per width; ValueError for another arch or empty widths."""
    widths = tuple(int(w) for w in widths)
    if arch not in ARCHS:
        raise ValueError(f"unknown arch {arch!r}: expected one of {ARCHS}")
    if not widths or min(widths) < 1:
        raise ValueError(f"widths must be positive whole numbers, got {widths}")
    layers, c = [], 3
    if arch == "cnn":
        side = 32
        for w in widths:
            layers += [nn.Conv2d(c, w, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)]
            c, side = w, side // 2
        return nn.Sequential(*layers, nn.Flatten(), nn.Linear(c * side * side, n_classes))
    for i, w in enumerate(widths):
        for _ in range(2):
            layers += [nn.Conv2d(c, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU()]
            c = w
        if i < len(widths) - 1:
            layers.append(nn.MaxPool2d(2))
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, n_classes)]
    return nn.Sequential(*layers)


def n_parameters(model: nn.Module) -> int:
    """The number of trainable parameters of a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
'''

# src/cifarprod/train.py
TRAIN_PY = r'''"""The training of cifarprod: the seeds, one evaluation, and train(), which returns the model with its normalization.

    result = train(X, y, X_val, y_val, arch="cnn", widths=(16, 32), epochs=1, lr=0.05, batch_size=128, seed=0)
    result["model"], result["mean"], result["std"], result["history"]["val_acc"]

SGD with momentum 0.9, the cross-entropy, mini-batches in the order of the generator of seed_everything; one INFO line
per epoch, one DEBUG line per mini-batch (logger "cifarprod.train"). Nothing is printed.
"""

from __future__ import annotations

import logging
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .data import channel_stats, make_loader
from .model import build_model

log = logging.getLogger(__name__)


def seed_everything(seed: int, deterministic: bool = True) -> torch.Generator:
    """Seed Python, NumPy and PyTorch, switch the deterministic algorithms on or off, and return a generator seeded
    with ``seed`` for the DataLoader."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(bool(deterministic))
    return torch.Generator().manual_seed(seed)


def evaluate(model: nn.Module, loader) -> tuple[float, float]:
    """(mean cross-entropy, accuracy) of the model on all the examples of the loader, in evaluation mode and without a
    graph; the mode of the model is restored afterwards."""
    was_training = model.training
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.inference_mode():
        for x, t in loader:
            out = model(x)
            total_loss += F.cross_entropy(out, t, reduction="sum").item()
            correct += int((out.argmax(dim=1) == t).sum())
            n += len(t)
    model.train(was_training)
    return total_loss / max(n, 1), correct / max(n, 1)


def train(X: np.ndarray, y: np.ndarray, X_val: np.ndarray, y_val: np.ndarray, *, arch: str = "cnn",
          widths=(16, 32), epochs: int = 1, lr: float = 0.05, batch_size: int = 128, seed: int = 0,
          deterministic: bool = True, num_workers: int = 0) -> dict:
    """Train ``build_model(arch, widths)`` on the uint8 images X (N, 32, 32, 3) and their labels y.

    The normalization (mean, std) is learned on X only (channel_stats). Returns {"model": the trained model in
    evaluation mode, "mean", "std", "history": {"train_loss", "val_loss", "val_acc"} (one value per epoch),
    "seconds"}."""
    start = time.perf_counter()
    seed_everything(seed, deterministic)               # the initialization of the weights
    mean, std = channel_stats(X)
    model = build_model(arch, widths)
    # the order of the mini-batches: a generator of its own, seeded with the same seed
    loader = make_loader(X, y, mean, std, batch_size, shuffle=True, seed=seed, num_workers=num_workers)
    val_loader = make_loader(X_val, y_val, mean, std, batch_size=500)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    log.info("training %s %s on %d images, %d epoch(s)", arch, list(widths), len(X), epochs)
    for epoch in range(1, epochs + 1):
        model.train()
        total, n = 0.0, 0
        for step, (x, t) in enumerate(loader, start=1):
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x), t)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(t)
            n += len(t)
            log.debug("epoch %d batch %d/%d loss %.4f", epoch, step, len(loader), loss.item())
        val_loss, val_acc = evaluate(model, val_loader)
        history["train_loss"].append(total / n)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        log.info("epoch %d/%d train_loss %.4f val_loss %.4f val_acc %.4f", epoch, epochs, total / n, val_loss, val_acc)
    model.eval()
    return {"model": model, "mean": mean, "std": std, "history": history, "seconds": time.perf_counter() - start}
'''


FILES_9 = {"pyproject.toml": PYPROJECT_TOML, ".gitignore": GITIGNORE, "src/cifarprod/__init__.py": INIT_PY,
           "src/cifarprod/data.py": DATA_PY, "src/cifarprod/model.py": MODEL_PY, "src/cifarprod/train.py": TRAIN_PY}
ok_9 = {}
with checked("B7.9"):
    wbp.write_files(PROJECT, FILES_9)
    missing_9 = [rel for rel, text in FILES_9.items() if text is ...]
    print("ℹ️ fichiers écrits dans le projet : " + ", ".join(r for r, t in FILES_9.items() if t is not ...)
          + (f" ; pas encore écrits : {', '.join(missing_9)}" if missing_9 else ""))
    pkg_9 = wbp.import_package(PROJECT)
    ok_9["import"] = getattr(pkg_9, "__version__", None) == "0.1.0"
    verdict("B7.9", ok_9["import"], "import cifarprod depuis src/ : version 0.1.0.",
            f"cifarprod.__version__ vaut {getattr(pkg_9, '__version__', None)!r}, attendu '0.1.0'.")

with checked("B7.9"):                                    # data.py
    data_9 = wbp.import_module(PROJECT, "data")
    X_9, y_9 = data_9.load_npz(DATA_FILE, "train")
    Xv_9, yv_9 = data_9.load_npz(DATA_FILE, "val")
    got_9 = [(np.asarray(a).dtype, np.asarray(a).shape) for a in (X_9, y_9, Xv_9, yv_9)]
    ok_9["load"] = (isinstance(X_9, np.ndarray) and X_9.dtype == np.uint8 and np.asarray(y_9).dtype == np.int64
                    and np.array_equal(X_9, SUBSET["x_train"]) and np.array_equal(y_9, SUBSET["y_train"])
                    and np.array_equal(Xv_9, SUBSET["x_val"]) and np.array_equal(yv_9, SUBSET["y_val"]))
    verdict("B7.9", ok_9["load"], "load_npz : les 2 000 images d'entraînement et les 500 de validation, uint8 et int64.",
            f"load_npz ne rend pas les tableaux du fichier en uint8 et int64 : reçu (dtype, forme) {got_9}.")
    bad_9 = scratch("bad")
    zeros_9 = np.zeros((4, 32, 32, 3), np.uint8)
    for name_9, (xb_9, yb_9) in {"des images float32": (zeros_9.astype(np.float32), np.arange(4)),
                                 "des images 32 × 32 × 1": (zeros_9[..., :1], np.arange(4)),
                                 "un label 10": (zeros_9, np.array([0, 1, 2, 10])),
                                 "3 labels pour 4 images": (zeros_9, np.arange(3))}.items():
        np.savez(bad_9 / "bad.npz", x_train=xb_9, y_train=yb_9, x_val=xb_9, y_val=yb_9)
        ok_9[name_9] = refuses("B7.9", f"load_npz d'un fichier avec {name_9}",
                               lambda: data_9.load_npz(bad_9 / "bad.npz", "train"))
    ok_9["part"] = refuses("B7.9", 'load_npz(..., "test")', lambda: data_9.load_npz(DATA_FILE, "test"))
    ok_9["absent"] = refuses("B7.9", "load_npz d'un fichier absent", lambda: data_9.load_npz(bad_9 / "absent.npz"),
                             error=FileNotFoundError)
    fp_9 = data_9.fingerprint(SUBSET["x_train"][:5], SUBSET["y_train"][:5])
    ok_9["fingerprint"] = fp_9 == wbp.array_fingerprint(SUBSET["x_train"][:5], SUBSET["y_train"][:5])
    verdict("B7.9", ok_9["fingerprint"], f"fingerprint de 5 images et de leurs labels : {str(fp_9)[:12]}…, comme "
            "hashlib sur le type, la forme et les octets.", f"fingerprint de 5 images et de leurs labels : "
            f"{str(fp_9)[:16]!r}…, attendu un autre SHA-256 (le type, la forme, puis les octets de chaque tableau).")
    mean_9, std_9 = data_9.channel_stats(SUBSET["x_train"])
    want_9 = (SUBSET["x_train"] / 255.0).mean(axis=(0, 1, 2)), (SUBSET["x_train"] / 255.0).std(axis=(0, 1, 2))
    ok_9["stats"] = (len(mean_9) == len(std_9) == 3 and all(type(v) is float for v in [*mean_9, *std_9])
                     and np.allclose(mean_9, want_9[0], atol=1e-10) and np.allclose(std_9, want_9[1], atol=1e-10))
    verdict("B7.9", ok_9["stats"], f"channel_stats : moyennes {np.round(mean_9, 4).tolist()}, écarts-types "
            f"{np.round(std_9, 4).tolist()} (trois floats Python chacun).",
            f"channel_stats rend {mean_9!r}, {std_9!r} : attendu les moyennes {np.round(want_9[0], 6).tolist()} et "
            f"les écarts-types {np.round(want_9[1], 6).tolist()}, trois floats Python chacun, des pixels de chaque canal "
            "dans [0, 1] (calcule en float64 : en float32, la moyenne de 2 millions de pixels s'écarte de 3e-4).")
    few_9 = SUBSET["x_val"][:4]
    std4_9 = np.asarray(data_9.channel_stats(few_9)[1], dtype=np.float64)
    want4_9 = (few_9 / 255.0).std(axis=(0, 1, 2))
    ok_9["ddof"] = std4_9.shape == (3,) and np.allclose(std4_9, want4_9, rtol=0, atol=1e-6)
    verdict("B7.9", ok_9["ddof"], "channel_stats de 4 images : l'écart-type divisé par n.",
            f"channel_stats de 4 images : écarts-types {np.round(std4_9, 6).tolist()}, attendu "
            f"{np.round(want4_9, 6).tolist()} (divisé par n ; np.std(..., ddof=1) et torch.std, par défaut, divisent "
            "par n - 1).")
    four_9 = SUBSET["x_val"][:4]
    want_x_9 = ((four_9 / 255.0 - np.array(want_9[0])) / np.array(want_9[1])).transpose(0, 3, 1, 2)
    x_9 = data_9.preprocess(four_9.copy(), tuple(want_9[0]), tuple(want_9[1]))
    ok_9["preprocess"] = (isinstance(x_9, torch.Tensor) and x_9.dtype == torch.float32
                          and same("B7.9", "preprocess de 4 images de validation", x_9, want_x_9, rtol=0, atol=1e-5))
    if not (isinstance(x_9, torch.Tensor) and x_9.dtype == torch.float32):
        verdict("B7.9", False, "", f"preprocess doit rendre un tenseur float32, reçu {type(x_9).__name__} "
                f"{getattr(x_9, 'dtype', '')}.")
    one_9 = data_9.preprocess(four_9[0], tuple(want_9[0]), tuple(want_9[1]))
    ok_9["one"] = tuple(getattr(one_9, "shape", ())) == (1, 3, 32, 32)
    verdict("B7.9", ok_9["one"], "preprocess d'une seule image (32, 32, 3) : (1, 3, 32, 32).",
            f"preprocess d'une seule image (32, 32, 3) : attendu (1, 3, 32, 32), reçu {tuple(getattr(one_9, 'shape', ()))}.")
    ok_9["float"] = refuses("B7.9", "preprocess d'images en float", lambda: data_9.preprocess(four_9 / 255.0, *want_9))

    def order_9(seed, global_seed):
        torch.manual_seed(global_seed)                  # the order must not depend on the global generator
        loader = data_9.make_loader(SUBSET["x_val"][:64], np.arange(64), *want_9, batch_size=16, shuffle=True, seed=seed)
        return np.concatenate([np.asarray(t) for _, t in loader])

    with wbp.torch_state_kept():
        plain_9 = data_9.make_loader(SUBSET["x_val"][:64], SUBSET["y_val"][:64], *want_9, batch_size=16)
        xb_9, tb_9 = next(iter(plain_9))
        a_9, b_9, c_9 = order_9(0, 1), order_9(0, 2), order_9(1, 1)
    ok_9["loader"] = (tuple(xb_9.shape) == (16, 3, 32, 32) and xb_9.dtype == torch.float32 and tb_9.dtype == torch.int64
                      and np.allclose(xb_9.numpy(), data_9.preprocess(SUBSET["x_val"][:16], *want_9).numpy())
                      and np.array_equal(tb_9.numpy(), SUBSET["y_val"][:16]))
    verdict("B7.9", ok_9["loader"], "make_loader sans mélange : les 16 premières images prétraitées et leurs labels, "
            "float32 et int64.", f"make_loader sans mélange : attendu le batch (16, 3, 32, 32) float32 des 16 premières "
            f"images prétraitées et leurs labels int64, reçu {tuple(xb_9.shape)} {xb_9.dtype}, {tb_9.dtype}.")
    ok_9["order"] = sorted(a_9) == list(range(64)) and np.array_equal(a_9, b_9) and not np.array_equal(a_9, c_9)
    verdict("B7.9", ok_9["order"], "make_loader mélangé : le même ordre pour la même graine, quel que soit le "
            "générateur global ; un autre pour une autre graine.",
            "make_loader mélangé : attendu le même ordre des 64 images pour seed=0 après torch.manual_seed(1) ou (2), "
            "et un autre pour seed=1 ; " + ("l'ordre suit le générateur global." if not np.array_equal(a_9, b_9)
                                           else "seed ne change pas l'ordre." if np.array_equal(a_9, c_9)
                                           else "des images manquent ou sont répétées."))
    Xs_9, ys_9 = data_9.synthetic(12, seed=3)
    ok_9["synthetic"] = (np.asarray(Xs_9).dtype == np.uint8 and np.asarray(Xs_9).shape == (12, 32, 32, 3)
                         and np.array_equal(Xs_9, np.random.default_rng(3).integers(0, 256, (12, 32, 32, 3), dtype=np.uint8))
                         and np.array_equal(ys_9, np.arange(12) % 10))
    verdict("B7.9", ok_9["synthetic"], "synthetic(12, seed=3) : 12 images uint8 de default_rng(3) et les labels i % 10.",
            "synthetic(12, seed=3) : attendu np.random.default_rng(3).integers(0, 256, (12, 32, 32, 3), "
            "dtype=np.uint8) et les labels np.arange(12) % 10.")

with checked("B7.9"):                                    # model.py
    model_9 = wbp.import_module(PROJECT, "model")
    for arch_9, widths_9 in (("cnn", (12, 24)), ("cnn", (8, 16, 32)), ("vgg", (16, 32, 64)), ("vgg", (8,))):
        mine_9 = model_9.build_model(arch_9, widths_9)
        want_m_9 = wbp.build_reference(arch_9, widths_9)
        shapes_9 = {k: tuple(v.shape) for k, v in mine_9.state_dict().items()}
        ref_shapes_9 = {k: tuple(v.shape) for k, v in want_m_9.state_dict().items()}
        out_9 = mine_9.eval()(torch.zeros(5, 3, 32, 32))
        ok_9[arch_9 + str(widths_9)] = shapes_9 == ref_shapes_9 and tuple(out_9.shape) == (5, 10)
        extra_9 = sorted(set(shapes_9) - set(ref_shapes_9))
        lacking_9 = sorted(set(ref_shapes_9) - set(shapes_9))
        other_9 = sorted(k for k in set(shapes_9) & set(ref_shapes_9) if shapes_9[k] != ref_shapes_9[k])
        diff_9 = "; ".join(part for part in [f"en trop : {extra_9[:4]}" if extra_9 else "",
                                             f"manquants : {lacking_9[:4]}" if lacking_9 else "",
                                             f"d'autres formes : {other_9[:4]}" if other_9 else ""] if part)
        verdict("B7.9", ok_9[arch_9 + str(widths_9)],
                f"build_model({arch_9!r}, {widths_9}) : {model_9.n_parameters(mine_9):_} paramètres, les mêmes couches "
                "que l'architecture du contrat, sortie (5, 10).".replace("_", " "),
                f"build_model({arch_9!r}, {widths_9}) : les noms et les formes des tenseurs du state_dict diffèrent de "
                f"ceux de l'architecture du contrat ({diff_9 or 'mêmes tenseurs'}) ; sortie {tuple(out_9.shape)}, "
                "attendu (5, 10).")
    n_9 = model_9.n_parameters(wbp.build_reference("cnn", (12, 24)))
    ok_9["n_parameters"] = n_9 == sum(p.numel() for p in wbp.build_reference("cnn", (12, 24)).parameters())
    verdict("B7.9", ok_9["n_parameters"], "n_parameters compte les paramètres entraînables.",
            f"n_parameters du CNN (12, 24) de la fiche : reçu {n_9}, attendu le nombre de valeurs de ses paramètres.")
    ok_9["default"] = ({k: tuple(v.shape) for k, v in model_9.build_model().state_dict().items()}
                       == {k: tuple(v.shape) for k, v in wbp.build_reference("cnn", (12, 24)).state_dict().items()})
    verdict("B7.9", ok_9["default"], "build_model() : le CNN (12, 24) de la fiche.",
            "build_model() sans argument doit construire le CNN (12, 24) de la fiche (arch=\"cnn\", widths=(12, 24)).")
    ok_9["arch"] = refuses("B7.9", 'build_model("resnet")', lambda: model_9.build_model("resnet"))

TRAINED_9 = None
with checked("B7.9"):                                    # train.py, in this notebook
    train_9 = wbp.import_module(PROJECT, "train")
    small_9 = dict(arch="cnn", widths=(8, 16), epochs=2, lr=0.05, batch_size=64, seed=0)
    printed_9 = io.StringIO()
    with wbp.torch_state_kept(), contextlib.redirect_stdout(printed_9):
        r1_9 = train_9.train(SUBSET["x_train"][:256], SUBSET["y_train"][:256], SUBSET["x_val"][:100],
                             SUBSET["y_val"][:100], **small_9)
        r2_9 = train_9.train(SUBSET["x_train"][:256], SUBSET["y_train"][:256], SUBSET["x_val"][:100],
                             SUBSET["y_val"][:100], **{**small_9, "seed": 1})
        r3_9 = train_9.train(SUBSET["x_train"][:256], SUBSET["y_train"][:256], SUBSET["x_val"][:100],
                             SUBSET["y_val"][:100], **small_9)
    keys_9 = {"model", "mean", "std", "history", "seconds"}
    if not (isinstance(r1_9, dict) and keys_9 <= set(r1_9)):
        verdict("B7.9", False, "", f"train doit rendre un dictionnaire avec les clés {sorted(keys_9)}, reçu "
                f"{type(r1_9).__name__} {sorted(r1_9) if isinstance(r1_9, dict) else ''}.")
        raise StopCheck
    h_9 = r1_9["history"]
    ok_9["history"] = (isinstance(h_9, dict) and all(len(h_9.get(k, [])) == 2 for k in ("train_loss", "val_loss", "val_acc"))
                       and all(type(v) is float for k in ("train_loss", "val_loss", "val_acc") for v in h_9[k])
                       and all(0 <= v <= 1 for v in h_9["val_acc"]))
    verdict("B7.9", ok_9["history"], "train sur 256 images, 2 epochs : un historique de deux floats par courbe.",
            f"train sur 256 images, 2 epochs : attendu history = {{train_loss, val_loss, val_acc}}, deux floats "
            f"chacune (val_acc dans [0, 1]), reçu {h_9!r:.200}.")
    stats_256_9 = (SUBSET["x_train"][:256] / 255.0).mean(axis=(0, 1, 2))
    ok_9["mean"] = np.allclose(r1_9["mean"], stats_256_9, atol=1e-10)
    verdict("B7.9", ok_9["mean"], "la normalisation est apprise sur les images d'entraînement reçues.",
            ("(channel_stats est déjà faux, ci-dessus) " if not ok_9.get("stats", True) else "")
            + "result['mean'] doit être channel_stats des images d'entraînement reçues par train (ni la validation, "
            "ni les statistiques d'ImageNet).")
    ok_9["eval"] = not r1_9["model"].training
    verdict("B7.9", ok_9["eval"], "le modèle rendu est en mode eval().", "le modèle rendu est encore en mode "
            "entraînement : relis la docstring de train.")
    ok_9["seed"] = wbp.same_weights(r1_9["model"].state_dict(), r3_9["model"].state_dict()) and \
        not wbp.same_weights(r1_9["model"].state_dict(), r2_9["model"].state_dict())
    verdict("B7.9", ok_9["seed"], "deux appels avec seed=0 donnent les mêmes poids, seed=1 d'autres.",
            "deux appels de train avec seed=0 doivent donner exactement les mêmes poids, et seed=1 d'autres poids "
            "(seed_everything au début de train, le générateur de make_loader).")
    tiny_9 = (SUBSET["x_train"][:64], SUBSET["y_train"][:64], SUBSET["x_val"][:32], SUBSET["y_val"][:32])
    with wbp.torch_state_kept(), contextlib.redirect_stdout(printed_9):
        z0_9 = train_9.train(*tiny_9, arch="cnn", widths=(4, 8), epochs=1, lr=1e-9, batch_size=64, seed=0)
        z1_9 = train_9.train(*tiny_9, arch="cnn", widths=(4, 8), epochs=1, lr=1e-9, batch_size=64, seed=1)
    ok_9["init"] = wbp.max_gap(z0_9["model"].state_dict(), z1_9["model"].state_dict()) > 1e-3
    verdict("B7.9", ok_9["init"], "l'initialisation dépend de la graine (avec lr = 1e-9, les poids rendus sont "
            "presque ceux du départ, et seed=1 en donne d'autres).",
            "avec lr = 1e-9 (les poids ne bougent presque pas), train(seed=0) et train(seed=1) rendent les mêmes poids : "
            "l'initialisation du modèle ne dépend pas de seed (seed_everything(seed, ...) avant build_model ?).")
    with wbp.torch_state_kept(), contextlib.redirect_stdout(printed_9):
        bn_9 = train_9.train(SUBSET["x_train"][:128], SUBSET["y_train"][:128], SUBSET["x_val"][:64],
                             SUBSET["y_val"][:64], arch="vgg", widths=(4,), epochs=2, batch_size=64, seed=0)
        probe_m_9 = wbp.build_reference("vgg", (4,)).train()
        train_9.evaluate(probe_m_9, data_9.make_loader(SUBSET["x_val"][:32], SUBSET["y_val"][:32], (0.5,) * 3,
                                                       (0.25,) * 3, batch_size=16))
    counted_9 = [int(v) for k, v in bn_9["model"].state_dict().items() if k.endswith("num_batches_tracked")][:1]
    ok_9["train mode"] = counted_9 == [4]
    verdict("B7.9", ok_9["train mode"], "un VGG entraîné 2 epochs de 2 mini-batches : sa batchnorm a vu les 4 "
            "mini-batches en mode train().", "un VGG entraîné 2 epochs de 2 mini-batches : sa première batchnorm en a "
            f"compté {counted_9[0] if counted_9 else '?'} en mode train(), attendu 4 (model.train() au début de chaque "
            "epoch ? evaluate rend-il le mode d'avant ? build_model fait-il un calcul en mode train ?).")
    ok_9["evaluate mode"] = probe_m_9.training
    verdict("B7.9", ok_9["evaluate mode"], "evaluate rend au modèle son mode d'avant (ici, train).",
            "evaluate laisse en mode eval() un modèle reçu en mode train() : rends-lui son mode d'avant "
            "(was_training = model.training, puis model.train(was_training)).")
    ok_9["print"] = printed_9.getvalue() == ""
    verdict("B7.9", ok_9["print"], "train n'écrit rien avec print.",
            f"train écrit avec print ({printed_9.getvalue().strip()[:80]!r}…) : passe par log.info et log.debug.")
    with wbp.torch_state_kept():
        TRAINED_9 = train_9.train(SUBSET["x_train"], SUBSET["y_train"], SUBSET["x_val"], SUBSET["y_val"])
    acc_9, s_9 = TRAINED_9["history"]["val_acc"][-1], TRAINED_9["seconds"]
    ok_9["smoke"] = s_9 < 60 and acc_9 > 0.15
    verdict("B7.9", ok_9["smoke"], f"entraînement fumée (réglages par défaut, 2 000 images, 1 epoch) : val_acc "
            f"{fr(acc_9, 3)} en {fr(s_9, 1)} s.", f"entraînement fumée (réglages par défaut, 2 000 images, 1 epoch) : "
            f"val_acc {fr(acc_9, 3)} en {fr(s_9, 1)} s ; attendu plus de 0,15 (le hasard : 0,10) en moins de 60 s.")

with checked("B7.9"):                                    # pip install -e, in a new environment
    if PYPROJECT_TOML is ...:
        print("⏳ Ex B7.9 : l'installation attend ton pyproject.toml.")
        raise StopCheck
    install_9 = wbp.install_editable(PROJECT)
    try:
        if install_9.ok is None:
            print("⏳ Ex B7.9 : ni pip ni uv dans cet environnement : l'installation éditable n'est pas vérifiée ici.")
            raise StopCheck
        ok_9["install"] = bool(install_9.ok)
        log_9 = install_9.log if len(install_9.log) <= 600 else install_9.log[-600:].split("\n", 1)[-1]
        why_9 = (" (ce backend n'est pas installé dans l'environnement du workbook : garde setuptools, comme le dit le "
                 "contrat)" if re.search(r"No module named '(hatchling|flit_core|poetry|pdm)", install_9.log) else "")
        verdict("B7.9", install_9.ok, f"pip install -e . dans un environnement neuf ({install_9.method}, "
                f"{fr(install_9.seconds, 1)} s).", f"pip install -e . échoue{why_9} : {log_9}")
        if not install_9.ok:
            raise StopCheck
        probe_9 = ("import sys, json, importlib.metadata as m, cifarprod, cifarprod.data, cifarprod.model, "
                   "cifarprod.train; print(json.dumps([cifarprod.__file__, m.version('cifarprod'), [e.value for e in "
                   "m.entry_points(group='console_scripts') if e.name == 'cifarprod'], 'wb' in sys.modules]))")
        seen_9 = install_9.run(["python", "-c", probe_9], scratch("elsewhere"))
        where_9, version_9, entry_9, uses_wb_9 = json.loads(seen_9.stdout) if seen_9.ok else (None, None, None, None)
        if seen_9.ok and where_9 is None:
            verdict("B7.9", False, "", "depuis un autre dossier, cifarprod.__file__ vaut None : sans "
                    "src/cifarprod/__init__.py, Python en fait un paquet-espace de noms (namespace package), sans "
                    "__version__ ; remets __init__.py.")
            raise StopCheck
        ok_9["no wb"] = uses_wb_9 is False
        if seen_9.ok and uses_wb_9:
            verdict("B7.9", False, "", "ton paquet importe wb (l'outillage du workbook) : un paquet installable ne "
                    "dépend que des dépendances déclarées dans son pyproject.toml.")
        ok_9["editable"] = seen_9.ok and Path(where_9).resolve().is_relative_to(PROJECT / "src")
        verdict("B7.9", ok_9["editable"], "depuis un autre dossier, import cifarprod lit ton dossier src/ (une "
                "installation éditable, pas une copie).", f"depuis un autre dossier, import cifarprod : {where_9 or seen_9.tail(1)}.")
        ok_9["metadata"] = version_9 == "0.1.0" and entry_9 == ["cifarprod.cli:main"]
        verdict("B7.9", ok_9["metadata"], "métadonnées installées : version 0.1.0, commande cifarprod → cifarprod.cli:main.",
                f"métadonnées installées : version {version_9!r} (attendu '0.1.0'), point d'entrée {entry_9!r} "
                "(attendu ['cifarprod.cli:main']).")
    finally:
        install_9.cleanup()
if ok_9 and all(ok_9.values()) and {"smoke", "editable", "metadata", "synthetic", "default"} <= set(ok_9):
    print("✅ Ex B7.9 : ton paquet s'installe, s'importe et s'entraîne : le notebook appelle, le paquet fait.")

### Ex B7.10 — Une interface en ligne de commande : argparse, logging et codes de sortie 📦 ★★ ⏱️ 30 min
**Objectif :** écrire la ligne de commande d'un entraînement : options, logging et codes de sortie qu'un autre programme sait lire.  
**Prérequis :** Ex B7.9 · fiche §207.2 (« argparse et le point d'entrée », « Le logging plutôt que print », « Les codes de sortie ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

**Le contrat.** Deux nouveaux fichiers ; la configuration, `config.py`, est fournie dans sa version de départ par la cellule suivante (une dataclass `TrainConfig` avec les réglages et leurs valeurs par défaut — `seed`, `epochs`, `lr`, `batch_size`, `n_train`, `arch`, `widths`, `data`, `deterministic`, `num_workers` —, et `load_config(path, overrides)`, sans aucune vérification des types ni des bornes : c'est B7.11). Elle n'est écrite que si `config.py` n'existe pas encore.

- `__main__.py` : `python -m cifarprod ...` appelle `main()` et transmet son code de sortie (207.2).
- `cli.py` : `build_parser()` rend un `ArgumentParser` de `prog="cifarprod"`, avec `--version` (« cifarprod 0.1.0 ») et la sous-commande **obligatoire** `train`, de quatre options : `--config FILE` (défaut `None` : les valeurs par défaut de `TrainConfig`), `--set KEY=VALUE` (répétable, défaut `[]`), `--out DIR` (défaut `runs/latest`) et `-v`/`--verbose`.
- `main(argv=None)` lit `argv` (`None` : `sys.argv`), configure le logging **une fois** (`logging.basicConfig`, niveau INFO, DEBUG avec `-v`, un format qui montre le niveau et le nom du logger : `%(levelname)s %(name)s: %(message)s`, sur stderr), puis : `load_config(args.config, args.set)` ; une `ValueError` ou une `OSError` → une ligne ERROR et le code **2** ; les parties `"train"` et `"val"` de `config.data` (un chemin relatif au dossier courant), dont il garde les `n_train` premières images d'entraînement ; des données illisibles ou trop peu d'images → une ligne ERROR et le code **1** ; `train(...)` avec les réglages de la configuration ; les poids dans `<out>/model.safetensors` (`safetensors.torch.save_file` du `state_dict`) ; une ligne INFO ; et sur stdout **une seule** ligne, le résumé JSON `{"status": "ok", "val_acc": <float>, "weights": <chemin absolu du fichier de poids>}` ; le code **0**. Toute autre exception pendant l'entraînement : `log.exception(...)` et le code 1. `main` **rend** son code (il n'appelle pas `sys.exit`, sauf argparse devant une mauvaise option, qui sort avec 2).

La cellule de vérification lance ta commande dans un autre processus (`python -m cifarprod`, depuis le dossier du projet, ton `src/` dans le `PYTHONPATH`) : `--version`, `--help`, sans sous-commande, avec une option inconnue, `--set lr` (sans `=`), des données absentes, un entraînement sur 200 images (son JSON, son fichier de poids, ses lignes INFO et l'absence de DEBUG), le même avec `-v`, un fichier `--config` sans aucun `--set`, et un réglage qui fait échouer l'entraînement (six étages de max pooling sur une image 32 × 32 : code 1) ; elle lit chaque fois le code de sortie, stdout et stderr. Puis elle appelle `main(["train", "--set", "lr"])` dans le notebook, qui doit rendre l'entier 2.

Dans tes notes : pourquoi le résumé va-t-il sur stdout et tout le reste sur stderr (pense à un script qui lit ce JSON) ? Pourquoi `main(argv)` reçoit-il ses arguments au lieu de lire `sys.argv`, et pourquoi rend-il son code au lieu d'appeler `sys.exit` ? Une intégration continue lance ta commande : que se passe-t-il si un entraînement qui échoue affiche son erreur mais rend 0 ?

In [ ]:
# the starting version of src/cifarprod/config.py, written by the check cell only if config.py does not exist yet (Ex B7.11 replaces it)
CONFIG_V0 = r'''"""The configuration of cifarprod, STARTING VERSION (given in Ex B7.10, replaced in Ex B7.11): the settings and their
defaults, read from a YAML file and the overrides "KEY=VALUE", without any check of the types or of the ranges."""

from __future__ import annotations

from dataclasses import asdict, dataclass, fields
from pathlib import Path

import yaml


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    epochs: int = 1
    lr: float = 0.05
    batch_size: int = 128
    n_train: int = 2000
    arch: str = "cnn"
    widths: tuple = (16, 32)
    data: str = "data/cifar10_2000.npz"
    deterministic: bool = True
    num_workers: int = 0

    def to_dict(self) -> dict:
        out = asdict(self)
        out["widths"] = list(self.widths)
        return out


def load_config(path=None, overrides=()):
    raw = dict(yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}) if path is not None else {}
    for item in overrides:
        key, sep, value = str(item).partition("=")
        if not sep:
            raise ValueError(f"--set expects KEY=VALUE, got {item!r}")
        raw[key.strip()] = yaml.safe_load(value)
    unknown = set(raw) - {f.name for f in fields(TrainConfig)}
    if unknown:
        raise ValueError(f"unknown settings: {sorted(unknown)}")
    return TrainConfig(**raw)
'''

In [ ]:
# Ex B7.10: the command line of the reference package. Its run journal (the import of .tracking, --runs, the record
# of each run) belongs to Ex B7.14, its artifact (save_artifact) to Ex B7.17: the solutions notebook writes the final
# cli.py at once, and tracking.py and artifact.py with it.

# src/cifarprod/__main__.py
MAIN_PY = r'''"""python -m cifarprod ...: the same command line as the `cifarprod` entry point."""

from .cli import main

raise SystemExit(main())
'''

# src/cifarprod/cli.py
CLI_PY = r'''"""The command line of cifarprod.

    cifarprod train --config configs/fast.yaml --set lr=0.01 -v      # or: python -m cifarprod train ...
    cifarprod --version

Exit codes: 0 success; 2 bad usage (argparse) or invalid configuration; 1 the data cannot be read or the training
failed. Messages go through logging (stderr); the only line written on stdout is the JSON summary of the run. The
folder --out receives the artifact of the model (model.safetensors and meta.json, artifact.py).
"""

from __future__ import annotations

import argparse
import json
import logging
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

from . import __version__
from .config import load_config
from .tracking import artifact, environment, log_run, new_run_id

log = logging.getLogger(__name__)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(prog="cifarprod", description="Train a CIFAR-10 classifier.")
    parser.add_argument("--version", action="version", version=f"%(prog)s {__version__}")
    sub = parser.add_subparsers(dest="command", required=True)
    train = sub.add_parser("train", help="train a model, save its weights and log the run")
    train.add_argument("--config", default=None, help="YAML file of settings (default: the defaults of TrainConfig)")
    train.add_argument("--set", action="append", default=[], metavar="KEY=VALUE", help="override a setting")
    train.add_argument("--out", default=None, help="folder of the weights (default: runs/<run id>)")
    train.add_argument("--runs", default="runs/runs.jsonl", help="the run journal (default: %(default)s)")
    train.add_argument("-v", "--verbose", action="store_true", help="also show the DEBUG messages")
    return parser


def run_train(args: argparse.Namespace) -> int:
    try:
        config = load_config(args.config, args.set)
    except (OSError, ValueError) as err:
        log.error("invalid configuration: %s", err)
        return 2
    run_id = new_run_id()
    out = Path(args.out) if args.out else Path("runs") / run_id
    record = {"run_id": run_id, "started": datetime.now(timezone.utc).isoformat(timespec="seconds"),
              "config": config.to_dict(), "environment": environment(), "status": "failed"}
    start = time.perf_counter()
    try:
        from .artifact import save_artifact
        from .data import CLASSES, fingerprint, load_npz
        from .train import train

        try:
            X, y = load_npz(config.data, "train")
            X_val, y_val = load_npz(config.data, "val")
            if config.n_train > len(X):
                raise ValueError(f"n_train = {config.n_train}, but {config.data} has {len(X)} training images")
        except (OSError, ValueError) as err:
            log.error("cannot load the data: %s", err)
            record["error"] = f"{type(err).__name__}: {err}"
            return 1
        X, y = X[:config.n_train], y[:config.n_train]
        log.info("loaded %d training and %d validation images from %s", len(X), len(X_val), config.data)
        record["data"] = {"path": config.data, "n_train": len(X), "sha256": fingerprint(X, y)}
        result = train(X, y, X_val, y_val, arch=config.arch, widths=config.widths, epochs=config.epochs,
                       lr=config.lr, batch_size=config.batch_size, seed=config.seed,
                       deterministic=config.deterministic, num_workers=config.num_workers)
        history = result["history"]
        weights, meta = out / "model.safetensors", out / "meta.json"
        save_artifact(out, result["model"], arch=config.arch, widths=config.widths, mean=result["mean"],
                      std=result["std"], classes=CLASSES, data={**record["data"], "run_id": run_id},
                      metrics={"val_acc": history["val_acc"][-1], "val_loss": history["val_loss"][-1]})
        record.update(status="ok", metrics={**history, "val_acc_final": history["val_acc"][-1]},
                      normalization={"mean": list(result["mean"]), "std": list(result["std"])},
                      artifacts=[artifact(weights), artifact(meta)])
        log.info("run %s: val_acc %.4f, weights in %s", run_id, history["val_acc"][-1], weights)
        print(json.dumps({"run_id": run_id, "status": "ok", "val_acc": history["val_acc"][-1],
                          "weights": str(weights.resolve())}))
        return 0
    except Exception as err:                      # noqa: BLE001 - logged with its traceback, exit code 1
        log.exception("training failed")
        record["error"] = f"{type(err).__name__}: {err}"
        return 1
    finally:
        record["seconds"] = round(time.perf_counter() - start, 3)
        log_run(args.runs, record)                # every run is logged, the failed ones too


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)        # a bad option: a message and exit code 2
    logging.basicConfig(level=logging.DEBUG if args.verbose else logging.INFO, stream=sys.stderr,
                        format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    return run_train(args)


if __name__ == "__main__":
    sys.exit(main())
'''

# src/cifarprod/tracking.py (Ex B7.14)
TRACKING_PY = r'''"""The run journal of cifarprod: one JSON line per run, appended to a JSONL file, never rewritten.

    run_id = new_run_id()                              # "20261104-161731-e06339"
    log_run(Path("runs/runs.jsonl"), {"run_id": run_id, "status": "ok", ...})
    runs = read_runs(Path("runs/runs.jsonl"))          # a list of dictionaries, oldest first

A run records what defines it (configuration, environment, data fingerprint), what it measures (metrics per epoch,
duration) and what it produces (its artifacts, with their path, size and fingerprint).
"""

from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import uuid
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path


def new_run_id() -> str:
    """A unique run identifier: the UTC date and time, then six random hexadecimal characters."""
    return f"{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:6]}"


def file_sha256(path: str | Path) -> str:
    """The SHA-256 of the bytes of a file (64 hexadecimal characters)."""
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def artifact(path: str | Path) -> dict:
    """{"path", "bytes", "sha256"} of a file written by the run."""
    path = Path(path)
    return {"path": str(path), "bytes": path.stat().st_size, "sha256": file_sha256(path)}


def _git(*args: str, cwd=None) -> str | None:
    try:
        done = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, timeout=10)
    except (OSError, subprocess.SubprocessError):
        return None
    return done.stdout.strip() if done.returncode == 0 else None


def environment(cwd=None) -> dict:
    """What runs the code: the versions of Python, NumPy and PyTorch, the platform, the git commit (None outside a
    repository) and whether the working tree has uncommitted changes."""
    versions = {}
    for name in ("numpy", "torch"):
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    status = _git("status", "--porcelain", cwd=cwd)
    return {"python": platform.python_version(), **versions, "platform": platform.platform(),
            "commit": _git("rev-parse", "--short", "HEAD", cwd=cwd), "dirty": bool(status) if status is not None else None}


def log_run(path: str | Path, record: dict) -> None:
    """Append one JSON line (keys sorted) to the journal ``path``; its folder is created if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, sort_keys=True) + "\n")


def read_runs(path: str | Path) -> list[dict]:
    """The runs of the journal, oldest first ([] if the file does not exist)."""
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
'''

# src/cifarprod/artifact.py (Ex B7.17: the final cli.py writes the artifact of each run)
ARTIFACT_PY = r'''"""The artifact of a model: everything needed to predict without the notebook, in one folder.

    save_artifact("artifact", model, arch="cnn", widths=(16, 32), mean=mean, std=std, classes=CLASSES,
                  data={"path": "data/cifar10_2000.npz", "n_train": 2000, "sha256": "..."})
    model, meta = load_artifact("artifact")                 # in eval() mode, the weights checked
    labels, proba = predict_images(model, meta, images)     # uint8 RGB (N, 32, 32, 3) -> names, probabilities

The folder holds model.safetensors (the weights: named tensors, nothing executed when loading) and meta.json (the
architecture, the input format, the normalization learned on the training images, the classes in the order of the
outputs, the data fingerprint, the metrics, the versions and the SHA-256 of the weights file).
"""

from __future__ import annotations

import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from . import __version__
from .data import preprocess
from .model import build_model
from .tracking import file_sha256

WEIGHTS, META = "model.safetensors", "meta.json"
INPUT = {"shape": [32, 32, 3], "dtype": "uint8", "channels": "RGB"}


def save_artifact(folder, model, *, arch, widths, mean, std, classes, data, metrics=None) -> Path:
    """Write the artifact of ``model`` into ``folder`` (created): model.safetensors (its state_dict, with the
    metadata {"package_version": __version__}) and meta.json (package_version, created, arch, widths, input, mean,
    std, classes, data, metrics, torch, python, weights_sha256). Returns the folder."""
    import torch
    from safetensors.torch import save_file

    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    state = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}
    save_file(state, str(folder / WEIGHTS), metadata={"package_version": __version__})
    meta = {"package_version": __version__, "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "arch": str(arch), "widths": [int(w) for w in widths], "input": dict(INPUT),
            "mean": [float(m) for m in mean], "std": [float(s) for s in std], "classes": [str(c) for c in classes],
            "data": dict(data), "metrics": dict(metrics or {}), "torch": torch.__version__,
            "python": platform.python_version(), "weights_sha256": file_sha256(folder / WEIGHTS)}
    (folder / META).write_text(json.dumps(meta, indent=1, sort_keys=True) + "\n", encoding="utf-8")
    return folder


def load_artifact(folder):
    """(model, meta): the model rebuilt from meta.json, its weights loaded from model.safetensors, in eval() mode.
    ValueError if the weights file is not the one meta.json describes (another SHA-256); FileNotFoundError if a file
    is missing."""
    from safetensors.torch import load_file

    folder = Path(folder)
    meta = json.loads((folder / META).read_text(encoding="utf-8"))
    sha = file_sha256(folder / WEIGHTS)
    if sha != meta.get("weights_sha256"):
        raise ValueError(f"the weights of {folder} do not match meta.json: SHA-256 {sha[:12]}…, expected "
                         f"{str(meta.get('weights_sha256'))[:12]}…")
    model = build_model(meta["arch"], meta["widths"], n_classes=len(meta["classes"]))
    model.load_state_dict(load_file(str(folder / WEIGHTS)))
    model.eval()
    return model, meta


def predict_images(model, meta, images):
    """(labels, probabilities) for uint8 RGB images (N, 32, 32, 3), or one image (32, 32, 3): the class names of
    meta["classes"] and a float32 array (N, n_classes) of softmax probabilities, computed without a graph, with the
    preprocessing of the training (data.preprocess and the normalization of the artifact)."""
    import torch

    x = preprocess(np.asarray(images), meta["mean"], meta["std"])
    with torch.inference_mode():
        proba = torch.softmax(model(x), dim=1).numpy()
    return [meta["classes"][k] for k in proba.argmax(axis=1)], proba
'''


ok_10 = {}
with checked("B7.10"):
    wbp.write_file(PROJECT, "src/cifarprod/config.py", CONFIG_V0, if_absent=True)
    files_10 = {"src/cifarprod/__main__.py": MAIN_PY, "src/cifarprod/cli.py": CLI_PY}
    if "TRACKING_PY" in globals() and TRACKING_PY is not ...:
        files_10["src/cifarprod/tracking.py"] = TRACKING_PY          # Ex B7.14, if it is already done
    if "ARTIFACT_PY" in globals() and ARTIFACT_PY is not ...:
        files_10["src/cifarprod/artifact.py"] = ARTIFACT_PY          # Ex B7.17, if it is already done
    wbp.write_files(PROJECT, files_10)
    if later_files_pending("B7.10"):
        raise StopCheck
    cli_10 = wbp.import_module(PROJECT, "cli")
    cli_10.build_parser()                                 # ⏳ while it raises NotImplementedError
    out_10 = scratch("out")
    r_10 = wbp.cli(PROJECT, "--version")
    ok_10["version"] = run_line("B7.10", r_10, r_10.ok and "0.1.0" in r_10.stdout, "code 0, « cifarprod 0.1.0 ».",
                                "attendu le code 0 et la version 0.1.0 sur stdout")
    r_10 = wbp.cli(PROJECT, "--help")
    ok_10["help"] = run_line("B7.10", r_10, r_10.ok and "train" in r_10.stdout, "code 0, l'aide cite train.",
                             "attendu le code 0 et une aide qui cite la sous-commande train")
    r_10 = wbp.cli(PROJECT)
    ok_10["no command"] = run_line("B7.10", r_10, r_10.returncode == 2 and "usage" in r_10.stderr.lower(),
                                   "code 2 et l'usage sur stderr (la sous-commande est obligatoire).",
                                   "attendu le code 2 d'argparse : la sous-commande train est obligatoire")
    r_10 = wbp.cli(PROJECT, "train", "--epochs", "3")
    ok_10["unknown option"] = run_line("B7.10", r_10, r_10.returncode == 2, "code 2 (option inconnue).",
                                       "attendu le code 2 d'argparse pour une option inconnue")
    r_10 = wbp.cli(PROJECT, "train", "--set", "lr", "--out", out_10 / "a")
    ok_10["bad set"] = run_line("B7.10", r_10, r_10.returncode == 2 and r_10.stderr_lines("ERROR") and not r_10.stdout.strip(),
                                "code 2, une ligne ERROR, rien sur stdout.",
                                "attendu le code 2 (configuration invalide), une ligne ERROR sur stderr et rien sur stdout")
    r_10 = wbp.cli(PROJECT, "train", "--set", "data=data/absent.npz", "--out", out_10 / "b")
    ok_10["no data"] = run_line("B7.10", r_10, r_10.returncode == 1 and r_10.stderr_lines("ERROR") and not r_10.stdout.strip(),
                                "code 1, une ligne ERROR, rien sur stdout.",
                                "attendu le code 1 (données illisibles), une ligne ERROR sur stderr et rien sur stdout")
    r_10 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--out", out_10 / "c")
    lines_10 = [line for line in r_10.stdout.splitlines() if line.strip()]
    summary_10 = r_10.last_json()
    ok_json_10 = (len(lines_10) == 1 and isinstance(summary_10, dict) and summary_10.get("status") == "ok"
                  and isinstance(summary_10.get("val_acc"), float) and 0 <= summary_10["val_acc"] <= 1)
    ok_10["train"] = run_line("B7.10", r_10, r_10.ok and ok_json_10,
                              f"code 0, une ligne JSON sur stdout (val_acc {fr(summary_10['val_acc'], 3) if ok_json_10 else '?'}).",
                              "attendu le code 0 et UNE ligne JSON sur stdout, {\"status\": \"ok\", \"val_acc\": <float>, "
                              f"\"weights\": ...}} ; reçu {r_10.stdout.strip()[:200]!r}")
    if ok_10["train"]:
        weights_10 = Path(summary_10.get("weights", ""))
        from safetensors.torch import load_file
        ok_10["weights"] = (weights_10.is_absolute() and weights_10 == (out_10 / "c" / "model.safetensors").resolve()
                            and weights_10.is_file() and set(load_file(str(weights_10)))
                            == set(wbp.build_reference("cnn", (16, 32)).state_dict()))
        verdict("B7.10", ok_10["weights"], "les poids sont dans <out>/model.safetensors (chemin absolu dans le JSON), "
                "ceux du CNN (16, 32).", f"attendu \"weights\" = le chemin absolu de <out>/model.safetensors, un "
                f"fichier safetensors du CNN (16, 32) ; reçu {str(weights_10)!r}.")
        infos_10 = [line for line in r_10.stderr_lines("INFO") if "cifarprod.train" in line]
        ok_10["logging"] = bool(infos_10) and not r_10.stderr_lines("DEBUG")
        verdict("B7.10", ok_10["logging"], f"stderr : {len(r_10.stderr_lines('INFO'))} lignes INFO, dont "
                f"« {infos_10[-1].split(': ', 1)[-1][:60] if infos_10 else ''} » de cifarprod.train ; aucune DEBUG.",
                "attendu sur stderr des lignes INFO (au moins une du logger cifarprod.train, avec le niveau et le nom "
                f"du logger) et aucune ligne DEBUG sans -v ; reçu {r_10.tail(3)[:300]!r}")
        r_10 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--out", out_10 / "d", "-v")
        ok_10["verbose"] = run_line("B7.10", r_10, r_10.ok and any("cifarprod" in line for line in r_10.stderr_lines("DEBUG")),
                                    "avec -v, les lignes DEBUG des mini-batches.",
                                    "attendu, avec -v, des lignes DEBUG du paquet sur stderr")
    (out_10 / "small.yaml").write_text("n_train: 200\n", encoding="utf-8")
    r_10 = wbp.cli(PROJECT, "train", "--config", out_10 / "small.yaml", "--out", out_10 / "e")
    ok_10["config"] = run_line("B7.10", r_10, r_10.ok, "code 0 avec --config et sans aucun --set.",
                               "attendu le code 0 pour train --config FILE sans aucun --set (le défaut de --set : [])")
    r_10 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--set", "widths=[4, 4, 4, 4, 4, 4]", "--out", out_10 / "f")
    ok_10["fails"] = run_line("B7.10", r_10, r_10.returncode == 1 and r_10.stderr_lines("ERROR") and not r_10.stdout.strip(),
                              "code 1, une ligne ERROR (avec sa trace), rien sur stdout, quand l'entraînement échoue.",
                              "un CNN à six étages de max pooling ne peut pas traiter une image 32 × 32 : attendu le code "
                              "1, une ligne ERROR (log.exception) et rien sur stdout quand l'entraînement échoue")
    with wbp.isolated_logging():                         # main() configures the logging of THIS process
        try:
            code_10 = cli_10.main(["train", "--set", "lr"])
        except SystemExit as exit_10:                     # sys.exit inside main: the notebook would stop there
            code_10 = f"SystemExit({exit_10.code!r})"
    ok_10["main"] = type(code_10) is int and code_10 == 2
    verdict("B7.10", ok_10["main"], 'main(["train", "--set", "lr"]) rend l\'entier 2 dans le notebook.',
            f'main(["train", "--set", "lr"]) doit rendre l\'entier 2 (sans sys.exit) ; reçu {code_10!r}.')
if ok_10 and all(ok_10.values()):
    print("✅ Ex B7.10 : ta ligne de commande parle avec des codes de sortie et du logging.")

### Ex B7.11 — Configuration : dataclass + YAML, surcharges en ligne de commande, validation 📦 ★★ ⏱️ 30 min
**Objectif :** remplacer les réglages en dur par une configuration typée, validée au chargement et surchargeable.  
**Prérequis :** Ex B7.10 · fiche §207.3 (le code de `TrainConfig` et de `load_config`, « Rappel outil — YAML avec PyYAML ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

**Le contrat** de `config.py` (il remplace la version de départ de B7.10 ; `cli.py` ne change pas) :

- `TrainConfig` est une dataclass **figée** (`frozen=True`) aux dix champs et aux valeurs par défaut de la version de départ (`seed=0`, `epochs=1`, `lr=0.05`, `batch_size=128`, `n_train=2000`, `arch="cnn"`, `widths=(16, 32)`, `data="data/cifar10_2000.npz"`, `deterministic=True`, `num_workers=0`) ;
- à la construction (`__post_init__`), chaque réglage a **exactement** le type de sa valeur par défaut (`True` n'est pas un entier, la chaîne `"1e-3"` n'est pas un float), sauf deux conversions permises : un entier pour un float (`lr=1` devient `1.0`) et une liste pour un tuple (`widths=[8, 16]` devient `(8, 16)`) ; puis les bornes : `seed` ≥ 0, `epochs` ≥ 1, $0 <$ `lr` $< 1$, `batch_size` ≥ 1, `n_train` ≥ 1, `num_workers` ≥ 0, `arch` dans `("cnn", "vgg")`, `widths` non vide et fait d'entiers ≥ 1. Toute violation lève une `ValueError` dont le message **nomme le réglage** ;
- `to_dict()` rend un dictionnaire que `json.dumps` accepte (`widths` en liste) ;
- `load_config(path=None, overrides=())` part des valeurs par défaut, puis du fichier YAML `path` (`yaml.safe_load` ; un fichier vide donne les valeurs par défaut ; un fichier qui n'est pas un dictionnaire, `ValueError` ; un fichier absent, `FileNotFoundError`), puis des surcharges `"KEY=VALUE"` (chaque valeur lue par `yaml.safe_load`, donc `widths=[8, 16]` est une liste ; une surcharge sans `=`, `ValueError`) ; une clé inconnue, du fichier ou d'une surcharge, lève une `ValueError` qui la nomme.

Deux fichiers de configuration aussi : `configs/fast.yaml`, l'entraînement fumée du notebook, dont les réglages sont **exactement** les valeurs par défaut (écrits dans le fichier, pour qu'il se suffise) ; `configs/smoke.yaml`, l'essai de bout en bout de l'intégration continue : les mêmes, sauf `n_train: 256`, `batch_size: 64` et `widths: [8, 16]`.

La cellule de vérification essaie ta configuration dans le notebook : valeurs par défaut et `to_dict` en JSON, dataclass figée, conversions, seize surcharges invalides (chacune doit lever une `ValueError` qui nomme le réglage), l'ordre défauts → fichier → surcharges, trois fichiers pièges (`lr: 1e-3`, une faute de frappe, une liste), un fichier vide, un fichier absent et un fichier avec un tag YAML `!!python/...` (que `yaml.safe_load` refuse), puis tes deux fichiers YAML (les dix réglages écrits dans chacun) ; enfin ta commande avec `--set epochs=0` (code 2, avant toute seconde de calcul) et `--config configs/smoke.yaml` (code 0).

Dans tes notes : que lit PyYAML dans `lr: 1e-3`, et pourquoi la validation des types est-elle la seule chose qui te sauve ici ? Pourquoi refuser une clé inconnue plutôt que l'ignorer ? Quels réglages ne mettrais-tu **jamais** dans ce fichier, et où iraient-ils (207.9) ?

In [ ]:
# Ex B7.11: the typed configuration of the reference package, and its two configuration files

# src/cifarprod/config.py
CONFIG_PY = r'''"""The configuration of cifarprod: a typed, frozen dataclass, read from a YAML file and overridden on the command line.

    config = load_config("configs/fast.yaml", ["lr=0.01", "widths=[8, 16]"])
    config.lr, config.to_dict()

Every setting is checked when the configuration is built: its type, its range, and no unknown key (a typo fails
instead of being ignored). A secret never goes here: a configuration is versioned and copied into the run journal.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, fields
from pathlib import Path

import yaml

ARCHS = ("cnn", "vgg")


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    epochs: int = 1
    lr: float = 0.05
    batch_size: int = 128
    n_train: int = 2000
    arch: str = "cnn"
    widths: tuple = (16, 32)
    data: str = "data/cifar10_2000.npz"
    deterministic: bool = True
    num_workers: int = 0

    def __post_init__(self) -> None:
        for f in fields(self):                          # a dataclass does not check the types: we do
            value, kind = getattr(self, f.name), type(f.default)
            if kind is float and type(value) is int:
                object.__setattr__(self, f.name, float(value))          # 1 -> 1.0 is fine
            elif kind is tuple and type(value) is list:
                object.__setattr__(self, f.name, tuple(value))          # a YAML list -> a tuple
            elif type(value) is not kind:                               # True is not an int here
                raise ValueError(f"{f.name}: expected {kind.__name__}, got {value!r}")
        if not self.widths or any(type(w) is not int or w < 1 for w in self.widths):
            raise ValueError(f"widths: expected positive whole numbers, got {self.widths!r}")
        if self.arch not in ARCHS:
            raise ValueError(f"arch: expected one of {ARCHS}, got {self.arch!r}")
        if self.seed < 0 or self.epochs < 1 or self.batch_size < 1 or self.n_train < 1 or self.num_workers < 0:
            raise ValueError(f"need seed >= 0, epochs >= 1, batch_size >= 1, n_train >= 1 and num_workers >= 0, got "
                             f"{self.seed}, {self.epochs}, {self.batch_size}, {self.n_train} and {self.num_workers}")
        if not 0 < self.lr < 1:
            raise ValueError(f"lr: expected 0 < lr < 1, got {self.lr}")

    def to_dict(self) -> dict:
        """The settings as a plain dictionary that json.dumps accepts (widths as a list)."""
        out = asdict(self)
        out["widths"] = list(self.widths)
        return out


def load_config(path: str | Path | None = None, overrides=()) -> TrainConfig:
    """The TrainConfig of the YAML file ``path`` (None: the defaults), then of the overrides "KEY=VALUE" (each VALUE
    read by yaml.safe_load). ValueError for a file that is not a mapping, an override without "=", an unknown key or a
    setting refused by TrainConfig; FileNotFoundError for a missing file."""
    raw = {}
    if path is not None:
        loaded = yaml.safe_load(Path(path).read_text(encoding="utf-8"))
        if loaded is not None and not isinstance(loaded, dict):
            raise ValueError(f"{path}: expected a mapping of settings, got {type(loaded).__name__}")
        raw.update(loaded or {})
    for item in overrides:
        key, sep, value = str(item).partition("=")
        if not sep or not key.strip():
            raise ValueError(f"--set expects KEY=VALUE, got {item!r}")
        raw[key.strip()] = yaml.safe_load(value)
    unknown = set(raw) - {f.name for f in fields(TrainConfig)}
    if unknown:
        raise ValueError(f"unknown settings: {sorted(unknown)}")
    return TrainConfig(**raw)
'''

# configs/fast.yaml
FAST_YAML = r'''# configs/fast.yaml: the smoke training of the notebook (2 000 images, 1 epoch, < 1 min on a CPU)
seed: 0
epochs: 1
lr: 0.05
batch_size: 128
n_train: 2000
arch: cnn
widths: [16, 32]
data: data/cifar10_2000.npz
deterministic: true
num_workers: 0
'''

# configs/smoke.yaml
SMOKE_YAML = r'''# configs/smoke.yaml: the end-to-end test of the continuous integration (a few seconds: it judges no accuracy)
seed: 0
epochs: 1
lr: 0.05
batch_size: 64
n_train: 256
arch: cnn
widths: [8, 16]
data: data/cifar10_2000.npz
deterministic: true
num_workers: 0
'''


import dataclasses

import yaml

ok_11 = {}
DEFAULTS_11 = DEFAULTS
with checked("B7.11"):
    wbp.write_files(PROJECT, {"src/cifarprod/config.py": CONFIG_PY, "configs/fast.yaml": FAST_YAML,
                              "configs/smoke.yaml": SMOKE_YAML})
    config_11 = wbp.import_module(PROJECT, "config")
    default_11 = config_11.TrainConfig()
    ok_11["defaults"] = default_11.to_dict() == DEFAULTS_11 and json.loads(json.dumps(default_11.to_dict())) == DEFAULTS_11
    verdict("B7.11", ok_11["defaults"], "TrainConfig() et to_dict() : les valeurs par défaut du contrat, en JSON.",
            f"TrainConfig().to_dict() doit valoir {DEFAULTS_11} ; reçu {default_11.to_dict()!r:.300}.")
    try:
        default_11.lr = 0.1
        ok_11["frozen"] = False
    except dataclasses.FrozenInstanceError:
        ok_11["frozen"] = True
    verdict("B7.11", ok_11["frozen"], "la configuration est figée (frozen=True).",
            "config.lr = 0.1 doit lever FrozenInstanceError : un run ne modifie pas sa configuration en route.")
    cast_11 = config_11.TrainConfig(lr=1 / 2 ** 4, epochs=2, widths=[8, 16])
    ok_11["cast"] = type(config_11.TrainConfig(seed=0, lr=0.5).lr) is float and cast_11.widths == (8, 16) \
        and type(config_11.TrainConfig(**{"lr": 1e-1}).lr) is float
    lr_int_11 = config_11.load_config(None, ["batch_size=64", "lr=0.25"])
    verdict("B7.11", ok_11["cast"] and lr_int_11.batch_size == 64, "un entier accepté pour un float, une liste YAML "
            "devient un tuple.", "attendu lr en float, widths=[8, 16] devenu (8, 16), et --set batch_size=64.")
    for override_11, name_11 in [("epochs=0", "epochs"), ("epochs=1.5", "epochs"), ("lr=0", "lr"), ("lr=1", "lr"),
                                 ("lr=1e-3", "lr"), ("batch_size=0", "batch_size"), ("batch_size=true", "batch_size"),
                                 ("n_train=0", "n_train"), ("seed=-1", "seed"), ("arch=resnet", "arch"),
                                 ("widths=[]", "widths"), ("widths=[8, 0]", "widths"), ("deterministic=yes2", "deterministic"),
                                 ("num_workers=-1", "num_workers"), ("lerning_rate=0.01", "lerning_rate"), ("lr", "lr")]:
        try:
            config_11.load_config(None, [override_11])
            ok_11[override_11] = False
            print(f"❌ Ex B7.11 : --set {override_11} : attendu ValueError, mais rien n'a été levé.")
        except ValueError as err_11:
            ok_11[override_11] = name_11 in str(err_11)
            if not ok_11[override_11]:
                print(f"❌ Ex B7.11 : --set {override_11} : ValueError levée, mais son message ne nomme pas {name_11!r} : "
                      f"« {str(err_11)[:120]} ».")
        except NotImplementedError:
            raise
        except Exception as err_11:  # noqa: BLE001 - the learner's code
            ok_11[override_11] = False
            print(f"❌ Ex B7.11 : --set {override_11} : attendu ValueError, reçu {type(err_11).__name__} : {str(err_11)[:120]}")
    refused_11 = [k for k in ok_11 if "=" in k or k == "lr"]
    if all(ok_11[k] for k in refused_11):
        print(f"✅ Ex B7.11 : {len(refused_11)} réglages invalides refusés, chacun par une ValueError qui le nomme "
              "(types, bornes, clé inconnue, --set sans =).")
    yaml_11 = scratch("yaml")
    (yaml_11 / "a.yaml").write_text("lr: 0.1\nepochs: 3\nwidths: [4, 8]\n", encoding="utf-8")
    both_11 = config_11.load_config(yaml_11 / "a.yaml", ["lr=0.01", "epochs=2"])
    ok_11["precedence"] = (both_11.lr, both_11.epochs, both_11.widths, both_11.seed) == (0.01, 2, (4, 8), 0)
    verdict("B7.11", ok_11["precedence"], "les valeurs par défaut, puis le fichier, puis les --set.",
            f"fichier lr 0.1, epochs 3, widths [4, 8] puis --set lr=0.01 epochs=2 : attendu (0.01, 2, (4, 8), seed 0), "
            f"reçu {(both_11.lr, both_11.epochs, both_11.widths, both_11.seed)}.")
    for text_11, what_11 in [("lr: 1e-3\n", "lr: 1e-3 (une chaîne pour PyYAML)"), ("lerning_rate: 0.01\n", "une faute de frappe"),
                             ("- lr\n- 0.1\n", "une liste au lieu d'un dictionnaire")]:
        (yaml_11 / "bad.yaml").write_text(text_11, encoding="utf-8")
        ok_11[what_11] = refuses("B7.11", f"un fichier avec {what_11}", lambda: config_11.load_config(yaml_11 / "bad.yaml"))
    (yaml_11 / "empty.yaml").write_text("# nothing\n", encoding="utf-8")
    ok_11["empty"] = config_11.load_config(yaml_11 / "empty.yaml").to_dict() == DEFAULTS_11
    verdict("B7.11", ok_11["empty"], "un fichier vide : les valeurs par défaut.", "un fichier vide (un commentaire) doit "
            "donner les valeurs par défaut (yaml.safe_load rend None).")
    ok_11["absent"] = refuses("B7.11", "un fichier absent", lambda: config_11.load_config(yaml_11 / "absent.yaml"),
                              error=FileNotFoundError)
    (yaml_11 / "tag.yaml").write_text('lr: !!python/object/apply:float ["0.1"]\n', encoding="utf-8")
    ok_11["safe_load"] = refuses("B7.11", "un fichier avec un tag YAML !!python (yaml.safe_load le refuse)",
                                 lambda: config_11.load_config(yaml_11 / "tag.yaml"), error=Exception)
    want_files_11 = {"configs/fast.yaml": DEFAULTS_11,
                     "configs/smoke.yaml": {**DEFAULTS_11, "n_train": 256, "batch_size": 64, "widths": [8, 16]}}
    for rel_11, want_11 in want_files_11.items():
        try:
            got_11 = config_11.load_config(PROJECT / rel_11).to_dict()
        except ValueError as err_11:
            ok_11[rel_11] = False
            print(f"❌ Ex B7.11 : {rel_11} est refusé par ta configuration : « {str(err_11)[:160]} » (pour PyYAML, un "
                  "nombre sans point, comme 5e-2, est une chaîne : écris 0.05).")
            continue
        try:
            written_11 = yaml.safe_load((PROJECT / rel_11).read_text(encoding="utf-8")) or {}
        except yaml.YAMLError:
            written_11 = {}
        absent_11 = sorted(set(want_11) - set(written_11)) if isinstance(written_11, dict) else sorted(want_11)
        ok_11[rel_11] = got_11 == want_11 and not absent_11
        verdict("B7.11", ok_11[rel_11], f"{rel_11} : les dix réglages du contrat, écrits dans le fichier.",
                (f"{rel_11} : attendu {want_11}, reçu {got_11}" if got_11 != want_11 else f"{rel_11} : les bonnes "
                 "valeurs, mais pas toutes écrites") + (f" ; réglages absents du fichier (il doit se suffire, sans "
                 f"compter sur les valeurs par défaut) : {absent_11}" if absent_11 else "") + ".")
    if later_files_pending("B7.11"):
        ok_11["cli"] = False
    else:
        out_11 = scratch("out")
        r_11 = wbp.cli(PROJECT, "train", "--set", "epochs=0", "--out", out_11 / "a")
        ok_11["cli epochs"] = run_line("B7.11", r_11, r_11.returncode == 2 and any("epochs" in l for l in r_11.stderr_lines("ERROR")),
                                       "code 2 et une ligne ERROR qui nomme epochs, avant toute seconde de calcul.",
                                       "attendu le code 2 et une ligne ERROR qui nomme epochs")
        r_11 = wbp.cli(PROJECT, "train", "--config", "configs/smoke.yaml", "--out", out_11 / "b")
        ok_11["cli smoke"] = run_line("B7.11", r_11, r_11.ok, f"code 0 en {fr(r_11.seconds, 1)} s.", "attendu le code 0")
if ok_11 and all(ok_11.values()):
    print("✅ Ex B7.11 : ta configuration est typée, validée au chargement et surchargeable.")

### Ex B7.12 — Prédire : même graine, puis num_workers = 4, puis GPU — résultats identiques ? 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que les workers d'un DataLoader et un GPU changent à un entraînement à graine fixée.  
**Prérequis :** Ex B7.11 · ch. 24 (workers) · fiche §207.4 (« Les graines et le déterminisme ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

L'expérience entraîne six fois un petit CNN (`wbp.workers_run`, 512 images fixes, 2 epochs, graine 0) et compare les poids **au bit près** (`torch.equal` sur chaque tenseur). Chaque run fait `torch.manual_seed(0)`, puis lit ses images par un `DataLoader(..., shuffle=True, generator=torch.Generator().manual_seed(0), num_workers=...)`, sans `worker_init_fn` ni `persistent_workers`. Le dataset applique une **augmentation** : dans son `__getitem__`, il retourne l'image (gauche-droite) quand `torch.rand(1) < 0.5`. Avec `num_workers = 0`, `__getitem__` tourne dans le processus du notebook ; avec `num_workers = 4`, dans quatre processus fils (chaque epoch en lance de nouveaux). Avec 4 workers sur une machine à 2 cœurs, PyTorch avertit qu'il en propose au plus 2 : l'avertissement est tu, il ne change rien aux nombres. **Avant d'exécuter**, écris tes quatre prédictions (`True` ou `False`) :
- a) `replay0_12` : deux runs avec `num_workers = 0` donnent-ils exactement les mêmes poids ?
- b) `replay4_12` : deux runs avec `num_workers = 4` donnent-ils exactement les mêmes poids ?
- c) `zero_four_12` : le run à `num_workers = 0` et le run à `num_workers = 4` donnent-ils exactement les mêmes poids ?
- d) `plain_12` : **sans** l'augmentation (aucun tirage dans `__getitem__`), `num_workers = 0` et `num_workers = 4` donnent-ils exactement les mêmes poids ?

Et, dans ta cellule 📝 seulement (aucune vérification) : e) sur un GPU, avec la même graine et `num_workers = 0`, les poids seraient-ils ceux du CPU ? Ceux d'un second run sur le même GPU ? La cellule d'expérience le mesure si un GPU est là (sur Colab), et le dit sinon.

La cellule d'expérience n'affiche rien tant que tes quatre prédictions ne sont pas écrites ; elle les vérifie à la fin. Rien n'y dépend de ton paquet.

In [ ]:
replay0_12, replay4_12, zero_four_12, plain_12 = True, True, False, True

In [ ]:
wb.record("B7.12a", True, mistakes={"relis le tableau : num_workers 0, essai 1 contre essai 2": False}, prediction=True)
wb.record("B7.12b", True, mistakes={"relis le tableau : num_workers 4, essai 1 contre essai 2": False}, prediction=True)
wb.record("B7.12c", False, mistakes={"relis le tableau : num_workers 0 contre 4, avec l'augmentation": True}, prediction=True)
wb.record("B7.12d", True, mistakes={"relis le tableau : num_workers 0 contre 4, sans l'augmentation": False}, prediction=True)

In [ ]:
if any(answer is ... or answer is None for answer in [replay0_12, replay4_12, zero_four_12, plain_12]):
    print("⏳ Ex B7.12 : écris d'abord tes quatre prédictions dans la cellule précédente.")
else:
    start_12 = time.perf_counter()
    RUNS_12 = {"0, essai 1": wbp.workers_run(0, seed=0), "0, essai 2": wbp.workers_run(0, seed=0),
               "4, essai 1": wbp.workers_run(4, seed=0), "4, essai 2": wbp.workers_run(4, seed=0),
               "0, sans retournement": wbp.workers_run(0, seed=0, augment=False),
               "4, sans retournement": wbp.workers_run(4, seed=0, augment=False)}
    PAIRS_12 = {"a": ("0, essai 1", "0, essai 2"), "b": ("4, essai 1", "4, essai 2"), "c": ("0, essai 1", "4, essai 1"),
                "d": ("0, sans retournement", "4, sans retournement")}
    print(f"Six entraînements courts en {fr(time.perf_counter() - start_12, 1)} s (512 images, 2 epochs, graine 0) :")
    for letter_12, (first_12, second_12) in PAIRS_12.items():
        a_12, b_12 = RUNS_12[first_12], RUNS_12[second_12]
        print(f"  {letter_12}) num_workers {first_12} contre {second_12} : "
              + ("mêmes poids au bit près" if wbp.same_weights(a_12, b_12) else f"poids différents (écart maximal "
                 f"{wbp.max_gap(a_12, b_12):.2e})"))
    if torch.cuda.is_available():
        gpu_12 = [wbp.workers_run(0, seed=0, device="cuda") for _ in range(2)]
        print(f"  e) sur le GPU ({torch.cuda.get_device_name(0)}) : deux runs "
              + ("identiques" if wbp.same_weights(*gpu_12) else f"différents ({wbp.max_gap(*gpu_12):.2e})")
              + " ; contre le CPU : " + ("identiques" if wbp.same_weights(gpu_12[0], RUNS_12["0, essai 1"])
                                         else f"différents ({wbp.max_gap(gpu_12[0], RUNS_12['0, essai 1']):.2e})"))
    else:
        print("  e) pas de GPU ici : ta prédiction e) se compare à 05_solutions.md (ou relance cette cellule sur un GPU "
              "de Colab).")
    wb.check("B7.12a", replay0_12)
    wb.check("B7.12b", replay4_12)
    wb.check("B7.12c", zero_four_12)
    wb.check("B7.12d", plain_12)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? D'où chaque worker tient-il son générateur, et pourquoi la graine du `DataLoader` suffit-elle à le rejouer ? Que faudrait-il pour qu'un run à 0 worker et un run à 4 workers tirent les mêmes retournements ? Que ferait `worker_init_fn` (207.4) pour les tirages de NumPy et de `random` dans un worker ? Pourquoi ces résultats ne disent-ils rien d'un GPU ?

### Ex B7.13 — Mesurer le non-déterminisme : graines fixées ou non, algorithmes déterministes, cinq exécutions 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer, sur ton propre entraînement, ce que changent la graine et les algorithmes déterministes, et rapporter la dispersion d'un score.  
**Prérequis :** Ex B7.12 · ✏️ B7.2 · fiche §207.4 (« Les graines et le déterminisme », « Rapporter un score ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

Tu lances **ta** commande cinq fois et tu compares les poids qu'elle écrit. **Contrat** des trois fonctions de la cellule suivante :
- `train_run_13(seed, deterministic=True)` lance une fois, par `wbp.cli(PROJECT, ...)`, la commande `train --config configs/fast.yaml --set seed=<seed> --set deterministic=<true|false> --out <un nouveau dossier temporaire>` ; elle rend `{"weights": le dictionnaire de tenseurs du fichier de poids écrit, "val_acc": la val_acc de la ligne JSON, "seconds": la durée de la commande}`, et lève une `RuntimeError` avec la fin de stderr si le code de sortie n'est pas 0 ;
- `same_weights_13(a, b)` rend `True` quand les deux dictionnaires ont les mêmes noms et exactement les mêmes tenseurs, `False` sinon (un booléen) ;
- `max_gap_13(a, b)` rend le plus grand écart absolu entre les tenseurs de même nom, un float Python.

La cellule de vérification essaie `same_weights_13` et `max_gap_13` sur trois petits cas (des copies, un poids qui diffère de $10^{-6}$, un tenseur en moins). L'expérience lance ensuite cinq runs de 2 000 images (≈ 4 s chacun) : A et B, graine 0 ; C, graine 0 sans les algorithmes déterministes ; D et E, graines 1 et 2. Elle vérifie que chaque run reçoit bien `--set seed=…` et `--set deterministic=…`, et que les graines 0, 1 et 2 donnent trois modèles différents ; elle affiche les écarts A-B, A-C, A-D et D-E, l'accuracy des graines 0, 1 et 2 avec son intervalle à 95 % (la formule de ✏️ B7.2, $t_{0{,}975;\,2} = 4{,}303$, donné par le tableau de 207.4), et vérifie par `wb.check` si A et B ont exactement les mêmes poids (rien à taper : la valeur vient de ton code).

Dans tes notes : que donne A contre C sur ce processeur, et pourquoi ce résultat ne vaut-il que pour **ce** matériel et **ce** logiciel (207.4) ? De quel ordre est l'écart d'une graine à l'autre, et l'écart d'accuracy ? Avec trois graines, que vaut l'intervalle, et que dirais-tu d'un concurrent annoncé 1 point au-dessus sur une seule graine ? Qu'est-ce qui, dans ton paquet, rend A et B identiques : liste les générateurs que la graine fixe.

In [ ]:
from safetensors.torch import load_file


def train_run_13(seed, deterministic=True):
    """Run YOUR command line once in PROJECT: python -m cifarprod train --config configs/fast.yaml --set seed=<seed>
    --set deterministic=<true|false> --out <a new temporary folder>, through wbp.cli. Returns {"weights": the dict of
    tensors of the model.safetensors it wrote, "val_acc": the val_acc of its JSON line, "seconds": its duration};
    RuntimeError with the end of stderr if the command does not end with the code 0."""
    out = scratch("run13")
    result = wbp.cli(PROJECT, "train", "--config", "configs/fast.yaml", "--set", f"seed={seed}",
                     "--set", f"deterministic={str(deterministic).lower()}", "--out", out)
    if not result.ok:
        raise RuntimeError(f"cifarprod train exited with {result.returncode}: {result.tail(3)}")
    summary = result.last_json()
    return {"weights": load_file(summary["weights"]), "val_acc": summary["val_acc"], "seconds": result.seconds}


def same_weights_13(a, b):
    """True when the two dicts of tensors have the same names and exactly the same values (torch.equal), else False."""
    return a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a)


def max_gap_13(a, b):
    """The largest absolute difference between the tensors of the same name of two dicts, as a Python float."""
    return max(float((a[k].double() - b[k].double()).abs().max()) for k in a)


tools_ok_13 = False
with checked("B7.13"):
    base_13 = {"w": torch.zeros(2, 3), "b": torch.ones(4), "n": torch.tensor(7)}
    twin_13 = {k: v.clone() for k, v in base_13.items()}
    tiny_13 = {**twin_13, "w": torch.tensor([[0.0, 0.0, 1e-6], [0.0, 0.0, 0.0]])}
    cases_13 = [("deux copies identiques", base_13, twin_13, True), ("un poids qui diffère de 1e-6", base_13, tiny_13, False),
                ("un tenseur en moins", base_13, {k: v for k, v in twin_13.items() if k != "b"}, False)]
    all_13 = True
    for label_13, a_13, b_13, want_13 in cases_13:
        got_13 = same_weights_13(a_13, b_13)
        good_13 = isinstance(got_13, (bool, np.bool_)) and bool(got_13) == want_13
        all_13 &= good_13
        verdict("B7.13", good_13, f"same_weights_13 sur {label_13} : {want_13}.",
                f"same_weights_13 sur {label_13} : attendu {want_13}, reçu {got_13!r}.")
    gap_13 = max_gap_13(base_13, tiny_13)
    ok_gap_13 = type(gap_13) is float and abs(gap_13 - float(torch.tensor(1e-6))) < 1e-12
    verdict("B7.13", ok_gap_13, "max_gap_13 : 1e-6 (un float Python).",
            f"max_gap_13 sur un poids qui diffère de 1e-6 : reçu {gap_13!r}, attendu le float 1e-6 (à l'arrondi du float32 près).")
    tools_ok_13 = bool(all_13 and ok_gap_13)
if tools_ok_13 and later_files_pending("B7.13"):
    tools_ok_13 = False
probe_13 = wbp.cli(PROJECT, "--version")
if tools_ok_13 and not probe_13.ok:
    tools_ok_13 = False
    print(f"⏳ Ex B7.13 : ta commande ne répond pas encore (cifarprod --version : code {probe_13.returncode}, "
          f"{probe_13.tail(1)[:160]}) : fais d'abord passer B7.10 et B7.11.")

In [ ]:
wb.record("B7.13", True, mistakes={"tes deux runs à même graine n'ont pas exactement les mêmes poids : qu'est-ce qui, "
                                   "dans ton entraînement, ne dépend pas de la graine de la configuration ?": False})

In [ ]:
RUNS_13 = {}
if not tools_ok_13:
    print("⏳ Ex B7.13 : l'expérience attend la cellule précédente (ce qui y manque est dit ci-dessus).")
else:
    with checked("B7.13"):
        PLAN_13 = {"A": (0, True), "B": (0, True), "C": (0, False), "D": (1, True), "E": (2, True)}
        CALLS_13, cli_13 = [], wbp.cli                   # the commands that train_run_13 runs through wbp.cli

        def spy_cli_13(project, *args, **kwargs):
            CALLS_13.append([str(a) for a in args])
            return cli_13(project, *args, **kwargs)

        wbp.cli = spy_cli_13
        try:
            for key_13, (seed_13, det_13) in PLAN_13.items():
                first_call_13 = len(CALLS_13)
                RUNS_13[key_13] = train_run_13(seed_13, det_13)
                print(f"run {key_13} : graine {seed_13}, algorithmes déterministes {'oui' if det_13 else 'non'} → "
                      f"val_acc {fr(RUNS_13[key_13]['val_acc'], 3)} en {fr(RUNS_13[key_13]['seconds'], 1)} s")
                set_13 = {}
                for args_13 in CALLS_13[first_call_13:]:
                    for a_13, b_13 in zip(args_13, args_13[1:]):
                        if a_13 == "--set" and "=" in b_13:
                            key_s_13, _, value_s_13 = b_13.partition("=")
                            try:
                                set_13[key_s_13.strip()] = yaml.safe_load(value_s_13)
                            except yaml.YAMLError:
                                set_13[key_s_13.strip()] = value_s_13
                if len(CALLS_13) > first_call_13 and (set_13.get("seed") != seed_13
                                                      or set_13.get("deterministic") is not det_13):
                    verdict("B7.13", False, "", f"run {key_13} : train_run_13({seed_13}, {det_13}) lance ta commande "
                            f"avec les surcharges {set_13} : attendu --set seed={seed_13} --set "
                            f"deterministic={str(det_13).lower()}.")
                    raise StopCheck
        finally:
            wbp.cli = cli_13
        if not CALLS_13:
            print("ℹ️ train_run_13 ne passe pas par wbp.cli : les arguments de ta commande ne sont pas vérifiés.")
        for first_13, second_13, what_13 in [("A", "B", "même graine, deux fois"), ("A", "C", "déterministe ou non"),
                                             ("A", "D", "graine 0 contre graine 1"), ("D", "E", "graine 1 contre graine 2")]:
            wa_13, wb_13 = RUNS_13[first_13]["weights"], RUNS_13[second_13]["weights"]
            print(f"  {first_13} contre {second_13} ({what_13}) : " + ("poids identiques au bit près"
                  if same_weights_13(wa_13, wb_13) else f"écart maximal {max_gap_13(wa_13, wb_13):.3g}"))
        accs_13 = np.array([RUNS_13[k]["val_acc"] for k in ("A", "D", "E")])
        half_13 = 4.303 * accs_13.std(ddof=1) / np.sqrt(3)                # t(0.975; 2), the table of 207.4
        print(f"accuracy sur 3 graines : {fr(100 * accs_13.mean(), 1)} % ± {fr(100 * half_13, 1)} (intervalle à 95 %, "
              f"t = 4,303 pour n = 3 ; valeurs {', '.join(fr(100 * a, 1) for a in accs_13)})")
        seeds_13 = not same_weights_13(RUNS_13["A"]["weights"], RUNS_13["D"]["weights"]) and \
            not same_weights_13(RUNS_13["D"]["weights"], RUNS_13["E"]["weights"])
        verdict("B7.13", seeds_13, "les graines 0, 1 et 2 donnent trois modèles différents : la graine va de la "
                "configuration jusqu'à l'entraînement.", "les graines 0, 1 et 2 donnent les mêmes poids : la graine "
                "n'arrive pas jusqu'à l'entraînement (train_run_13 passe-t-elle --set seed=… ? ta commande "
                "transmet-elle config.seed à train ?).")
        if not seeds_13:
            raise StopCheck
        wb.check("B7.13", same_weights_13(RUNS_13["A"]["weights"], RUNS_13["B"]["weights"]))

### Ex B7.14 — Suivi d'expériences maison : identifiant de run, configuration, métriques et artefacts en JSONL 📦 ★★ ⏱️ 30 min
**Objectif :** faire laisser à chaque run une ligne qui dit ce qui le définit, ce qu'il mesure et ce qu'il produit.  
**Prérequis :** Ex B7.11 · ch. 24 (journal d'expériences) · 🔁 R1 · fiche §207.4 (versions, empreinte), §207.5 · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

> **Mode d'emploi du paquet** : chaque fichier de ton paquet est une chaîne de la cellule à compléter (`DATA_PY = r'''...'''`) ; la cellule de vérification l'**écrit** dans ton projet (`mon_travail/b7_production/cifar-prod/`, à côté de ce notebook), puis l'importe ou le lance dans un autre processus. Pour corriger un fichier, corrige sa chaîne et réexécute les deux cellules. Si tu préfères un éditeur, modifie le fichier sur le disque : la cellule le voit et ne l'écrase plus (elle le dit).

**Le contrat.** Un nouveau fichier, `tracking.py` (la cellule suivante : exécute-la d'abord), et `cli.py` à **compléter dans la cellule de B7.10** ; réexécute ensuite la cellule de vérification de B7.10 (qui écrit aussi `tracking.py`, et vérifie que rien n'a cassé), puis celle-ci. Sur un projet neuf (une autre machine, un dossier supprimé), « Tout exécuter » arrive aux vérifications de B7.10 à B7.13 avant la cellule de `tracking.py` : elles l'attendent (⏳) ; une fois cette cellule exécutée, tout repasse.
- `tracking.py` : `new_run_id()` (la date et l'heure UTC puis six caractères hexadécimaux au hasard : `20261104-161731-e06339`, comme 207.5) ; `file_sha256(path)` ; `artifact(path)` rend `{"path": str(path), "bytes": sa taille, "sha256": file_sha256(path)}` ; `environment()` rend `{"python", "numpy", "torch", "platform", "commit", "dirty"}` (`platform.python_version()`, `importlib.metadata.version(...)`, `platform.platform()`, le commit court `git rev-parse --short HEAD` et si `git status --porcelain` n'est pas vide, `None` hors d'un dépôt git) ; `log_run(path, record)` ajoute **une** ligne JSON (clés triées) à la fin du journal, dont il crée le dossier ; `read_runs(path)` rend la liste des runs, du plus ancien au plus récent (`[]` sans fichier) ;
- `cli.py` : une option de plus, `--runs FILE` (défaut `runs/runs.jsonl`) ; sans `--out`, les poids vont dans `runs/<run_id>/` ; **chaque** run qui a commencé (après une configuration valide) laisse une ligne, réussi ou non (un `finally`), avec les champs `run_id`, `started` (la date ISO 8601 UTC), `status` (`"ok"` ou `"failed"`), `seconds`, `config` (`to_dict()`), `environment` ; un run réussi ajoute `data` (`{"path", "n_train", "sha256"}` : l'empreinte `fingerprint(X, y)` des images **et** des labels d'entraînement utilisés), `metrics` (les trois courbes de `history` et `val_acc_final`), `normalization` (`{"mean", "std"}`) et `artifacts` (la liste des `artifact(...)` des fichiers écrits : ici, les poids) ; un run raté garde son `error` (le type et le message de l'exception) ; le résumé JSON de stdout ajoute le `run_id`.

La cellule de vérification essaie `tracking.py` dans le notebook (trois identifiants, l'empreinte d'un fichier de 300 000 octets contre `hashlib`, l'environnement, deux lignes ajoutées puis relues), puis lance trois runs dans un même journal : 200 images, puis `n_train=99999` (plus que les données : code 1), puis 200 images avec la graine 1 sans `--out`. Elle relit le journal (trois lignes, la première jamais réécrite) et compare chaque champ à ce qu'elle sait : l'heure UTC de l'identifiant et de `started`, la configuration demandée, l'empreinte des 200 premières images et de leurs labels, le fichier de poids (taille et SHA-256, différent d'une graine à l'autre), la val_acc de stdout, l'erreur du run raté ; elle affiche enfin le journal par `pandas.json_normalize`.

Dans tes notes : six mois plus tard, quelle ligne du journal te dit avec quelles données, quel code et quels réglages le modèle livré a été entraîné ? Pourquoi garder les runs ratés ? Pourquoi l'empreinte porte-t-elle aussi sur les labels ? Que manque-t-il encore pour rejouer un run sur une autre machine (207.4) ?

In [ ]:
# Ex B7.14: tracking.py (the solutions notebook has written it with cli.py in Ex B7.10: the same text again)

# src/cifarprod/tracking.py
TRACKING_PY = r'''"""The run journal of cifarprod: one JSON line per run, appended to a JSONL file, never rewritten.

    run_id = new_run_id()                              # "20261104-161731-e06339"
    log_run(Path("runs/runs.jsonl"), {"run_id": run_id, "status": "ok", ...})
    runs = read_runs(Path("runs/runs.jsonl"))          # a list of dictionaries, oldest first

A run records what defines it (configuration, environment, data fingerprint), what it measures (metrics per epoch,
duration) and what it produces (its artifacts, with their path, size and fingerprint).
"""

from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import uuid
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path


def new_run_id() -> str:
    """A unique run identifier: the UTC date and time, then six random hexadecimal characters."""
    return f"{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:6]}"


def file_sha256(path: str | Path) -> str:
    """The SHA-256 of the bytes of a file (64 hexadecimal characters)."""
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def artifact(path: str | Path) -> dict:
    """{"path", "bytes", "sha256"} of a file written by the run."""
    path = Path(path)
    return {"path": str(path), "bytes": path.stat().st_size, "sha256": file_sha256(path)}


def _git(*args: str, cwd=None) -> str | None:
    try:
        done = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, timeout=10)
    except (OSError, subprocess.SubprocessError):
        return None
    return done.stdout.strip() if done.returncode == 0 else None


def environment(cwd=None) -> dict:
    """What runs the code: the versions of Python, NumPy and PyTorch, the platform, the git commit (None outside a
    repository) and whether the working tree has uncommitted changes."""
    versions = {}
    for name in ("numpy", "torch"):
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    status = _git("status", "--porcelain", cwd=cwd)
    return {"python": platform.python_version(), **versions, "platform": platform.platform(),
            "commit": _git("rev-parse", "--short", "HEAD", cwd=cwd), "dirty": bool(status) if status is not None else None}


def log_run(path: str | Path, record: dict) -> None:
    """Append one JSON line (keys sorted) to the journal ``path``; its folder is created if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, sort_keys=True) + "\n")


def read_runs(path: str | Path) -> list[dict]:
    """The runs of the journal, oldest first ([] if the file does not exist)."""
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
'''


import hashlib
import platform
from datetime import datetime, timedelta, timezone
from importlib import metadata

import pandas as pd

ok_14 = {}
with checked("B7.14"):
    wbp.write_files(PROJECT, {"src/cifarprod/tracking.py": TRACKING_PY, "src/cifarprod/cli.py": CLI_PY})
    tracking_14 = wbp.import_module(PROJECT, "tracking")
    ids_14 = [tracking_14.new_run_id() for _ in range(3)]
    today_14 = time.strftime("%Y%m%d", time.gmtime())
    ok_14["id"] = (all(isinstance(i, str) and re.fullmatch(r"\d{8}-\d{6}-[0-9a-f]{6}", i) for i in ids_14)
                   and len(set(ids_14)) == 3)
    if ok_14["id"]:
        stamp_14 = datetime.strptime(ids_14[0][:15], "%Y%m%d-%H%M%S").replace(tzinfo=timezone.utc)
        ok_14["id"] = abs((datetime.now(timezone.utc) - stamp_14).total_seconds()) < 120
    verdict("B7.14", ok_14["id"], f"new_run_id : {ids_14[0]}, {ids_14[1]}… (date et heure UTC, six caractères au hasard).",
            f"new_run_id : attendu trois identifiants distincts de la forme {today_14}-HHMMSS-xxxxxx, la date et "
            f"l'heure de l'UTC (datetime.now(timezone.utc), pas l'heure locale) ; reçu {ids_14!r:.120}.")
    blob_14 = scratch("blob") / "weights.bin"
    blob_14.write_bytes(np.random.default_rng(0).bytes(300_000))
    want_art_14 = {"path": str(blob_14), "bytes": 300_000, "sha256": hashlib.sha256(blob_14.read_bytes()).hexdigest()}
    ok_14["artifact"] = tracking_14.file_sha256(blob_14) == want_art_14["sha256"] and tracking_14.artifact(blob_14) == want_art_14
    verdict("B7.14", ok_14["artifact"], "file_sha256 et artifact d'un fichier de 300 000 octets : comme hashlib.",
            f"artifact(fichier de 300 000 octets) : attendu {{'path', 'bytes', 'sha256'}} = {want_art_14['bytes']} octets "
            f"et le SHA-256 de hashlib, reçu {tracking_14.artifact(blob_14)!r:.200}.")
    env_14 = tracking_14.environment()
    ok_14["environment"] = (isinstance(env_14, dict) and {"python", "numpy", "torch", "platform", "commit", "dirty"} <= set(env_14)
                            and env_14["python"] == platform.python_version() and env_14["torch"] == metadata.version("torch"))
    verdict("B7.14", ok_14["environment"], f"environment : Python {env_14.get('python')}, torch {env_14.get('torch')}, "
            f"commit {env_14.get('commit')!r}.", "environment doit rendre {python, numpy, torch, platform, commit, dirty} "
            f"avec platform.python_version() et importlib.metadata.version ; reçu {env_14!r:.200}.")
    journal_14 = scratch("journal") / "sub" / "runs.jsonl"
    tracking_14.log_run(journal_14, {"run_id": "a", "config": {"lr": 0.1}})
    first_14 = journal_14.read_text(encoding="utf-8")
    tracking_14.log_run(journal_14, {"run_id": "b", "metrics": [1, 2]})
    lines_14 = journal_14.read_text(encoding="utf-8").splitlines()
    ok_14["append"] = (len(lines_14) == 2 and journal_14.read_text(encoding="utf-8").startswith(first_14)
                       and [json.loads(l) for l in lines_14] == [{"run_id": "a", "config": {"lr": 0.1}}, {"run_id": "b", "metrics": [1, 2]}]
                       and tracking_14.read_runs(journal_14) == [json.loads(l) for l in lines_14]
                       and tracking_14.read_runs(journal_14.with_name("absent.jsonl")) == [])
    verdict("B7.14", ok_14["append"], "log_run ajoute une ligne JSON par run (dossier créé), read_runs les relit ([] sans fichier).",
            f"log_run puis read_runs : attendu deux lignes JSON, la première inchangée, relues dans l'ordre, et [] pour "
            f"un journal absent ; le fichier contient {lines_14!r:.200}.")

ready_14 = (written(lambda: wbp.import_module(PROJECT, "tracking").new_run_id())
            and written(lambda: wbp.import_module(PROJECT, "cli").build_parser())
            and written(lambda: wbp.import_module(PROJECT, "config").TrainConfig()))
if not ready_14:
    print("⏳ Ex B7.14 : le journal des runs attend tracking.py, et ta commande et ta configuration (B7.10, B7.11).")
elif later_files_pending("B7.14"):
    ok_14["later files"] = False
else:
    with checked("B7.14"):                                 # the command line logs every run
        out_14 = scratch("runs")
        journal_14 = out_14 / "runs.jsonl"
        r1_14 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--out", out_14 / "a", "--runs", journal_14)
        ok_14["run 1"] = run_line("B7.14", r1_14, r1_14.ok, "code 0.", "attendu le code 0")
        first_lines_14 = journal_14.read_text(encoding="utf-8").splitlines() if journal_14.is_file() else []
        first_line_14 = first_lines_14[0] if first_lines_14 else None
        r2_14 = wbp.cli(PROJECT, "train", "--set", "n_train=99999", "--runs", journal_14)
        ok_14["run 2"] = run_line("B7.14", r2_14, r2_14.returncode == 1, "code 1 (n_train plus grand que les données).",
                                  "attendu le code 1 : n_train = 99 999 dépasse les 2 000 images")
        r3_14 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--set", "seed=1", "--runs", journal_14)
        ok_14["run 3"] = run_line("B7.14", r3_14, r3_14.ok, "code 0, sans --out.", "attendu le code 0")
        lines_14 = journal_14.read_text(encoding="utf-8").splitlines() if journal_14.is_file() else []
        runs_14 = []
        for line_14 in lines_14:
            try:
                record_14 = json.loads(line_14)
            except ValueError:
                record_14 = None
            runs_14.append(record_14 if isinstance(record_14, dict) else {"status": f"ligne illisible : {line_14[:60]!r}"})
        ok_14["three"] = [r.get("status") for r in runs_14] == ["ok", "failed", "ok"] and lines_14[0] == first_line_14
        verdict("B7.14", ok_14["three"], "trois lignes dans le journal, statuts ok, failed, ok ; la première jamais réécrite.",
                f"attendu trois lignes de statuts ['ok', 'failed', 'ok'], la première inchangée ; reçu "
                f"{[r.get('status') for r in runs_14]}.")
        if not ok_14["three"]:
            raise StopCheck
        keys_14 = {"run_id", "started", "status", "seconds", "config", "environment", "data", "metrics", "normalization",
                   "artifacts"}
        ok_rec_14 = all(keys_14 <= set(r) for r in (runs_14[0], runs_14[2])) and {"run_id", "status", "seconds", "config",
                                                                                   "error"} <= set(runs_14[1])
        verdict("B7.14", ok_rec_14, "chaque ligne a les champs du contrat (un run raté : son erreur).",
                f"champs manquants : run réussi {sorted(keys_14 - set(runs_14[0]))}, run raté "
                f"{sorted({'run_id', 'status', 'seconds', 'config', 'error'} - set(runs_14[1]))}.")
        if not ok_rec_14:
            raise StopCheck
        rec_14 = runs_14[0]
        ok_14["ids"] = len({r["run_id"] for r in runs_14}) == 3 and r1_14.last_json().get("run_id") == rec_14["run_id"] \
            and r3_14.last_json().get("run_id") == runs_14[2]["run_id"]
        verdict("B7.14", ok_14["ids"], "trois identifiants distincts, ceux des lignes JSON de stdout.",
                "attendu trois run_id distincts, et sur stdout le run_id de la ligne du journal.")
        ok_14["config"] = rec_14["config"] == {**DEFAULTS, "n_train": 200} and runs_14[2]["config"] == {**DEFAULTS, "n_train": 200, "seed": 1}
        verdict("B7.14", ok_14["config"], "la configuration complète du run (to_dict), surcharges comprises.",
                f"config du premier run : attendu {{**DEFAULTS, 'n_train': 200}}, reçu {rec_14['config']!r:.200}.")
        want_sha_14 = wbp.array_fingerprint(SUBSET["x_train"][:200], SUBSET["y_train"][:200])
        ok_14["data"] = rec_14["data"].get("sha256") == want_sha_14 and rec_14["data"].get("n_train") == 200
        verdict("B7.14", ok_14["data"], f"l'empreinte des 200 images d'entraînement et de leurs labels : {want_sha_14[:12]}….",
                f"data : attendu n_train 200 et sha256 = fingerprint(X[:200], y[:200]) des images ET des labels utilisés, "
                f"reçu {rec_14['data']!r:.200}.")
        arts_14 = rec_14["artifacts"]
        art_path_14 = (PROJECT / arts_14[0]["path"]).resolve() if arts_14 else None
        ok_14["artifacts"] = (len(arts_14) >= 1 and art_path_14 == (out_14 / "a" / "model.safetensors").resolve()
                              and arts_14[0].get("sha256") == hashlib.sha256(art_path_14.read_bytes()).hexdigest()
                              and arts_14[0].get("bytes") == art_path_14.stat().st_size)
        verdict("B7.14", ok_14["artifacts"], "artifacts : le fichier des poids, sa taille et son SHA-256.",
                f"artifacts : attendu [{{path, bytes, sha256}}] du fichier <out>/model.safetensors, reçu {arts_14!r:.200}.")
        arts_3_14 = runs_14[2]["artifacts"]
        ok_14["seed"] = bool(arts_14 and arts_3_14) and arts_14[0].get("sha256") != arts_3_14[0].get("sha256")
        verdict("B7.14", ok_14["seed"], "les runs de graines 0 et 1 ont écrit des poids différents.",
                "les runs de graines 0 et 1 ont écrit les mêmes poids (même SHA-256) : la configuration enregistrée "
                "(seed 1) ne dit pas ce qui a été entraîné ; ta commande transmet-elle config.seed à train ?")
        try:
            started_14 = datetime.fromisoformat(str(rec_14["started"]))
            ok_14["started"] = (started_14.utcoffset() == timedelta(0)
                                and abs((datetime.now(timezone.utc) - started_14).total_seconds()) < 600)
        except (TypeError, ValueError):
            ok_14["started"] = False
        verdict("B7.14", ok_14["started"], f"started : {rec_14['started']} (ISO 8601, UTC).",
                f"started : attendu la date ISO 8601 en UTC, avec son fuseau (datetime.now(timezone.utc).isoformat()), "
                f"reçu {rec_14['started']!r}.")
        default_out_14 = (PROJECT / runs_14[2]["artifacts"][0]["path"]).resolve() if runs_14[2]["artifacts"] else None
        ok_14["default out"] = default_out_14 == (PROJECT / "runs" / runs_14[2]["run_id"] / "model.safetensors").resolve()
        verdict("B7.14", ok_14["default out"], "sans --out, les poids vont dans runs/<run_id>/.",
                f"sans --out, attendu runs/<run_id>/model.safetensors, reçu {default_out_14}.")
        metrics_14 = rec_14["metrics"]
        ok_14["metrics"] = (len(metrics_14.get("val_acc", [])) == 1 and metrics_14.get("val_acc_final") == r1_14.last_json().get("val_acc")
                            and np.allclose(rec_14["normalization"]["mean"], (SUBSET["x_train"][:200] / 255).mean(axis=(0, 1, 2))))
        verdict("B7.14", ok_14["metrics"], "metrics (une valeur par epoch, val_acc_final) et la normalisation apprise.",
                "metrics : attendu train_loss, val_loss, val_acc (une valeur par epoch) et val_acc_final, la val_acc de "
                "stdout ; normalization : la moyenne et l'écart-type des images d'entraînement.")
        ok_14["failed"] = "n_train" in runs_14[1].get("error", "")
        verdict("B7.14", ok_14["failed"], f"le run raté garde son erreur : « {runs_14[1].get('error', '')[:70]} ».",
                f"le run raté doit garder son erreur (qui nomme n_train) ; reçu {runs_14[1].get('error')!r}.")
        TABLE_14 = pd.json_normalize(runs_14)[["run_id", "status", "config.seed", "config.n_train", "metrics.val_acc_final",
                                               "seconds"]]
        print(TABLE_14.to_string(index=False))
if ok_14 and all(ok_14.values()) and len(ok_14) >= 17:
    print("✅ Ex B7.14 : chaque run laisse sa ligne : ce qui le définit, ce qu'il mesure, ce qu'il produit.")

### Ex B7.15 — Tests pytest pour le ML : formes, overfitting d'un mini-batch, prétraitement, schéma des données 📦 ★★★ ⏱️ 45 min
**Objectif :** écrire les tests qui protègent un projet de ML, et vérifier qu'ils attrapent de vraies erreurs.  
**Prérequis :** Ex B7.11 (et B7.14) · ch. 21 (tests paramétrés, 21.16) · fiche §207.6 (« Quatre familles de tests ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

La fiche donne en entier deux tests du modèle et sa fixture : ils sont dans `TEST_MODEL_PY`, avec les fixtures de `CONFTEST_PY` (`model`, le CNN de la fiche ; `data_file`, le chemin de `data/cifar10_2000.npz`, qui fait sauter le test sans données, comme dans l'intégration continue ; `synthetic_file`, un petit `.npz` au même schéma, 96 images d'entraînement et 32 de validation). **À toi** `tests/test_data.py` et `tests/test_cli.py`, avec **au moins 8 tests** (les cas d'un `parametrize` comptent chacun) dans ces deux fichiers, qui couvrent :
- le **prétraitement** d'une image constante (tout à 0, tout à 255 : la valeur attendue de chaque canal se calcule à la main) et l'ordre des canaux ;
- le **schéma des données** : types, formes, labels de 0 à 9, chaque classe présente, et le refus d'un fichier mal formé ;
- **aucun recouvrement** entre l'entraînement et la validation, par l'empreinte de chaque image ;
- une **configuration invalide refusée** (au moins `epochs=0`, un learning rate hors de $]0, 1[$, une clé inconnue) ;
- le **code de sortie** de `main(argv)`, appelé dans le test (2 pour une configuration invalide, 1 sans données) ;
- un **entraînement fumée** de bout en bout, marqué `@pytest.mark.slow`, sur `synthetic_file` : code 0, le fichier de poids écrit, et le run dans le journal si tu as fait B7.14.

Chaque `assert` a un message dont la **première ligne dit ce qui était attendu** (`f"expected ..., got ..."`), comme dans la fiche : quand l'intégration continue échoue, c'est tout ce que tu lis. Passe `--runs` et `--out` vers `tmp_path` (la fixture de pytest) pour ne rien écrire dans le projet. Si tu reviens après B7.18, ajoute un test de l'API (une réponse 422 pour une image de 28 × 28).

La cellule de vérification écrit tes tests dans `tests/`, les lance par pytest sur ton paquet, fichier par fichier (le nom de chaque test en échec, puis sa raison), vérifie qu'ils n'ont rien écrit dans le projet, et compte tes tests et ceux marqués `slow`. Puis elle les lance sur le **paquet de référence** du chapitre (`solutions/b7_cifarprod`, qui suit le contrat de B7.9 à B7.14 : un test qui y échoue exige autre chose que le contrat) et sur **six versions fautives** de ce paquet, chacune avec une erreur que tes tests doivent voir : le prétraitement qui ne divise plus par 255, `load_npz` qui rend des floats sans vérifier le schéma, une validation qui recouvre l'entraînement, une configuration qui accepte un learning rate hors de $]0, 1[$, `main` qui rend 0 sur une configuration invalide, et une commande `train` qui rend 0 sans écrire ses poids (vue seulement par un test `slow`). Pour chacune, elle dit quel test l'a attrapée, et si la première ligne de son message dit ce qui était attendu.

Dans tes notes : lequel de tes tests attrape le plus de versions fautives, et pourquoi ? Pourquoi un test qui ne fait que « ça ne plante pas » n'en attrape-t-il aucune ? Que verrait l'overfitting d'un mini-batch que les autres ne voient pas (pense à un `zero_grad` oublié) ? Pourquoi l'entraînement fumée est-il marqué `slow`, et quand le lancerais-tu quand même ?

In [ ]:
# Ex B7.15: the tests of the reference package

# tests/conftest.py
CONFTEST_PY = r'''"""Fixtures of the tests of cifarprod: a fresh model, the data file of the project, a small synthetic data file."""

from pathlib import Path

import numpy as np
import pytest
import torch

from cifarprod.data import synthetic
from cifarprod.model import build_model

DATA = Path(__file__).resolve().parents[1] / "data" / "cifar10_2000.npz"


@pytest.fixture
def model():
    torch.manual_seed(0)
    return build_model(widths=(12, 24))


@pytest.fixture
def data_file():
    """The CIFAR-10 subset of the project; the tests that need it are skipped without it (in the CI, for example)."""
    if not DATA.is_file():
        pytest.skip(f"no data file {DATA}")
    return DATA


@pytest.fixture
def synthetic_file(tmp_path):
    """A small .npz file with the schema of the data (96 training and 32 validation random images)."""
    X, y = synthetic(96, seed=0)
    X_val, y_val = synthetic(32, seed=1)
    path = tmp_path / "synthetic.npz"
    np.savez(path, x_train=X, y_train=y, x_val=X_val, y_val=y_val)
    return path
'''

# tests/test_model.py
TEST_MODEL_PY = r'''"""Tests of the model: the shape of the output, and a mini-batch learned by heart (fiche, 207.6)."""

import pytest
import torch
import torch.nn.functional as F

from cifarprod.model import build_model


@pytest.mark.parametrize("batch", [1, 7])
def test_output_shape(model, batch):
    out = model(torch.zeros(batch, 3, 32, 32))
    assert out.shape == (batch, 10), f"expected ({batch}, 10), got {tuple(out.shape)}"


def test_overfits_one_batch(model):
    g = torch.Generator().manual_seed(0)
    x, y = torch.randn(16, 3, 32, 32, generator=g), torch.arange(16) % 10
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(100):
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
    assert loss.item() < 0.05, f"expected a loss < 0.05 on 16 memorised images, got {loss.item():.3f}"


@pytest.mark.parametrize("arch", ["cnn", "vgg"])
def test_both_archs_give_ten_logits(arch):
    out = build_model(arch, widths=(8, 16))(torch.zeros(3, 3, 32, 32))
    assert out.shape == (3, 10), f"expected (3, 10) for {arch}, got {tuple(out.shape)}"
'''

# tests/test_data.py
TEST_DATA_PY = r'''"""Tests of the data: the preprocessing on a constant image, the schema of the data file, no image shared by the
training and the validation parts, the fingerprint."""

import numpy as np
import pytest
import torch

from cifarprod.data import channel_stats, fingerprint, load_npz, preprocess

MEAN, STD = (0.5, 0.4, 0.3), (0.25, 0.2, 0.1)


@pytest.mark.parametrize("value", [0, 255])
def test_constant_image_preprocessed(value):
    x = preprocess(np.full((32, 32, 3), value, dtype=np.uint8), MEAN, STD)
    want = torch.tensor([(value / 255 - m) / s for m, s in zip(MEAN, STD)]).view(1, 3, 1, 1).expand(1, 3, 32, 32)
    assert x.shape == (1, 3, 32, 32), f"expected the shape (1, 3, 32, 32), got {tuple(x.shape)}"
    assert x.dtype == torch.float32, f"expected float32, got {x.dtype}"
    assert torch.allclose(x, want, atol=1e-6), f"expected {want[0, :, 0, 0].tolist()} per channel, got {x[0, :, 0, 0].tolist()}"


def test_channels_stay_in_rgb_order():
    image = np.zeros((32, 32, 3), dtype=np.uint8)
    image[..., 0] = 255                                   # a red image
    x = preprocess(image, (0, 0, 0), (1, 1, 1))
    assert x[0, 0].min() == 1 and x[0, 1:].max() == 0, f"expected red in channel 0 only, got the means {x[0].mean(dim=(1, 2)).tolist()}"


@pytest.mark.parametrize("part, n", [("train", 2000), ("val", 500)])
def test_data_schema(data_file, part, n):
    X, y = load_npz(data_file, part)
    assert X.dtype == np.uint8, f"expected uint8 images, got {X.dtype}"
    assert X.shape == (n, 32, 32, 3), f"expected the shape ({n}, 32, 32, 3), got {X.shape}"
    assert y.shape == (n,) and np.issubdtype(y.dtype, np.integer), f"expected {n} integer labels, got {y.shape} {y.dtype}"
    assert set(np.unique(y)) == set(range(10)), f"expected the labels 0 to 9, got {sorted(set(np.unique(y)))}"


def test_no_image_in_both_parts(data_file):
    train = {fingerprint(image) for image in load_npz(data_file, "train")[0]}
    shared = sum(fingerprint(image) in train for image in load_npz(data_file, "val")[0])
    assert shared == 0, f"expected no validation image in the training part, got {shared}"


@pytest.mark.parametrize("bad", ["float", "label", "shape"])
def test_load_npz_refuses_a_bad_schema(tmp_path, bad):
    X, y = np.zeros((4, 32, 32, 3), np.uint8), np.arange(4)
    X = X.astype(np.float32) if bad == "float" else X[:, :28] if bad == "shape" else X
    y = y + 7 if bad == "label" else y
    np.savez(tmp_path / "bad.npz", x_train=X, y_train=y, x_val=X, y_val=y)
    with pytest.raises(ValueError):
        load_npz(tmp_path / "bad.npz", "train")


def test_fingerprint_sees_one_pixel():
    X = np.zeros((2, 32, 32, 3), dtype=np.uint8)
    Y = X.copy()
    Y[1, 5, 5, 2] = 1
    assert fingerprint(X) != fingerprint(Y), "expected another fingerprint when one pixel changes"
    assert fingerprint(X) != fingerprint(X.astype(np.int64)), "expected another fingerprint for another dtype"


def test_channel_stats_on_training_images():
    rng = np.random.default_rng(0)
    X = rng.integers(0, 256, (10, 32, 32, 3), dtype=np.uint8)
    mean, std = channel_stats(X)
    want = (X / 255).mean(axis=(0, 1, 2)), (X / 255).std(axis=(0, 1, 2))
    assert np.allclose(mean, want[0]) and np.allclose(std, want[1]), f"expected {want}, got {(mean, std)}"
'''

# tests/test_cli.py
TEST_CLI_PY = r'''"""Tests of the configuration and of the command line: invalid settings refused, exit codes, a smoke training."""

import json

import pytest
from safetensors.torch import load_file

from cifarprod.cli import main
from cifarprod.config import load_config


@pytest.mark.parametrize("override", ["epochs=0", "lr=0", "lr=5", "lr=1e-3", "lerning_rate=0.01", "widths=[]",
                                      "arch=resnet", "batch_size=true", "seed"])
def test_invalid_config_refused(override):
    with pytest.raises(ValueError):
        load_config(None, [override])


def test_overrides_win_over_the_file(tmp_path):
    path = tmp_path / "c.yaml"
    path.write_text("lr: 0.1\nepochs: 3\nwidths: [4, 8]\n", encoding="utf-8")
    config = load_config(path, ["lr=0.01", "epochs=2"])
    assert (config.lr, config.epochs, config.widths) == (0.01, 2, (4, 8)), f"expected (0.01, 2, (4, 8)), got {(config.lr, config.epochs, config.widths)}"


def test_exit_code_of_an_invalid_config(tmp_path):
    code = main(["train", "--set", "epochs=0", "--runs", str(tmp_path / "runs.jsonl")])
    assert code == 2, f"expected the exit code 2 for epochs=0, got {code}"


def test_exit_code_without_data(tmp_path):
    code = main(["train", "--set", f"data={tmp_path / 'absent.npz'}", "--runs", str(tmp_path / "runs.jsonl")])
    assert code == 1, f"expected the exit code 1 without data file, got {code}"


@pytest.mark.slow
def test_smoke_train(tmp_path, synthetic_file):
    runs = tmp_path / "runs.jsonl"
    code = main(["train", "--set", f"data={synthetic_file}", "--set", "n_train=96", "--set", "widths=[4, 8]",
                 "--out", str(tmp_path / "run"), "--runs", str(runs)])
    assert code == 0, f"expected the exit code 0, got {code}"
    weights = tmp_path / "run" / "model.safetensors"
    assert weights.is_file(), f"expected the weights in {weights}"
    assert "0.weight" in load_file(weights), f"expected the weights of the model, got {sorted(load_file(weights))[:4]}"
    record = json.loads(runs.read_text(encoding="utf-8").splitlines()[-1])
    assert record["status"] == "ok", f"expected a run logged with the status ok, got {record['status']!r}"
'''


ok_15 = {}
PYTEST_ARGS_15 = ["-c", str(PROJECT / "pyproject.toml"), "--rootdir", str(PROJECT), "--strict-markers"]
ENV_15 = {"PYTHONPATH": str(PROJECT / "src")}
TESTS_15 = {"tests/conftest.py": CONFTEST_PY, "tests/test_model.py": TEST_MODEL_PY, "tests/test_data.py": TEST_DATA_PY,
            "tests/test_cli.py": TEST_CLI_PY}


def project_files_15():
    """{file of the project (caches excepted), relative to it: its modification time}."""
    return {str(p.relative_to(PROJECT)): p.stat().st_mtime_ns for p in PROJECT.rglob("*")
            if p.is_file() and not {"__pycache__", ".pytest_cache"} & set(p.parts)}


if TEST_DATA_PY is ... or TEST_CLI_PY is ...:
    print("⏳ Ex B7.15 : écris d'abord TEST_DATA_PY et TEST_CLI_PY.")
elif later_files_pending("B7.15"):
    pass
else:
    with checked("B7.15"):
        wbp.write_files(PROJECT, TESTS_15)
        before_15 = project_files_15()
        own_15 = {}
        for rel_15 in ("tests/test_model.py", "tests/test_data.py", "tests/test_cli.py"):
            print(f"— {rel_15}, sur ton paquet :")
            own_15[rel_15] = run_pytest_group(rel_15, root=PROJECT, env=ENV_15, args=PYTEST_ARGS_15)
        ok_15["own"] = all(r.ok for r in own_15.values())
        verdict("B7.15", ok_15["own"], "tes tests passent sur ton paquet.", "tes tests doivent d'abord passer sur ton "
                "paquet (les lignes ci-dessus) ; si c'est ton paquet qui a tort, corrige-le dans sa cellule.")
        new_15 = sorted(rel for rel, stamp in project_files_15().items() if before_15.get(rel) != stamp)
        ok_15["clean"] = not new_15
        verdict("B7.15", ok_15["clean"], "tes tests n'écrivent rien dans le projet (tmp_path).",
                f"tes tests ont écrit dans le projet : {new_15[:3]}{' …' if len(new_15) > 3 else ''} : passe --out et "
                "--runs vers tmp_path.")
        collected_15 = {}
        for marker_15 in ("", "slow"):
            listing_15 = wbp.run([sys.executable, "-m", "pytest", "tests/test_data.py", "tests/test_cli.py", "--collect-only",
                                  "-q", "-p", "no:cacheprovider", *PYTEST_ARGS_15, *(["-m", marker_15] if marker_15 else [])],
                                 PROJECT, src=[PROJECT / "src"], env={"COLUMNS": "1000"})
            collected_15[marker_15] = sum("::" in line for line in listing_15.stdout.splitlines())
        ok_15["count"] = collected_15[""] >= 8 and collected_15["slow"] >= 1
        verdict("B7.15", ok_15["count"], f"{collected_15['']} tests dans test_data.py et test_cli.py (paramètres compris), "
                f"dont {collected_15['slow']} marqué(s) slow.", f"{collected_15['']} tests dans test_data.py et test_cli.py, "
                f"{collected_15['slow']} marqué(s) slow : attendu au moins 8 tests (paramètres compris), dont un marqué slow.")
        if not ok_15["own"]:
            raise StopCheck
        base_15 = scratch("mutants")
        reference_15 = wbp.run_tests(wbp.mutant_package(None, base_15 / "reference"), PROJECT / "tests", data=DATA_FILE)
        ok_15["reference"] = reference_15.ok
        verdict("B7.15", reference_15.ok, f"tes tests passent aussi sur le paquet de référence ({reference_15.summary}).",
                f"tes tests doivent passer sur le paquet de référence, qui suit le contrat ({reference_15.summary}) :")
        for name_15, reason_15 in reference_15.failures[:6]:
            print(f"   ❌ {name_15}\n      {reason_15[:300]}")
        if not reference_15.ok:
            print("   un test qui échoue ici exige plus (ou autre chose) que le contrat de B7.9 à B7.14.")
            raise StopCheck
        for k_15, mutant_15 in enumerate(wbp.MUTANTS, start=1):
            package_15 = wbp.mutant_package(mutant_15.key, base_15 / mutant_15.key)
            run_15 = wbp.run_tests(package_15, PROJECT / "tests", data=DATA_FILE,
                                   marker="slow" if mutant_15.slow else "not slow", maxfail=1)
            if run_15.failed + run_15.errors == 0:
                ok_15[mutant_15.key] = False
                print(f"❌ Ex B7.15 : version fautive {k_15} ({mutant_15.what}) : elle passe tous tes tests"
                      + (" marqués slow" if mutant_15.slow else "") + f" ({run_15.summary}). Quel test aurait dû la voir ?")
                continue
            name_15, reason_15 = run_15.failures[0] if run_15.failures else ("?", "")
            ok_15[mutant_15.key] = wbp.says_expected(reason_15)
            verdict("B7.15", ok_15[mutant_15.key], f"version fautive {k_15} ({mutant_15.what}) : attrapée par {name_15} "
                    f"(« {reason_15[:110]} »).", f"version fautive {k_15} ({mutant_15.what}) : attrapée par {name_15}, "
                    f"mais la première ligne de son message ne dit pas ce qui était attendu : « {reason_15[:160]} ».")
if ok_15 and all(ok_15.values()) and len(ok_15) == 4 + len(wbp.MUTANTS):
    print(f"✅ Ex B7.15 : tes tests passent sur un paquet juste et attrapent les {len(wbp.MUTANTS)} versions fautives.")

## Partie B · Le service : les écarts entre entraînement et production, l'artefact, l'API, la latence et l'export

Le modèle sort maintenant du projet pour servir. Tu commences par réparer le service d'un collègue, dont le modèle marche dans son notebook et pas en production (B7.16) : cinq écarts entre la préparation de l'entraînement et celle du service, le training-serving skew du ch. 12. Le modèle du collègue est le VGG 16-32-64 livré par la solution du MP5 (`model.pt` et `model.json`, versionnés avec le dépôt), entraîné sur 45 000 images : la partie B sert un vrai modèle, pas le petit CNN de la partie A. Puis tu le livres : un artefact qui se suffit (B7.17), une API qui le sert (B7.18), dont tu prédis puis mesures la latence (B7.19) et que tu accélères sur CPU (B7.20), et un export qui se recharge sans ton code (B7.21). Le modèle servi est le même VGG, `MODEL_B` (`SOURCE_B` dans la cellule donnée de B7.17).

### Ex B7.16 — Le modèle qui marche dans le notebook et pas en production : cinq écarts à retrouver 🐛 ★★ ⏱️ 30 min
**Objectif :** retrouver, par des diagnostics enchaînés, les écarts entre la préparation de l'entraînement et celle d'un service.  
**Prérequis :** Ex B7.15 · ch. 12 (training-serving skew) · ch. 20 (`eval()`) · fiche §207.7 (« L'artefact du modèle »), §207.10 · **Fil rouge :** CIFAR-10 · **Parcours :** C

Un collègue a entraîné le VGG du MP5 dans son notebook : 85 % sur ces images de test. Il l'a ensuite emballé en service : `save_artifact_16` écrit l'artefact à la fin de l'entraînement ; le service le recharge (`load_service_16`), lit chaque image envoyée par un client (`read_upload_16`, sur un fichier PNG), la prépare (`preprocess_16`) et répond le nom français de la classe (`predict_16`). **Ce que faisait son notebook**, la référence : des images `uint8` RGB `(N, 32, 32, 3)`, divisées par 255, normalisées canal par canal par la moyenne et l'écart-type de **ses images d'entraînement** (`INFO_16["mean"]`, `INFO_16["std"]`), en `float32`, le modèle en mode `eval()`, et la sortie $k$ du modèle est la classe `CLASSES[k]`. En production, le service répond n'importe quoi. **Répare son code sur place** (mêmes noms, mêmes signatures, les docstrings sont le contrat) : cinq écarts s'y cachent, et un écart peut en cacher un autre.

La cellule donnée prépare le modèle du collègue (`MODEL_16`, `INFO_16`, une copie neuve à chaque appel de `wbp.reference_model()`), 200 images de test (`X_16`, `Y_16`) écrites en PNG comme les enverrait un client (`PATHS_16`), ce que le notebook du collègue répond pour elles (`WANT_16`), et les cinq diagnostics. Chaque diagnostic écrit un **nouvel** artefact avec ton `save_artifact_16`, le recharge avec ton `load_service_16`, et ne regarde qu'une partie du service : 1. le service répond-il, et en quel type ; 2. l'image lue d'un fichier ; 3. le prétraitement, sur des images bien lues ; 4. le modèle rechargé, sur des images bien préparées ; 5. les noms rendus pour les 200 images, comparés à ceux du notebook (un **test de non-régression**, ch. 7), puis pour des copies de ces images sous d'autres noms, et pour un modèle dont les sorties sont rangées dans un autre ordre. Ils s'enchaînent : le premier qui échoue arrête les suivants (⏳). La cellule suivante montre le service tel qu'il est sur 8 images, puis son accuracy sur les 200.

Dans tes notes : pour chaque écart, le symptôme, la cause, la ligne corrigée, et le test qui l'aurait attrapé avant la mise en production. Pourquoi certains écarts ne faisaient-ils **pas** planter le service ? Lequel en cachait un autre, et pourquoi (pense à ce que fait une normalisation par les statistiques du batch à des pixels de 0 à 255) ? Qu'aurait-il fallu mettre dans l'artefact dès le départ (207.7) ?

In [ ]:
# The colleague's model (the VGG 16-32-64 delivered by the MP5 solution, trained in their notebook on uint8 RGB images
# divided by 255, then normalized with the mean and the std of its training images, in float32; output k = CLASSES[k]),
# 200 test images saved as the PNG files a client uploads, what the evaluation of the notebook gives on them, and the
# five diagnoses, in a row
from safetensors.torch import load_file, save_file

MODEL_16, INFO_16 = wbp.reference_model()               # a new copy at each call: the diagnoses never share it
X_16, Y_16 = wbp.test_images(20)                         # 200 test images (20 per class), uint8 RGB (200, 32, 32, 3)
UPLOADS_16 = scratch("uploads")
PATHS_16 = [wbp.write_png(image, UPLOADS_16 / f"upload_{i:03d}.png") for i, image in enumerate(X_16)]
MEAN_16, STD_16 = np.asarray(INFO_16["mean"], np.float32), np.asarray(INFO_16["std"], np.float32)
FRENCH_16 = {"airplane": "avion", "automobile": "automobile", "bird": "oiseau", "cat": "chat", "deer": "cerf",
             "dog": "chien", "frog": "grenouille", "horse": "cheval", "ship": "bateau", "truck": "camion"}


def offline_input_16(images):
    """The input of the model in the colleague's notebook: uint8 RGB (N, 32, 32, 3) -> float32 (N, 3, 32, 32)."""
    x = (np.asarray(images, np.float32) / np.float32(255) - MEAN_16) / STD_16
    return torch.from_numpy(np.ascontiguousarray(x.transpose(0, 3, 1, 2)))


with torch.inference_mode():
    OFFLINE_16 = MODEL_16(offline_input_16(X_16)).argmax(dim=1).numpy()      # the evaluation of the notebook
WANT_16 = [FRENCH_16[CLASSES[k]] for k in OFFLINE_16]                         # what the service should answer
print(f"Dans le notebook du collègue : accuracy {fr((OFFLINE_16 == Y_16).mean(), 3)} sur ces 200 images de test.")


def fresh_service_16(model=None, classes=CLASSES):
    """(model, meta) of the service, from a NEW artifact written by YOUR save_artifact_16 (the colleague's model, or
    `model` whose outputs are the classes `classes`), loaded by YOUR load_service_16."""
    folder = scratch("artifact")
    save_artifact_16(model if model is not None else wbp.reference_model()[0], folder, list(INFO_16["mean"]),
                     list(INFO_16["std"]), list(classes))
    found = sorted(p.name for p in folder.iterdir())
    if found != ["meta.json", "model.safetensors"]:
        raise RuntimeError(f"l'artefact contient {found} : attendu model.safetensors (les poids, sans pickle) et "
                           "meta.json, comme le dit la docstring de save_artifact_16")
    return load_service_16(folder)


def diagnose_16(ex_id="B7.16"):
    """The five diagnoses of the service, in a row: the first one that fails stops the chain (one gap can hide
    another). Returns True when the five pass."""

    def point1():                                         # does the service answer?
        model, meta = fresh_service_16()
        seen = []
        handle = model.register_forward_pre_hook(lambda module, args: seen.append(args[0].dtype))
        try:
            names = predict_16(model, meta, PATHS_16[:8])
        except Exception as error:  # noqa: BLE001 - the learner's code: the symptom
            return False, f"le service plante sur 8 images envoyées : {type(error).__name__} : {str(error)[:160]}", \
                "lis la dernière ligne de l'erreur : quel tenseur ne va pas avec les poids du modèle ?"
        finally:
            handle.remove()
        if not (isinstance(names, list) and len(names) == 8 and all(isinstance(n, str) for n in names)):
            return False, f"predict_16 rend {type(names).__name__} {str(names)[:80]}", "attendu une liste de 8 noms (str)"
        weights = {p.dtype for p in model.parameters()}
        if weights != {torch.float32} or set(seen) != {torch.float32}:
            return False, (f"le modèle du service a des poids {sorted(map(str, weights))} et reçoit {[str(d) for d in seen]}, "
                           "alors que l'artefact et l'entraînement sont en float32"), "où ce type apparaît-il ?"
        return True, "le service répond 8 noms, en float32 comme l'entraînement", ""

    def point2():                                         # the image read from the file
        image = read_upload_16(PATHS_16[0])
        got = np.asarray(image)
        want = X_16[0].astype(np.float32) / 255
        if got.shape != (32, 32, 3):
            return False, f"l'image lue a la forme {got.shape}", "attendu (32, 32, 3), comme à l'entraînement"
        got = got.astype(np.float64)
        if np.allclose(got, want, atol=1e-6):
            return True, "l'image lue est celle de l'entraînement : RGB, des floats dans [0, 1]", ""
        symptoms, scaled = [], got
        if got.max() > 1.5:
            symptoms.append(f"ses valeurs vont de {got.min():.0f} à {got.max():.0f}, alors que l'entraînement voit des "
                            "pixels dans [0, 1]")
            scaled = got / 255
        if not np.allclose(scaled, want, atol=1e-6) and np.allclose(scaled, want[..., ::-1], atol=1e-6):
            symptoms.append("son canal 0 est le canal 2 de l'image envoyée (le bleu à la place du rouge)")
        elif not np.allclose(scaled, want, atol=1e-6):
            symptoms.append(f"elle s'écarte de l'image envoyée de {np.abs(scaled - want).max():.3f}")
        return False, "l'image lue n'est pas celle que voit l'entraînement : " + " ; ".join(symptoms), \
            "compare-la à X_16[0] / 255 : que rend la fonction de lecture ?"

    def point3():                                         # the preprocessing
        model, meta = fresh_service_16()
        images = X_16[:8].astype(np.float32) / 255
        batch = torch.as_tensor(preprocess_16(images.copy(), meta)).double()
        if tuple(batch.shape) != (8, 3, 32, 32):
            return False, f"preprocess_16 de 8 images rend la forme {tuple(batch.shape)}", "attendu (8, 3, 32, 32)"
        alone = torch.as_tensor(preprocess_16(images[:1].copy(), meta)).double()
        want = offline_input_16(X_16[:8]).double()
        if not torch.allclose(alone[0], batch[0], atol=1e-5):
            return False, (f"une image prétraitée seule n'a pas les mêmes valeurs que dans un batch de 8 (écart "
                           f"{float((alone[0] - batch[0]).abs().max()):.3f}) : son prétraitement dépend des autres images "
                           "envoyées"), "d'où viennent la moyenne et l'écart-type qu'utilise le service ?"
        if not torch.allclose(batch, want, atol=1e-4):
            return False, (f"le prétraitement du service s'écarte de celui de l'entraînement de "
                           f"{float((batch - want).abs().max()):.3f}"), "relis comment le notebook préparait les images"
        return True, "le prétraitement est celui de l'entraînement, image par image", ""

    def point4():                                         # the model of the service
        model, meta = fresh_service_16()
        x = offline_input_16(X_16[:8])
        before = {k: v.clone() for k, v in model.state_dict().items()}
        with torch.inference_mode():
            together, alone = model(x), model(x[:1])
        changed = any(not torch.equal(before[k], v) for k, v in model.state_dict().items())
        if changed or not torch.allclose(together[:1], alone, atol=1e-5):
            return False, ("la même image, bien prétraitée, reçoit d'autres scores seule et dans un batch de 8 (écart "
                           f"{float((together[:1] - alone).abs().max()):.3f})"
                           + (", et chaque appel change les statistiques internes du modèle (running_mean)" if changed else "")), \
                "compare le modèle que rend load_service_16 à celui de l'évaluation hors ligne : qu'est-ce qui les distingue ?"
        with torch.inference_mode():
            want = MODEL_16(x)
        if not torch.allclose(together, want, atol=1e-4):
            return False, f"les scores du service s'écartent de ceux du notebook de {float((together - want).abs().max()):.3f}", \
                "les poids chargés sont-ils ceux de l'artefact ?"
        return True, "le modèle du service donne les scores du notebook, seule ou en batch", ""

    def point5():                                         # the names returned: the non-regression test
        model, meta = fresh_service_16()
        names = predict_16(model, meta, PATHS_16)
        agree = sum(a == b for a, b in zip(names, WANT_16))
        if agree != len(WANT_16):
            pairs = {}
            for got, want in zip(names, WANT_16):
                if got != want:
                    pairs[(want, got)] = pairs.get((want, got), 0) + 1
            (want, got), count = max(pairs.items(), key=lambda item: item[1])
            return False, (f"sur 200 images, le service ne rend le nom du notebook que pour {agree} ; par exemple, "
                           f"{count} images que le notebook classe « {want} » reçoivent « {got} »"), \
                "le nom que rend le service correspond-il à la sortie du modèle ?"
        order = np.random.default_rng(5).permutation(len(PATHS_16))[:40]     # other files, other names, other order
        again = scratch("again")
        copies = [Path(shutil.copy(PATHS_16[k], again / f"img_{j:03d}.png")) for j, k in enumerate(order)]
        agree = sum(a == WANT_16[k] for a, k in zip(predict_16(model, meta, copies), order))
        if agree != len(order):
            return False, (f"les mêmes images, copiées sous d'autres noms et dans un autre ordre, ne reçoivent le nom du "
                           f"notebook que pour {agree} sur {len(order)}"), "le service doit répondre d'après l'image seule"
        perm = np.random.default_rng(16).permutation(10)
        shuffled = wbp.reference_model()[0]
        with torch.no_grad():
            shuffled[-1].weight.copy_(shuffled[-1].weight[perm].clone())
            shuffled[-1].bias.copy_(shuffled[-1].bias[perm].clone())
        model, meta = fresh_service_16(shuffled, [CLASSES[k] for k in perm])
        agree = sum(a == b for a, b in zip(predict_16(model, meta, PATHS_16), WANT_16))
        if agree != len(WANT_16):
            return False, (f"le même modèle, ses sorties rangées dans un autre ordre (et classes dans cet ordre), ne "
                           f"reçoit le bon nom que pour {agree} images sur 200"), \
                "d'où le service tient-il l'ordre des classes ?"
        return True, ("les 200 noms sont ceux du notebook, même pour un modèle dont les sorties sont rangées dans un "
                      "autre ordre"), ""

    points = [("le service répond", point1), ("l'image lue", point2), ("le prétraitement", point3),
              ("le modèle du service", point4), ("les noms rendus (test de non-régression)", point5)]
    for k, (name, run) in enumerate(points, start=1):
        try:
            with wbp.torch_state_kept():
                ok, measured, why = run()
        except NotImplementedError:
            raise
        except Exception as error:  # noqa: BLE001 - the learner's code: one ❌ line, then the chain stops
            ok, measured, why = False, f"une erreur {type(error).__name__} : {str(error)[:200]}", "relis la fonction en cause"
        if ok:
            print(f"✅ Ex {ex_id} : point {k}, {name} ({measured}).")
        else:
            print(f"❌ Ex {ex_id} : point {k}, {name} : {measured} ; {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex {ex_id} : point {later} : corrige d'abord le point {k}.")
            return False
    print(f"✅ Ex {ex_id} : les cinq diagnostics passent : le service répond comme le notebook.")
    return True

In [ ]:
# The colleague's service, repaired: five changes, numbered in the comments
def save_artifact_16(model, folder, mean, std, classes):
    """Training side: the artifact of `model` in `folder`: model.safetensors (the weights) and meta.json (what the
    service needs). mean, std: the normalization learned on the training images; classes: the names of the outputs,
    in their order."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    save_file(model.state_dict(), str(folder / "model.safetensors"))
    meta = {"arch": "vgg", "widths": [16, 32, 64], "mean": list(mean), "std": list(std),
            "classes": list(classes)}                             # 3. the order of the outputs
    (folder / "meta.json").write_text(json.dumps(meta), encoding="utf-8")


def load_service_16(folder):
    """Service side: (model, meta), the model rebuilt from the artifact, ready to predict."""
    meta = json.loads((Path(folder) / "meta.json").read_text(encoding="utf-8"))
    model = wbp.build_reference(meta["arch"], meta["widths"])
    model.load_state_dict(load_file(str(Path(folder) / "model.safetensors")))
    model.eval()                                                  # 2. the evaluation mode
    return model, meta


def read_upload_16(path):
    """Service side: an uploaded image file, as the array the model was trained on."""
    return wbp.imread_bgr(path)[..., ::-1].astype(np.float32) / 255   # 4. RGB, floats in [0, 1]


def preprocess_16(images, meta):
    """Service side: the images (N, 32, 32, 3) read by read_upload_16, as the input tensor (N, 3, 32, 32) of the model."""
    x = np.asarray(images, dtype=np.float32)                      # 5. float32, as in training
    mean = np.asarray(meta["mean"], dtype=x.dtype)                # 1. the statistics of the
    std = np.asarray(meta["std"], dtype=x.dtype)                  #    training, from the artifact
    x = (x - mean) / std
    return torch.from_numpy(x).permute(0, 3, 1, 2)


def predict_16(model, meta, paths):
    """Service side: the French name of the class of each uploaded image (a list of str)."""
    images = np.stack([read_upload_16(p) for p in paths])
    with torch.inference_mode():
        k = model(preprocess_16(images, meta)).argmax(dim=1)
    return [FRENCH_16[meta["classes"][i]] for i in k.tolist()]   # 3. the order of the artifact


repaired_16_ok = False
with checked("B7.16"):
    repaired_16_ok = diagnose_16()

In [ ]:
# The service, as it is now (it shows the symptoms before the repair), on 8 uploads and on the 200 images
try:
    with wbp.torch_state_kept():
        model_16, meta_16 = fresh_service_16()
        names_16 = predict_16(model_16, meta_16, PATHS_16)
except Exception as error_16:  # noqa: BLE001 - the learner's code
    names_16 = None
    print(f"⏳ Ex B7.16 : le service lève {type(error_16).__name__} ({str(error_16)[:160]}) : rien à montrer.")
if isinstance(names_16, list) and len(names_16) == len(PATHS_16):
    print("image   vraie classe   notebook   service")
    for i_16 in range(8):
        print(f"{i_16:5d}   {FRENCH_16[CLASSES[Y_16[i_16]]]:12s}   {WANT_16[i_16]:10s} {names_16[i_16]}")
    truth_16 = [FRENCH_16[CLASSES[k]] for k in Y_16]
    print(f"accuracy sur 200 images : notebook {fr((OFFLINE_16 == Y_16).mean(), 3)}, service "
          f"{fr(np.mean([a == b for a, b in zip(names_16, truth_16)]), 3)} ; accord service-notebook "
          f"{sum(a == b for a, b in zip(names_16, WANT_16))} sur 200.")

### Ex B7.17 — L'artefact du modèle : safetensors, configuration, normalisation, classes, version, empreinte des données 📦 ★★ ⏱️ 30 min
**Objectif :** livrer un modèle dans un dossier qui suffit à prédire, sans le notebook ni ses variables.  
**Prérequis :** Ex B7.14 · Ex B7.16 · ch. 23 (fiche modèle, `state_dict`) · fiche §207.4 (empreinte), §207.7 (« L'artefact du modèle ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

> **Mode d'emploi du paquet** : chaque fichier de ton paquet est une chaîne de la cellule à compléter (`DATA_PY = r'''...'''`) ; la cellule de vérification l'**écrit** dans ton projet (`mon_travail/b7_production/cifar-prod/`, à côté de ce notebook), puis l'importe ou le lance dans un autre processus. Pour corriger un fichier, corrige sa chaîne et réexécute les deux cellules. Si tu préfères un éditeur, modifie le fichier sur le disque : la cellule le voit et ne l'écrase plus (elle le dit).

**Le contrat.** Un nouveau fichier, `src/cifarprod/artifact.py`, de trois fonctions (leurs docstrings, dans la cellule à compléter, sont le contrat) :
- `save_artifact(folder, model, *, arch, widths, mean, std, classes, data, metrics=None)` écrit `model.safetensors` (le `state_dict`, avec les métadonnées safetensors `{"package_version": "0.1.0"}`) et `meta.json`, aux clés `package_version`, `created` (la date ISO 8601 en UTC), `arch`, `widths`, `input` (`{"shape": [32, 32, 3], "dtype": "uint8", "channels": "RGB"}`), `mean`, `std`, `classes` (dans l'ordre des sorties), `data` (le dictionnaire reçu, tel quel), `metrics` (`{}` sans métriques), `torch`, `python` et `weights_sha256` (le SHA-256 du fichier de poids, par `file_sha256` de B7.14) ; elle rend le dossier ;
- `load_artifact(folder)` rend `(model, meta)` : le modèle reconstruit par `build_model(meta["arch"], meta["widths"], len(meta["classes"]))`, ses poids chargés, en mode `eval()` ; elle refuse (`ValueError`) un fichier de poids dont le SHA-256 n'est pas celui de `meta.json` ;
- `predict_images(model, meta, images)` rend `(labels, probabilities)` pour des images `uint8` RGB `(N, 32, 32, 3)` ou une seule `(32, 32, 3)` : les noms `meta["classes"][k]` et un tableau `float32` `(N, n_classes)` de probabilités softmax, calculés sans graphe avec `preprocess` de B7.9 et la normalisation de l'artefact.

Puis, dans la cellule de **B7.10** (`CLI_PY`), fais écrire l'artefact à `cifarprod train` : `save_artifact(out, ...)` remplace l'écriture des seuls poids (`arch` et `widths` de la configuration, la normalisation apprise, `CLASSES`, et pour `data` l'entrée `data` du journal de B7.14) ; la liste `artifacts` du journal garde les poids en premier, puis `meta.json`. Réexécute la cellule de vérification de B7.10 (elle écrit aussi `artifact.py`), puis celle-ci. Comme pour `tracking.py`, sur un projet neuf, « Tout exécuter » arrive aux vérifications qui lancent ta commande (B7.10 à B7.15) avant la cellule d'`artifact.py` : elles l'attendent (⏳) jusqu'à ce qu'elle soit exécutée.

La cellule donnée charge le modèle servi par la partie B, `MODEL_B` (le VGG du MP5, `INFO_B` : sa configuration, sa normalisation, ses classes, sa val_acc ; `DATA_B` : l'empreinte de ses 45 000 images d'entraînement, calculée une fois par le chapitre), et 200 images de test (`X_B`, `Y_B`). La cellule de vérification emballe `MODEL_B` avec ton `save_artifact`, relit les deux fichiers (tenseurs au bit près, en-tête, champs de `meta.json`), lance un **processus neuf** qui recharge l'artefact avec ton `load_artifact` et prédit les 200 images (mêmes noms et mêmes probabilités que le notebook, à $10^{-5}$ près), essaie une image seule, un modèle aux sorties rangées dans un autre ordre (les noms doivent suivre) et un fichier de poids remplacé (refusé) ; elle lance enfin `cifarprod train --set n_train=200` et lit l'artefact du run : architecture, classes, normalisation des 200 images et empreinte de leurs données. L'artefact du VGG reste dans `ARTIFACT_B`, que servent B7.18 à B7.21.

Dans tes notes : sur les cinq écarts de B7.16, lesquels un tel artefact aurait-il évités, et lesquels restent possibles malgré lui ? Pourquoi garder l'empreinte des données **et** celle des poids, et à quelles questions répondent-elles six mois plus tard ? Que manque-t-il encore pour qu'un autre que toi rejoue l'entraînement (207.4) ? Où placerais-tu la fiche modèle du ch. 23 (ou de 🛠️ B6.28) par rapport à ce dossier ?

In [ ]:
# The model that part B serves: the VGG 16-32-64 delivered by the MP5 solution (versioned with the workbook), with its
# normalization, its classes and the fingerprint of its 45 000 training images; 200 test images it has never seen
import hashlib
import importlib.util
import threading

from safetensors.torch import load_file, save_file

SOURCE_B = "reference"      # or "learner": YOUR model of the MP5, if it is a "vgg" (its arch and widths go in the artifact)
MODEL_B, INFO_B = wbp.base_model(SOURCE_B)
CONFIG_B = {"arch": INFO_B["config"]["arch"], "widths": list(INFO_B["config"]["widths"])}
DATA_B = INFO_B.get("data") or {"path": "CIFAR-10 train (your MP5 split)", "n_train": None, "sha256": None}
X_B, Y_B = wbp.test_images(20)                         # 200 test images (20 per class), uint8 RGB (200, 32, 32, 3)


def model_input(images, mean, std):
    """uint8 RGB images (N, 32, 32, 3) -> the float32 input (N, 3, 32, 32) of the model, as in training."""
    x = (np.asarray(images, np.float32) / np.float32(255) - np.asarray(mean, np.float32)) / np.asarray(std, np.float32)
    return torch.from_numpy(np.ascontiguousarray(x.transpose(0, 3, 1, 2)))


def offline_b(model, images, mean, std):
    """What the notebook computes, the reference of every check: (softmax probabilities float64 (N, 10), argmax)."""
    with torch.inference_mode():
        out = model(model_input(images, mean, std))
    return torch.softmax(out, dim=1).double().numpy(), out.argmax(dim=1).numpy()


def permuted_b(seed=17):
    """The reference VGG with its outputs in another order, and the classes in that order (the same predictions)."""
    perm = np.random.default_rng(seed).permutation(10)
    model = wbp.reference_model()[0]
    with torch.no_grad():
        model[-1].weight.copy_(model[-1].weight[perm].clone())
        model[-1].bias.copy_(model[-1].bias[perm].clone())
    return model, [CLASSES[k] for k in perm]


REF_INFO = wbp.reference_model()[1]                     # the reference VGG (B7.19 to B7.21 measure it)
PROBA_B, OFFLINE_B = offline_b(MODEL_B, X_B, INFO_B["mean"], INFO_B["std"])
NAMES_B = [INFO_B["classes"][k] for k in OFFLINE_B]     # what the service must answer for X_B
print(f"ℹ️ le modèle servi : {CONFIG_B}, accuracy {fr((OFFLINE_B == Y_B).mean(), 3)} sur ces 200 images de test ; "
      f"données d'entraînement : {DATA_B['n_train']} images, empreinte {str(DATA_B['sha256'])[:12]}….")

In [ ]:
# Ex B7.17: src/cifarprod/artifact.py (cli.py, written in Ex B7.10, already calls save_artifact)
ARTIFACT_PY = r'''"""The artifact of a model: everything needed to predict without the notebook, in one folder.

    save_artifact("artifact", model, arch="cnn", widths=(16, 32), mean=mean, std=std, classes=CLASSES,
                  data={"path": "data/cifar10_2000.npz", "n_train": 2000, "sha256": "..."})
    model, meta = load_artifact("artifact")                 # in eval() mode, the weights checked
    labels, proba = predict_images(model, meta, images)     # uint8 RGB (N, 32, 32, 3) -> names, probabilities

The folder holds model.safetensors (the weights: named tensors, nothing executed when loading) and meta.json (the
architecture, the input format, the normalization learned on the training images, the classes in the order of the
outputs, the data fingerprint, the metrics, the versions and the SHA-256 of the weights file).
"""

from __future__ import annotations

import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from . import __version__
from .data import preprocess
from .model import build_model
from .tracking import file_sha256

WEIGHTS, META = "model.safetensors", "meta.json"
INPUT = {"shape": [32, 32, 3], "dtype": "uint8", "channels": "RGB"}


def save_artifact(folder, model, *, arch, widths, mean, std, classes, data, metrics=None) -> Path:
    """Write the artifact of ``model`` into ``folder`` (created): model.safetensors (its state_dict, with the
    metadata {"package_version": __version__}) and meta.json (package_version, created, arch, widths, input, mean,
    std, classes, data, metrics, torch, python, weights_sha256). Returns the folder."""
    import torch
    from safetensors.torch import save_file

    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    state = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}
    save_file(state, str(folder / WEIGHTS), metadata={"package_version": __version__})
    meta = {"package_version": __version__, "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "arch": str(arch), "widths": [int(w) for w in widths], "input": dict(INPUT),
            "mean": [float(m) for m in mean], "std": [float(s) for s in std], "classes": [str(c) for c in classes],
            "data": dict(data), "metrics": dict(metrics or {}), "torch": torch.__version__,
            "python": platform.python_version(), "weights_sha256": file_sha256(folder / WEIGHTS)}
    (folder / META).write_text(json.dumps(meta, indent=1, sort_keys=True) + "\n", encoding="utf-8")
    return folder


def load_artifact(folder):
    """(model, meta): the model rebuilt from meta.json, its weights loaded from model.safetensors, in eval() mode.
    ValueError if the weights file is not the one meta.json describes (another SHA-256); FileNotFoundError if a file
    is missing."""
    from safetensors.torch import load_file

    folder = Path(folder)
    meta = json.loads((folder / META).read_text(encoding="utf-8"))
    sha = file_sha256(folder / WEIGHTS)
    if sha != meta.get("weights_sha256"):
        raise ValueError(f"the weights of {folder} do not match meta.json: SHA-256 {sha[:12]}…, expected "
                         f"{str(meta.get('weights_sha256'))[:12]}…")
    model = build_model(meta["arch"], meta["widths"], n_classes=len(meta["classes"]))
    model.load_state_dict(load_file(str(folder / WEIGHTS)))
    model.eval()
    return model, meta


def predict_images(model, meta, images):
    """(labels, probabilities) for uint8 RGB images (N, 32, 32, 3), or one image (32, 32, 3): the class names of
    meta["classes"] and a float32 array (N, n_classes) of softmax probabilities, computed without a graph, with the
    preprocessing of the training (data.preprocess and the normalization of the artifact)."""
    import torch

    x = preprocess(np.asarray(images), meta["mean"], meta["std"])
    with torch.inference_mode():
        proba = torch.softmax(model(x), dim=1).numpy()
    return [meta["classes"][k] for k in proba.argmax(axis=1)], proba
'''


ok_17 = {}
ARTIFACT_B = None                                        # the artifact of the served model, for B7.18 to B7.21
with checked("B7.17"):
    wbp.write_files(PROJECT, {"src/cifarprod/artifact.py": ARTIFACT_PY})
    art_17 = wbp.import_module(PROJECT, "artifact")
    folder_17 = scratch("artifact") / "vgg"
    with wbp.torch_state_kept():
        given_17 = art_17.save_artifact(folder_17, wbp.base_model(SOURCE_B, quiet=True)[0], **CONFIG_B,
                                        mean=INFO_B["mean"], std=INFO_B["std"], classes=INFO_B["classes"],
                                        data=DATA_B, metrics={"val_acc": INFO_B["val_acc"]})
    weights_17, meta_path_17 = folder_17 / "model.safetensors", folder_17 / "meta.json"
    ok_17["files"] = weights_17.is_file() and meta_path_17.is_file() and given_17 is not None \
        and Path(given_17).resolve() == folder_17.resolve()
    verdict("B7.17", ok_17["files"], "save_artifact écrit model.safetensors et meta.json, et rend le dossier.",
            f"save_artifact doit écrire {folder_17.name}/model.safetensors et {folder_17.name}/meta.json, et rendre le "
            f"dossier ; fichiers présents : {sorted(p.name for p in folder_17.glob('*')) if folder_17.exists() else []}, "
            f"rendu {given_17!r}.")
    if not ok_17["files"]:
        raise StopCheck
    from safetensors import safe_open
    try:
        with safe_open(str(weights_17), "pt") as f_17:
            header_17 = f_17.metadata() or {}
        tensors_17 = load_file(str(weights_17))
    except Exception as err_17:  # noqa: BLE001 - not a safetensors file: say it plainly
        ok_17["weights"] = False
        verdict("B7.17", False, "", f"model.safetensors n'est pas un fichier safetensors ({type(err_17).__name__} : "
                f"{str(err_17)[:80]}) : un pickle de torch.save ? Écris-le par safetensors.torch.save_file("
                "model.state_dict(), ...).")
        raise StopCheck
    state_17 = MODEL_B.state_dict()
    ok_17["weights"] = tensors_17.keys() == state_17.keys() and all(torch.equal(tensors_17[k], state_17[k]) for k in state_17)
    verdict("B7.17", ok_17["weights"], f"model.safetensors : les {len(state_17)} tenseurs du state_dict, au bit près.",
            f"model.safetensors doit contenir exactement le state_dict du modèle ({len(state_17)} tenseurs, buffers des "
            f"batchnorms compris) ; reçu {len(tensors_17)} tenseurs.")
    ok_17["header"] = header_17.get("package_version") == "0.1.0"
    verdict("B7.17", ok_17["header"], "l'en-tête safetensors porte package_version 0.1.0.",
            f"métadonnées safetensors : attendu {{'package_version': '0.1.0'}} (des chaînes), reçu {header_17}.")
    meta_17 = json.loads(meta_path_17.read_text(encoding="utf-8"))
    want_17 = {"package_version": "0.1.0", "arch": CONFIG_B["arch"], "widths": CONFIG_B["widths"],
               "input": {"shape": [32, 32, 3], "dtype": "uint8", "channels": "RGB"}, "classes": list(INFO_B["classes"]),
               "data": json.loads(json.dumps(DATA_B)), "metrics": {"val_acc": INFO_B["val_acc"]}, "torch": torch.__version__,
               "weights_sha256": hashlib.sha256(weights_17.read_bytes()).hexdigest()}
    wrong_17 = [k for k, v in want_17.items() if meta_17.get(k) != v]
    for key_17 in ("mean", "std"):
        if not (isinstance(meta_17.get(key_17), list) and len(meta_17[key_17]) == 3
                and np.allclose(meta_17[key_17], INFO_B[key_17], rtol=0, atol=1e-12)):
            wrong_17.append(key_17)
    try:
        from datetime import datetime
        stamp_17 = datetime.fromisoformat(str(meta_17.get("created")))
        if stamp_17.utcoffset() is None or stamp_17.utcoffset().total_seconds() != 0:
            wrong_17.append("created")
    except ValueError:
        wrong_17.append("created")
    if not isinstance(meta_17.get("python"), str):
        wrong_17.append("python")
    ok_17["meta"] = not wrong_17
    verdict("B7.17", ok_17["meta"], "meta.json : version, date UTC, architecture, format d'entrée, normalisation, "
            "classes, données, métriques, versions et SHA-256 des poids.",
            f"meta.json : champs absents ou faux : {wrong_17} (relis la docstring de save_artifact ; widths et classes "
            "en listes, created en ISO 8601 avec le fuseau UTC, weights_sha256 du fichier écrit).")
    ARTIFACT_B = folder_17

with checked("B7.17"):                                    # a NEW process reloads the artifact and predicts
    if ARTIFACT_B is None:
        raise StopCheck
    io_17 = scratch("io")
    np.save(io_17 / "x.npy", X_B)
    probe_17 = ("import json, sys\nimport numpy as np\nfrom cifarprod.artifact import load_artifact, predict_images\n"
                "model, meta = load_artifact(sys.argv[1])\nlabels, proba = predict_images(model, meta, np.load(sys.argv[2]))\n"
                "np.save(sys.argv[3], np.asarray(proba, dtype=np.float64))\n"
                "print(json.dumps({'labels': [str(v) for v in labels], 'training': model.training, "
                "'dtype': str(np.asarray(proba).dtype)}))\n")
    (io_17 / "reload.py").write_text(probe_17, encoding="utf-8")
    r_17 = wbp.run([sys.executable, io_17 / "reload.py", ARTIFACT_B, io_17 / "x.npy", io_17 / "p.npy"], PROJECT,
                   src=[PROJECT / "src"])
    seen_17 = r_17.last_json() if r_17.ok else None
    if not run_line("B7.17", r_17, r_17.ok and isinstance(seen_17, dict), "un processus neuf recharge l'artefact.",
                    "attendu un processus neuf qui recharge l'artefact (load_artifact) et prédit les 200 images"):
        raise StopCheck
    agree_17 = sum(a == b for a, b in zip(seen_17["labels"], NAMES_B))
    proba_17 = np.load(io_17 / "p.npy")
    ok_17["fresh"] = (agree_17 == len(NAMES_B) and not seen_17["training"] and proba_17.shape == PROBA_B.shape
                      and np.allclose(proba_17, PROBA_B, rtol=0, atol=1e-5) and seen_17["dtype"] == "float32")
    verdict("B7.17", ok_17["fresh"], "dans un processus neuf, les 200 images reçoivent les noms et les probabilités du "
            "notebook (à 1e-5 près), le modèle en mode eval().",
            f"dans un processus neuf : {agree_17} noms sur 200 comme le notebook, modèle en mode "
            f"{'entraînement' if seen_17['training'] else 'eval()'}, probabilités {seen_17['dtype']} de forme "
            f"{proba_17.shape}, écart {float(np.abs(proba_17 - PROBA_B).max()) if proba_17.shape == PROBA_B.shape else '?'} "
            "(attendu 200, eval(), float32 (200, 10), au plus 1e-5)"
            + (" : un écart sur les probabilités, predict_images normalise-t-il par meta['mean'] et meta['std'] ?"
               if not seen_17["training"] and proba_17.shape == PROBA_B.shape else "") + ".")
    with wbp.torch_state_kept():
        model_17, meta_l_17 = art_17.load_artifact(ARTIFACT_B)
        alone_17, p_alone_17 = art_17.predict_images(model_17, meta_l_17, X_B[3])
    ok_17["one"] = list(alone_17) == [NAMES_B[3]] and np.allclose(np.asarray(p_alone_17).reshape(-1), PROBA_B[3], atol=1e-5)
    verdict("B7.17", ok_17["one"], "une image seule (32, 32, 3) reçoit le nom et les probabilités qu'elle a dans le batch.",
            f"predict_images d'une image (32, 32, 3) : attendu [{NAMES_B[3]!r}] et ses probabilités du batch, reçu "
            f"{list(alone_17)!r}, écart des probabilités "
            f"{float(np.abs(np.asarray(p_alone_17, dtype=float).reshape(-1)[:10] - PROBA_B[3]).max()) if np.asarray(p_alone_17).size == 10 else '?'}.")
    perm_model_17, perm_classes_17 = permuted_b()
    folder_p_17 = scratch("artifact") / "permuted"
    with wbp.torch_state_kept():
        art_17.save_artifact(folder_p_17, perm_model_17, arch="vgg", widths=[16, 32, 64], mean=REF_INFO["mean"],
                             std=REF_INFO["std"], classes=perm_classes_17, data=DATA_B)
        names_p_17 = art_17.predict_images(*art_17.load_artifact(folder_p_17), X_B)[0]
    ref_names_17 = [CLASSES[k] for k in offline_b(wbp.reference_model()[0], X_B, REF_INFO["mean"], REF_INFO["std"])[1]]
    ok_17["classes"] = list(names_p_17) == ref_names_17
    verdict("B7.17", ok_17["classes"], "un modèle aux sorties rangées dans un autre ordre garde les bons noms : les "
            "classes voyagent avec l'artefact.",
            ("(les prédictions sont déjà fausses ci-dessus) " if not ok_17.get("fresh", True) else "") + f"un modèle dont les sorties sont rangées dans un autre ordre (et classes dans cet ordre) ne reçoit le bon "
            f"nom que pour {sum(a == b for a, b in zip(names_p_17, ref_names_17))} images sur 200 : d'où viennent les "
            "noms de predict_images ?")
    save_file(wbp.reference_model()[0].state_dict(), str(folder_p_17 / "model.safetensors"))   # another weights file
    ok_17["sha"] = refuses("B7.17", "load_artifact d'un dossier dont on a remplacé model.safetensors",
                           lambda: art_17.load_artifact(folder_p_17))

with checked("B7.17"):                                    # the command line writes the artifact of each run
    if ARTIFACT_B is None:
        raise StopCheck
    out_17 = scratch("out") / "run"
    r_17 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--out", out_17, "--runs", scratch("runs") / "runs.jsonl")
    if not run_line("B7.17", r_17, r_17.ok and (out_17 / "meta.json").is_file(),
                    "cifarprod train écrit l'artefact (model.safetensors et meta.json).",
                    "attendu le code 0 et <out>/meta.json à côté de model.safetensors : dans CLI_PY (cellule de B7.10), "
                    "remplace l'écriture des poids par save_artifact, puis réexécute la cellule de vérification de B7.10"):
        raise StopCheck
    meta_c_17 = json.loads((out_17 / "meta.json").read_text(encoding="utf-8"))
    want_mean_17 = (SUBSET["x_train"][:200] / 255.0).mean(axis=(0, 1, 2))
    want_sha_17 = wbp.array_fingerprint(SUBSET["x_train"][:200], SUBSET["y_train"][:200])
    got_c_17 = {"arch": meta_c_17.get("arch"), "widths": meta_c_17.get("widths"), "classes": meta_c_17.get("classes"),
                "sha256": (meta_c_17.get("data") or {}).get("sha256")}
    ok_17["cli"] = (got_c_17 == {"arch": "cnn", "widths": [16, 32], "classes": list(CLASSES), "sha256": want_sha_17}
                    and np.allclose(meta_c_17.get("mean", [0, 0, 0]), want_mean_17, rtol=0, atol=1e-12))
    verdict("B7.17", ok_17["cli"], f"l'artefact du run : cnn [16, 32], les classes, la normalisation des 200 images et "
            f"l'empreinte de leurs données ({want_sha_17[:12]}…).",
            f"l'artefact de cifarprod train --set n_train=200 : attendu arch cnn, widths [16, 32], classes = CLASSES, "
            f"mean = channel_stats des 200 images et data.sha256 = fingerprint(X[:200], y[:200]) ; reçu "
            f"{ {k: (v[:12] + '…' if isinstance(v, str) and len(v) > 40 else v) for k, v in got_c_17.items()} }.")
    with wbp.torch_state_kept():
        art_17.load_artifact(out_17)
if ok_17 and all(ok_17.values()) and len(ok_17) == 9:
    print("✅ Ex B7.17 : ton artefact se suffit : un autre processus le recharge et prédit comme le notebook.")

### Ex B7.18 — Une API de prédiction avec FastAPI (repli : http.server) : /health, /predict, validation des entrées 📦 ★★★ ⏱️ 45 min
**Objectif :** servir un artefact par une API HTTP qui valide ses entrées, répond comme l'évaluation hors ligne et s'arrête proprement.  
**Prérequis :** Ex B7.17 · ⚖️ B7.7 (le champ `input`) · fiche §207.7 (« En ligne ou par batches, et une API HTTP ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

> **Mode d'emploi du paquet** : chaque fichier de ton paquet est une chaîne de la cellule à compléter (`DATA_PY = r'''...'''`) ; la cellule de vérification l'**écrit** dans ton projet (`mon_travail/b7_production/cifar-prod/`, à côté de ce notebook), puis l'importe ou le lance dans un autre processus. Pour corriger un fichier, corrige sa chaîne et réexécute les deux cellules. Si tu préfères un éditeur, modifie le fichier sur le disque : la cellule le voit et ne l'écrase plus (elle le dit).

**Le contrat** de `src/cifarprod/serve.py` (sa docstring et celles de chaque objet sont dans la cellule à compléter) :
- `Image` et `Batch`, deux modèles pydantic : `{"pixels": 32 × 32 × 3 entiers de 0 à 255}` (celui de la fiche) et `{"images": une liste de 1 à 64 telles images}` ; tout le reste lève une `ValueError` dans leur validateur ;
- `Service(artifact_dir)` charge l'artefact (`load_artifact`) : `model`, `meta`, `version` (les 12 premiers caractères de `meta["weights_sha256"]`) ; `health()` rend `{"status": "ok", "package_version", "model": {"arch", "widths", "classes", "weights_sha256", "data_sha256", "created"}, "torch"}` ; `predict(images)` rend une liste de `{"label", "probability"}` (la probabilité du label, arrondie à 4 décimales), par `predict_images` ;
- `create_app(artifact_dir)`, l'application FastAPI, son `Service` dans `app.state.service` : `GET /health` ; `POST /predict` (`Image`) → `{"label", "probability", "model_version"}` ; `POST /predict_batch` (`Batch`) → `{"predictions": [...], "model_version"}` ; une requête mal formée → **422** `{"detail": [{"loc", "msg", "type"}, ...]}` **sans** l'entrée envoyée (FastAPI la renvoie par défaut dans le champ `input` : ⚖️ B7.7) ; une exception du modèle → **500** `{"detail": "internal error"}`, et une ligne `log.exception` ;
- `Handler` et `make_server(artifact_dir, host="127.0.0.1", port=0)`, le **repli** sans FastAPI : un `ThreadingHTTPServer` (pas encore démarré), les mêmes routes, la même validation (les mêmes modèles pydantic), les mêmes réponses et codes de statut, 404 pour une autre route ; son `Service` dans `server.service` ;
- `running(artifact_dir, backend="auto")`, un gestionnaire de contexte (`with running(...) as url:`) qui sert l'artefact dans un **thread**, sur un port libre, rend l'URL de base (`"http://127.0.0.1:<port>"`) et **arrête toujours** le serveur à la sortie du bloc, même après une erreur ; `backend` : `"fastapi"` (uvicorn), `"stdlib"` (`make_server`) ou `"auto"` (FastAPI s'il s'importe) ;
- `main(argv)`, pour `python -m cifarprod.serve --artifact DIR --host HOST --port 8000 --backend auto|fastapi|stdlib` (par défaut, `--artifact` vaut la variable d'environnement `MODEL_DIR`, sinon `artifact` ; `--host` `127.0.0.1` ; `--backend` `auto`) : sert jusqu'à l'arrêt ; code 1 et une ligne ERROR si l'artefact ne se charge pas.

La cellule de vérification sert `ARTIFACT_B` (B7.17) : avec FastAPI par `TestClient` (sans serveur), puis avec ton `make_server` dans un thread qu'elle démarre et **arrête** elle-même, elle envoie les mêmes requêtes aux deux : `/health`, huit images (les noms du notebook : un test de non-régression), huit requêtes mal formées (422, sans l'entrée dans la réponse), un batch de 32, un modèle qui plante (500 propre), une route inconnue. Elle essaie ensuite ton `running` avec les deux backends, une erreur levée dans le bloc (le port doit être libéré), puis ta commande `python -m cifarprod.serve` dans un **autre processus** (arrêté ensuite), et un dossier sans artefact. L'avertissement de Starlette 1.10 sur `httpx2` est tu (`TestClient` marche avec le `httpx` du workbook).

Dans tes notes : pourquoi la validation des entrées appartient-elle à l'API et pas au modèle ? Pourquoi une réponse 500 ne contient-elle ni la trace de l'erreur ni l'entrée ? Que gagne-t-on à ce que FastAPI et `http.server` partagent `Service` et les modèles pydantic, et que perdrait-on à les écrire deux fois ? Qu'arrive-t-il à un notebook dont le serveur n'est pas arrêté, quand on réexécute la cellule ?

In [ ]:
# Ex B7.18: src/cifarprod/serve.py
SERVE_PY = r'''"""The prediction API of cifarprod: FastAPI when it is installed, else the standard library (http.server).

    app = create_app("artifact")                      # FastAPI; or: uvicorn --factory cifarprod.serve:app_from_env
    server = make_server("artifact", port=0)          # the fallback: a ThreadingHTTPServer with the same routes
    with running("artifact") as url:                  # in a thread, on a free port, always stopped at the end
        requests.post(f"{url}/predict", json={"pixels": image.tolist()})
    python -m cifarprod.serve --artifact artifact --port 8000

Routes: GET /health -> {"status": "ok", "package_version", "model": {"arch", "widths", "classes", "weights_sha256",
"data_sha256", "created"}, "torch"}; POST /predict {"pixels": 32 x 32 x 3 integers in 0..255, RGB} -> {"label",
"probability", "model_version"}; POST /predict_batch {"images": [1 to 64 such images]} -> {"predictions": [{"label",
"probability"}, ...], "model_version"}. A body that does not follow the schema: 422 {"detail": [{"loc", "msg",
"type"}]}, without the input (it may be personal data); an error of the model: 500 {"detail": "internal error"}.
Both servers validate with the same pydantic models and predict with the same Service (artifact.predict_images, the
preprocessing of the training).
"""

from __future__ import annotations

import argparse
import contextlib
import json
import logging
import os
import sys
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import numpy as np
from pydantic import BaseModel, ValidationError, field_validator

from . import __version__
from .artifact import load_artifact, predict_images

log = logging.getLogger(__name__)
MAX_BATCH = 64


def _check_image(value) -> None:
    a = np.asarray(value)
    if a.shape != (32, 32, 3):
        raise ValueError(f"expected shape (32, 32, 3), got {a.shape}")
    if a.size and (a.min() < 0 or a.max() > 255):
        raise ValueError("pixel values must be in 0..255")


class Image(BaseModel):
    pixels: list[list[list[int]]]                          # 32 x 32 x 3 RGB values in 0..255

    @field_validator("pixels")
    @classmethod
    def check_pixels(cls, v):
        _check_image(v)
        return v


class Batch(BaseModel):
    images: list[list[list[list[int]]]]                    # 1 to MAX_BATCH images of 32 x 32 x 3

    @field_validator("images")
    @classmethod
    def check_images(cls, v):
        if not 1 <= len(v) <= MAX_BATCH:
            raise ValueError(f"expected 1 to {MAX_BATCH} images, got {len(v)}")
        for image in v:
            _check_image(image)
        return v


def errors_without_input(error: ValidationError | list) -> list[dict]:
    """The errors of pydantic without the field "input" (the request may contain personal data)."""
    errors = error.errors() if isinstance(error, ValidationError) else error
    return [{"loc": list(e.get("loc", ())), "msg": e.get("msg", ""), "type": e.get("type", "")} for e in errors]


class Service:
    """The model of an artifact and its answers (shared by the two servers)."""

    def __init__(self, artifact_dir) -> None:
        self.model, self.meta = load_artifact(artifact_dir)
        self.version = self.meta["weights_sha256"][:12]

    def health(self) -> dict:
        import torch

        m = self.meta
        return {"status": "ok", "package_version": __version__,
                "model": {"arch": m["arch"], "widths": m["widths"], "classes": m["classes"],
                          "weights_sha256": m["weights_sha256"], "data_sha256": m.get("data", {}).get("sha256"),
                          "created": m.get("created")},
                "torch": torch.__version__}

    def predict(self, images: list) -> list[dict]:
        labels, proba = predict_images(self.model, self.meta, np.asarray(images, dtype=np.uint8))
        return [{"label": label, "probability": round(float(p.max()), 4)} for label, p in zip(labels, proba)]


def create_app(artifact_dir):
    """The FastAPI application of the artifact ``artifact_dir`` (its Service in ``app.state.service``)."""
    from fastapi import FastAPI, Request
    from fastapi.exceptions import RequestValidationError
    from fastapi.responses import JSONResponse

    service = Service(artifact_dir)
    app = FastAPI(title="cifar-api", version=__version__)
    app.state.service = service

    @app.exception_handler(RequestValidationError)
    async def invalid(request: Request, exc: RequestValidationError):
        return JSONResponse(status_code=422, content={"detail": errors_without_input(exc.errors())})

    @app.get("/health")
    def health() -> dict:
        return app.state.service.health()

    @app.post("/predict")
    def predict(image: Image):
        try:
            answer = app.state.service.predict([image.pixels])[0]
        except Exception:                                  # noqa: BLE001 - logged, a clean 500 for the client
            log.exception("prediction failed")
            return JSONResponse(status_code=500, content={"detail": "internal error"})
        return {**answer, "model_version": app.state.service.version}

    @app.post("/predict_batch")
    def predict_batch(batch: Batch):
        try:
            answers = app.state.service.predict(batch.images)
        except Exception:                                  # noqa: BLE001
            log.exception("prediction failed")
            return JSONResponse(status_code=500, content={"detail": "internal error"})
        return {"predictions": answers, "model_version": app.state.service.version}

    return app


def app_from_env():
    """create_app on the folder of the variable MODEL_DIR (default "artifact"): ``uvicorn --factory``."""
    return create_app(os.environ.get("MODEL_DIR", "artifact"))


class Handler(BaseHTTPRequestHandler):
    """The same routes with the standard library; the Service is ``self.server.service``."""

    protocol_version = "HTTP/1.1"                           # keep-alive: one connection for many requests
    disable_nagle_algorithm = True                          # headers and body leave at once (no 40 ms wait)

    def _send(self, status: int, body: dict) -> None:
        data = json.dumps(body).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self) -> None:
        if self.path == "/health":
            self._send(200, self.server.service.health())
        else:
            self._send(404, {"detail": "Not Found"})

    def do_POST(self) -> None:
        routes = {"/predict": Image, "/predict_batch": Batch}
        body = self.rfile.read(int(self.headers.get("Content-Length") or 0))
        if self.path not in routes:
            self._send(404, {"detail": "Not Found"})
            return
        try:
            request = routes[self.path].model_validate_json(body or b"{}")
        except ValidationError as error:
            self._send(422, {"detail": errors_without_input(error)})
            return
        service = self.server.service
        try:
            if self.path == "/predict":
                answer = {**service.predict([request.pixels])[0], "model_version": service.version}
            else:
                answer = {"predictions": service.predict(request.images), "model_version": service.version}
        except Exception:                                  # noqa: BLE001 - logged, a clean 500 for the client
            log.exception("prediction failed")
            self._send(500, {"detail": "internal error"})
            return
        self._send(200, answer)

    def log_message(self, format, *args) -> None:          # one DEBUG line per request, not stderr
        log.debug("%s %s", self.address_string(), format % args)


def make_server(artifact_dir, host: str = "127.0.0.1", port: int = 0) -> ThreadingHTTPServer:
    """The fallback server (http.server), not started: ``server.serve_forever()``; port 0 = a free port
    (``server.server_address[1]``); its Service in ``server.service``."""
    server = ThreadingHTTPServer((host, port), Handler)
    server.daemon_threads = True
    server.service = Service(artifact_dir)
    return server


def has_fastapi() -> bool:
    try:
        import fastapi  # noqa: F401
        import uvicorn  # noqa: F401
    except ImportError:
        return False
    return True


@contextlib.contextmanager
def running(artifact_dir, backend: str = "auto", host: str = "127.0.0.1"):
    """Serve the artifact in a thread, on a free port, and yield its base URL ("http://127.0.0.1:<port>"); the server
    is ALWAYS stopped and its port freed at the end of the block, even after an error. backend: "fastapi" (uvicorn),
    "stdlib" (http.server) or "auto" (FastAPI if installed)."""
    if backend == "auto":
        backend = "fastapi" if has_fastapi() else "stdlib"
    if backend == "fastapi":
        import uvicorn

        server = uvicorn.Server(uvicorn.Config(create_app(artifact_dir), host=host, port=0, log_level="warning",
                                               lifespan="off"))
        thread = threading.Thread(target=server.run, daemon=True)
        thread.start()
        try:
            deadline = time.monotonic() + 30
            while not server.started:
                if not thread.is_alive() or time.monotonic() > deadline:
                    raise RuntimeError("the uvicorn server did not start")
                time.sleep(0.01)
            port = server.servers[0].sockets[0].getsockname()[1]
            yield f"http://{host}:{port}"
        finally:
            server.should_exit = True
            thread.join(10)
    elif backend == "stdlib":
        server = make_server(artifact_dir, host, 0)
        thread = threading.Thread(target=server.serve_forever, daemon=True)
        thread.start()
        try:
            yield f"http://{host}:{server.server_address[1]}"
        finally:
            server.shutdown()                              # stop serve_forever, then free the port
            server.server_close()
            thread.join(10)
    else:
        raise ValueError(f"backend must be 'auto', 'fastapi' or 'stdlib', got {backend!r}")


def main(argv: list[str] | None = None) -> int:
    """python -m cifarprod.serve [--artifact DIR] [--host HOST] [--port PORT] [--backend auto|fastapi|stdlib]."""
    parser = argparse.ArgumentParser(prog="cifarprod.serve", description="Serve the artifact of a model over HTTP.")
    parser.add_argument("--artifact", default=os.environ.get("MODEL_DIR", "artifact"),
                        help="the artifact folder (default: $MODEL_DIR, else %(default)s)")
    parser.add_argument("--host", default=os.environ.get("HOST", "127.0.0.1"))
    parser.add_argument("--port", type=int, default=int(os.environ.get("PORT", "8000")))
    parser.add_argument("--backend", choices=["auto", "fastapi", "stdlib"], default="auto")
    args = parser.parse_args(argv)
    logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    backend = args.backend if args.backend != "auto" else ("fastapi" if has_fastapi() else "stdlib")
    try:
        if backend == "fastapi":
            import uvicorn

            app = create_app(args.artifact)
            log.info("serving %s on http://%s:%d (FastAPI)", args.artifact, args.host, args.port)
            uvicorn.run(app, host=args.host, port=args.port, log_level="warning")
        else:
            server = make_server(args.artifact, args.host, args.port)
            log.info("serving %s on http://%s:%d (http.server)", args.artifact, args.host, server.server_address[1])
            try:
                server.serve_forever()
            except KeyboardInterrupt:
                pass
            finally:
                server.server_close()
    except (OSError, ValueError, KeyError) as err:
        log.error("cannot serve %s: %s", args.artifact, err)
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''


ok_18 = {}
warnings.filterwarnings("ignore", message=r".*Using `httpx` with `starlette.testclient`.*")   # Starlette 1.10
BAD_18 = {"une image 28 × 28": {"pixels": np.zeros((28, 28, 3), int).tolist()},
          "un pixel à 256": {"pixels": np.full((32, 32, 3), 256).tolist()},
          "un pixel à -1": {"pixels": np.full((32, 32, 3), -1).tolist()},
          "pixels en chaîne": {"pixels": "a cat"},
          "sans le champ pixels": {"image": X_B[0].tolist()}}
BAD_BATCH_18 = {"un batch vide": {"images": []}, "65 images": {"images": np.zeros((65, 32, 32, 3), int).tolist()},
                "une image 28 × 28 dans le batch": {"images": [X_B[0].tolist(), np.zeros((28, 28, 3), int).tolist()]}}


class Failing_18(torch.nn.Module):
    """A model that fails at every call (a bug, an out-of-memory...): the API must answer a clean 500."""

    def forward(self, x):
        raise RuntimeError("the model failed (a test of the check cell)")


@contextlib.contextmanager
def logged_18():
    """The records of the logger cifarprod.serve during the block, kept here instead of being printed."""
    import logging
    logger, records = logging.getLogger("cifarprod.serve"), []
    handler = logging.Handler(level=logging.DEBUG)
    handler.emit = records.append
    keep = logger.propagate, logger.level
    logger.addHandler(handler)
    logger.propagate = False
    logger.setLevel(logging.DEBUG)
    try:
        yield records
    finally:
        logger.removeHandler(handler)
        logger.propagate, logger.level = keep[0], keep[1]


def api_cases_18(get, post, label, records=()):
    """The same cases for both servers: get(path) and post(path, body) return (status, body as a dict or a text);
    records: the log records of the server (logged_18), to show the cause of a 500."""
    oks = {}
    status, body = get("/health")
    body = body if isinstance(body, dict) else {}
    model_info = body.get("model", {}) if isinstance(body.get("model"), dict) else {}
    want_health = {"status": "ok", "package_version": "0.1.0"}
    want_model = {"arch": META_18["arch"], "widths": META_18["widths"], "classes": META_18["classes"],
                  "weights_sha256": META_18["weights_sha256"], "data_sha256": META_18["data"].get("sha256"),
                  "created": META_18.get("created")}
    wrong = [k for k, v in want_health.items() if body.get(k) != v] + \
        [f"model.{k}" for k, v in want_model.items() if model_info.get(k) != v] + \
        ([] if isinstance(body.get("torch"), str) else ["torch"])
    oks["health"] = status == 200 and not wrong
    verdict("B7.18", oks["health"], f"{label} GET /health : 200, la version du paquet, le SHA-256 des poids et des données.",
            f"{label} GET /health : attendu 200 et {{status, package_version, model: {{arch, widths, classes, "
            f"weights_sha256, data_sha256, created}}, torch}} ; reçu {status}, champs absents ou faux : {wrong}.")
    answers = [post("/predict", {"pixels": X_B[i].tolist()}) for i in range(8)]
    labels = [b.get("label") if isinstance(b, dict) else None for _, b in answers]
    probs_ok = all(isinstance(b, dict) and abs(float(b.get("probability", -1)) - PROBA_B[i].max()) <= 6e-5
                   and b.get("model_version") == META_18["weights_sha256"][:12] for i, (_, b) in enumerate(answers))
    oks["predict"] = all(s == 200 for s, _ in answers) and labels == NAMES_B[:8] and probs_ok
    verdict("B7.18", oks["predict"], f"{label} POST /predict : les 8 noms du notebook, leur probabilité (4 décimales) "
            "et la version du modèle (test de non-régression).",
            f"{label} POST /predict sur 8 images : attendu 200, les noms {NAMES_B[:8]}, la probabilité de chacun à 4 "
            f"décimales et model_version = les 12 premiers caractères du SHA-256 des poids ; reçu "
            f"{[s for s, _ in answers]} {labels} {str(answers[0][1])[:120]}.")
    if any(s == 500 for s, _ in answers):
        last = next((r for r in reversed(list(records)) if r.exc_info), None)
        if last is not None:
            print(f"   côté serveur : {last.exc_info[0].__name__} : {str(last.exc_info[1])[:200]}")
    refused = {}
    for what, body_bad in {**BAD_18, **{k: v for k, v in BAD_BATCH_18.items()}}.items():
        path = "/predict_batch" if what in BAD_BATCH_18 else "/predict"
        status, answer = post(path, body_bad)
        text = json.dumps(answer) if isinstance(answer, dict) else str(answer)
        refused[what] = (status, status == 422 and isinstance(answer, dict) and '"input"' not in text
                         and "a cat" not in text)
    oks["422"] = all(good for _, good in refused.values())
    not_422 = [f"{w} ({s})" for w, (s, good) in refused.items() if s != 422]
    echoed = [w for w, (s, good) in refused.items() if s == 422 and not good]
    verdict("B7.18", oks["422"], f"{label} {len(refused)} requêtes mal formées : 422, sans l'entrée envoyée dans la réponse.",
            f"{label} attendu 422 en JSON, sans l'entrée envoyée, pour chaque requête mal formée ; "
            + " ; ".join(part for part in [
                f"pas de 422 : {', '.join(not_422)} (les modèles pydantic doivent refuser ces entrées : forme, bornes, "
                "taille du batch)" if not_422 else "",
                f"422 avec l'entrée renvoyée : {', '.join(echoed)} (retire input de chaque erreur : une donnée "
                "personnelle, peut-être)" if echoed else ""] if part) + ".")
    status, body = post("/predict_batch", {"images": X_B[:32].tolist()})
    batch_labels = [p.get("label") for p in body.get("predictions", [])] if isinstance(body, dict) else []
    oks["batch"] = status == 200 and batch_labels == NAMES_B[:32] and body.get("model_version") == META_18["weights_sha256"][:12]
    verdict("B7.18", oks["batch"], f"{label} POST /predict_batch de 32 images : les 32 noms du notebook.",
            f"{label} POST /predict_batch de 32 images : attendu 200, {{predictions: [{{label, probability}}] × 32, "
            f"model_version}} aux noms du notebook ; reçu {status}, {sum(a == b for a, b in zip(batch_labels, NAMES_B))} "
            "noms justes.")
    return oks


with checked("B7.18"):
    if ARTIFACT_B is None:
        print("⏳ Ex B7.18 : l'API sert l'artefact de B7.17 : fais d'abord passer B7.17.")
        raise StopCheck
    wbp.write_files(PROJECT, {"src/cifarprod/serve.py": SERVE_PY})
    serve_18 = wbp.import_module(PROJECT, "serve")
    META_18 = json.loads((ARTIFACT_B / "meta.json").read_text(encoding="utf-8"))
    ok_18["service"] = serve_18.Service(ARTIFACT_B).version == META_18["weights_sha256"][:12]
    verdict("B7.18", ok_18["service"], "Service charge l'artefact ; sa version : les 12 premiers caractères du SHA-256 "
            "des poids.", "Service(dossier).version doit valoir meta['weights_sha256'][:12].")
    if importlib.util.find_spec("fastapi") is None:
        print("ℹ️ FastAPI n'est pas installé ici : seul le repli http.server est vérifié.")
    else:
        from fastapi.testclient import TestClient
        app_18 = serve_18.create_app(ARTIFACT_B)
        client_18 = TestClient(app_18, raise_server_exceptions=False)

        def as_body_18(response):
            try:
                return response.json()
            except ValueError:
                return response.text

        with logged_18() as kept_records_18:              # the server's log lines are kept, not printed
            ok_18.update({f"fastapi {k}": v for k, v in api_cases_18(
                lambda path: (lambda r: (r.status_code, as_body_18(r)))(client_18.get(path)),
                lambda path, body: (lambda r: (r.status_code, as_body_18(r)))(client_18.post(path, json=body)),
                "FastAPI :", kept_records_18).items()})
        kept_18 = app_18.state.service.model
        app_18.state.service.model = Failing_18()
        try:
            with logged_18() as records_18:
                r_18 = client_18.post("/predict", json={"pixels": X_B[0].tolist()})
        finally:
            app_18.state.service.model = kept_18
        traced_18 = any(r.levelname == "ERROR" and r.exc_info for r in records_18)
        ok_18["fastapi 500"] = r_18.status_code == 500 and as_body_18(r_18) == {"detail": "internal error"} and traced_18
        verdict("B7.18", ok_18["fastapi 500"], "FastAPI : un modèle qui plante donne un 500 propre, {\"detail\": "
                "\"internal error\"}, et une ligne log.exception (ici retenue, pas affichée).",
                f"FastAPI : un modèle qui lève une exception doit donner 500 et le JSON {{\"detail\": \"internal "
                f"error\"}} (sans trace pour le client), et une ligne log.exception côté serveur ; reçu {r_18.status_code} "
                f"{str(as_body_18(r_18))[:120]}, {'une' if traced_18 else 'aucune'} ligne ERROR avec sa trace.")

with checked("B7.18"):                                    # the fallback, started and ALWAYS stopped by this cell
    if "serve_18" not in globals() or ARTIFACT_B is None:
        raise StopCheck
    import requests
    server_18 = serve_18.make_server(ARTIFACT_B, "127.0.0.1", 0)
    thread_18 = threading.Thread(target=server_18.serve_forever, daemon=True)
    thread_18.start()
    try:
        url_18 = f"http://127.0.0.1:{server_18.server_address[1]}"
        session_18 = requests.Session()

        def answer_18(response):
            try:
                return response.status_code, response.json()
            except ValueError:
                return response.status_code, response.text

        with logged_18() as kept_records_18:
            ok_18.update({f"stdlib {k}": v for k, v in api_cases_18(
                lambda path: answer_18(session_18.get(url_18 + path, timeout=10)),
                lambda path, body: answer_18(session_18.post(url_18 + path, json=body, timeout=10)),
                "http.server :", kept_records_18).items()})
        kept_18 = server_18.service.model
        server_18.service.model = Failing_18()
        try:
            with logged_18():
                status_18, body_18 = answer_18(session_18.post(url_18 + "/predict", json={"pixels": X_B[0].tolist()},
                                                               timeout=10))
        finally:
            server_18.service.model = kept_18
        ok_18["stdlib 500"] = status_18 == 500 and body_18 == {"detail": "internal error"}
        verdict("B7.18", ok_18["stdlib 500"], "http.server : un 500 propre aussi.",
                f"http.server : un modèle qui lève une exception doit donner 500 {{\"detail\": \"internal error\"}} ; reçu "
                f"{status_18} {str(body_18)[:120]}.")
        status_18, _ = answer_18(session_18.get(url_18 + "/nowhere", timeout=10))
        ok_18["stdlib 404"] = status_18 == 404
        verdict("B7.18", ok_18["stdlib 404"], "http.server : 404 pour une autre route.",
                f"http.server : GET /nowhere doit rendre 404, reçu {status_18}.")
    finally:
        if "session_18" in globals():
            session_18.close()                               # the keep-alive connection: its handler thread ends with it
        server_18.shutdown()
        server_18.server_close()
        thread_18.join(10)

with checked("B7.18"):                                    # running(): always stopped, even after an error
    if "serve_18" not in globals() or ARTIFACT_B is None:
        raise StopCheck
    import requests
    backends_18 = ["stdlib"] + (["fastapi"] if importlib.util.find_spec("fastapi") and importlib.util.find_spec("uvicorn") else [])
    for backend_18 in backends_18:
        threads_before_18 = threading.active_count()
        port_18, health_18 = None, None
        try:
            with serve_18.running(ARTIFACT_B, backend=backend_18) as url_18:
                port_18 = int(str(url_18).rsplit(":", 1)[-1])
                health_18 = requests.get(f"{url_18}/health", timeout=10).status_code
                raise KeyError("an error inside the block (a test of the check cell)")
        except KeyError:
            pass
        time.sleep(0.2)
        stopped_18 = port_18 is not None and not wbp.port_open(port_18)
        ok_18[f"running {backend_18}"] = health_18 == 200 and stopped_18 and threading.active_count() <= threads_before_18
        verdict("B7.18", ok_18[f"running {backend_18}"], f"running(backend=\"{backend_18}\") : /health répond, puis le "
                "serveur s'arrête et libère son port, même après une erreur dans le bloc.",
                f"running(backend=\"{backend_18}\") : /health {health_18}, port {port_18} "
                f"{'encore ouvert' if port_18 and not stopped_18 else 'fermé'}, threads {threads_before_18} → "
                f"{threading.active_count()} : attendu 200, puis le serveur arrêté (shutdown, server_close, ou "
                "should_exit pour uvicorn) dans un finally.")
        url_a_18 = url_b_18 = None
        try:
            with serve_18.running(ARTIFACT_B, backend=backend_18) as url_a_18, \
                    serve_18.running(ARTIFACT_B, backend=backend_18) as url_b_18:
                pass
        except Exception as err_18:  # noqa: BLE001 - the learner's code: "Address already in use", a server that did not start
            url_b_18 = f"une erreur ({type(err_18).__name__} : {str(err_18)[:80]})"
        ok_18[f"free port {backend_18}"] = url_a_18 is not None and url_b_18 is not None and url_a_18 != url_b_18
        verdict("B7.18", ok_18[f"free port {backend_18}"], f"running(backend=\"{backend_18}\") deux fois à la fois : deux "
                "ports libres.", f"deux running(backend=\"{backend_18}\") à la fois donnent {url_a_18} et {url_b_18} : un "
                "port en dur ne sert qu'une fois (port 0 : le système choisit un port libre).")
    try:
        with wbp.serve_process(PROJECT / "src", ARTIFACT_B, backend="stdlib", timeout=60) as proc_18:
            code_health_18 = requests.get(f"{proc_18.url}/health", timeout=10).status_code
    except RuntimeError as err_18:
        code_health_18 = f"pas de réponse ({str(err_18)[:200]})"
    ok_18["main"] = code_health_18 == 200
    verdict("B7.18", ok_18["main"], "python -m cifarprod.serve --artifact ... --port ... sert l'artefact dans un autre "
            "processus (arrêté ensuite).", f"python -m cifarprod.serve --artifact ... --port ... --backend stdlib : "
            f"/health {code_health_18} ; attendu 200.")
    r_18 = wbp.run([sys.executable, "-m", "cifarprod.serve", "--artifact", scratch("empty"), "--port",
                    str(wbp.free_port())], PROJECT, src=[PROJECT / "src"], timeout=60)
    ok_18["main error"] = run_line("B7.18", r_18, r_18.returncode == 1 and bool(r_18.stderr_lines("ERROR")),
                                   "un dossier sans artefact : code 1 et une ligne ERROR.",
                                   "attendu, pour un dossier sans artefact, le code 1 et une ligne ERROR")
if ok_18 and all(ok_18.values()) and len(ok_18) >= 9:
    print("✅ Ex B7.18 : ton API répond comme le notebook, refuse proprement ce qui est mal formé, et s'arrête toujours.")

### Ex B7.19 — Prédire la latence p95 de l'API, requête par requête puis par batches de 32 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que coûte le chemin HTTP d'une prédiction, et ce que change un batch, avant de le mesurer.  
**Prérequis :** Ex B7.18 · ✏️ B7.3 · fiche §207.7 (« Latence, percentiles et débit »), §207.8 · **Fil rouge :** CIFAR-10 · **Parcours :** C

L'expérience sert le VGG du MP5 par l'API **de référence** du chapitre (FastAPI et uvicorn, dans un autre processus, arrêté à la fin), pour que tes prédictions se comparent à des mesures qui ne dépendent pas de ton code ; ton API est mesurée ensuite, sans vérification. Un client `requests` (une `Session`, qui garde la connexion) envoie 200 requêtes `POST /predict` d'une image, après 20 appels à blanc, puis 40 requêtes `POST /predict_batch` de 32 images ; elle mesure aussi l'appel direct du modèle (`model(x)`, sans HTTP, sous `torch.inference_mode()`). Le modèle (appel direct) et le serveur tournent sur **un** thread, comme un service lancé en plusieurs processus sur une machine à peu de cœurs (207.8 : threads × processus ≈ cœurs). Les durées sont celles que voit le client, en millisecondes. **Avant d'exécuter**, écris tes deux prédictions (une lettre chacune) :
- a) `http_19` : le p95 de `POST /predict` (une image), divisé par le p95 de l'appel direct du modèle sur une image, vaut : **A** moins de 2 (le HTTP ne coûte presque rien) ; **B** de 2 à 20 ; **C** plus de 20 (le modèle ne pèse presque rien dans la requête) ;
- b) `batch_19` : le p95 de `POST /predict_batch` (32 images), divisé par le p95 de `POST /predict` (une image), vaut : **A** moins de 3 (32 images coûtent à peine plus qu'une) ; **B** de 3 à 32 ; **C** plus de 32 (pas mieux que 32 requêtes l'une après l'autre).

Ce sont des **rapports** de durées mesurées sur la même machine : ils dépendent bien moins du processeur que les durées elles-mêmes, que la cellule rapporte aussi à la vitesse de la machine de génération du chapitre. La cellule d'expérience n'affiche rien tant que tes deux prédictions ne sont pas écrites ; elle les vérifie à la fin.

In [ ]:
http_19, batch_19 = "B", "B"

In [ ]:
wb.record("B7.19a", "B", mistakes={"relis le tableau : la ligne POST /predict, 1 image, contre l'appel direct": {"A", "C"}}, prediction=True)
wb.record("B7.19b", "B", mistakes={"relis le tableau : le p95 de /predict_batch contre celui de /predict": {"A", "C"}}, prediction=True)

In [ ]:
if any(answer is ... or answer is None for answer in [http_19, batch_19]):
    print("⏳ Ex B7.19 : écris d'abord tes deux prédictions dans la cellule précédente.")
else:
    import requests

    start_19 = time.perf_counter()
    CHOICES_19 = {"a": {"A": "< 2", "B": "2 à 20", "C": "> 20"}, "b": {"A": "< 3", "B": "3 à 32", "C": "> 32"}}


    def category_19(question, ratio):
        bounds = {"a": (2, 20), "b": (3, 32)}[question]
        return "A" if ratio < bounds[0] else "B" if ratio <= bounds[1] else "C"


    folder_19 = wbp.reference_artifact(scratch("artifact19") / "vgg")
    model_19, info_19 = wbp.reference_model()
    x_19 = model_input(X_B, info_19["mean"], info_19["std"])
    with wbp.torch_state_kept(), torch.inference_mode():        # one thread, as the server: one image at a time
        torch.set_num_threads(1)
        direct1_19 = wbp.latencies(lambda: model_19(x_19[:1]), 200, 20)
        direct32_19 = wbp.latencies(lambda: model_19(x_19[:32]), 50, 5)
    one_body_19, batch_body_19 = {"pixels": X_B[0].tolist()}, {"images": X_B[:32].tolist()}
    with wbp.serve_process(wbp.reference_package() / "src", folder_19, backend="auto", env={"OMP_NUM_THREADS": "1"}) as server_19:
        with requests.Session() as session_19:                  # closed at the end: no connection left open
            http1_19 = wbp.latencies(lambda: session_19.post(f"{server_19.url}/predict", json=one_body_19, timeout=30), 200, 20)
            http32_19 = wbp.latencies(lambda: session_19.post(f"{server_19.url}/predict_batch", json=batch_body_19, timeout=30),
                                      40, 3)
    rows_19 = {"appel direct, 1 image": (direct1_19, 1), "appel direct, batch de 32": (direct32_19, 32),
               "POST /predict, 1 image": (http1_19, 1), "POST /predict_batch, 32 images": (http32_19, 32)}
    print(f"{'':34s} {'p50 (ms)':>9s} {'p95 (ms)':>9s} {'images/s':>9s}")
    for name_19, (ms_19, n_19) in rows_19.items():
        p50_19, p95_19 = np.percentile(ms_19, [50, 95])
        print(f"{name_19:34s} {p50_19:9.2f} {p95_19:9.2f} {n_19 * 1000 / p50_19:9.0f}")
    ratio_a_19 = np.percentile(http1_19, 95) / np.percentile(direct1_19, 95)
    ratio_b_19 = np.percentile(http32_19, 95) / np.percentile(http1_19, 95)
    print(f"a) p95 de /predict ÷ p95 de l'appel direct : {fr(ratio_a_19, 1)} (catégorie {category_19('a', ratio_a_19)})")
    print(f"b) p95 de /predict_batch (32 images) ÷ p95 de /predict : {fr(ratio_b_19, 1)} (catégorie {category_19('b', ratio_b_19)}) ; "
          f"débit : {fr(32 * np.percentile(http1_19, 50) / np.percentile(http32_19, 50), 1)} fois celui des requêtes d'une image")
    print(f"ℹ️ vitesse de cette machine : un appel direct à une image, p50 {fr(np.percentile(direct1_19, 50), 2)} ms (machine de "
          f"génération du chapitre : 0,96 ms) ; les catégories sont des rapports, qui dépendent peu de la machine. "
          f"Mesures en {fr(time.perf_counter() - start_19, 0)} s, serveur arrêté.")
    if (category_19("a", ratio_a_19), category_19("b", ratio_b_19)) != (str(http_19).strip().upper(), str(batch_19).strip().upper()):
        print("ℹ️ ta prédiction et la mesure de cette machine diffèrent : la vérification ci-dessous juge la catégorie "
              "mesurée sur la machine de génération ; compare aussi aux mesures de plusieurs exécutions (05_solutions.md).")
    wb.check("B7.19a", http_19)
    wb.check("B7.19b", batch_19)

In [ ]:
if any(answer is ... or answer is None for answer in [http_19, batch_19]):
    print('⏳ Ex B7.19 : la mesure de ton API vient après tes prédictions.')
else:
    # The same measures on YOUR API (B7.18), if it passed: nothing is checked here
    if "ok_18" in globals() and ok_18 and all(ok_18.values()):
        try:
            with serve_18.running(ARTIFACT_B) as url_own_19, requests.Session() as s_own_19:
                own1_19 = wbp.latencies(lambda: s_own_19.post(f"{url_own_19}/predict", json=one_body_19, timeout=30), 100, 10)
                own32_19 = wbp.latencies(lambda: s_own_19.post(f"{url_own_19}/predict_batch", json=batch_body_19, timeout=30), 20, 2)
            print(f"ton API : /predict p50 {fr(np.percentile(own1_19, 50), 2)} ms, p95 {fr(np.percentile(own1_19, 95), 2)} ms ; "
                  f"/predict_batch p50 {fr(np.percentile(own32_19, 50), 1)} ms, p95 {fr(np.percentile(own32_19, 95), 1)} ms.")
            slow_19 = np.percentile(own1_19, 50) / np.percentile(http1_19, 50)
            if slow_19 > 3:
                print(f"ℹ️ ton API est {fr(slow_19, 0)} fois plus lente que celle de référence pour la même requête : ≈ 40 ms de "
                      "plus par requête est le signe d'un accusé de réception retardé (algorithme de Nagle) : un socket sans "
                      "TCP_NODELAY (uvicorn avec sockets=[...]), ou des en-têtes et un corps envoyés en plusieurs fois ; "
                      "compare avec running(..., backend=\"stdlib\").")
        except Exception as error_19:  # noqa: BLE001 - the learner's code: say it, nothing else depends on it
            print(f"ℹ️ ton API n'a pas pu être mesurée : {type(error_19).__name__} : {str(error_19)[:160]}")
    else:
        print("⏳ Ex B7.19 : la mesure de ton API attend B7.18 (toutes ses vérifications).")

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Où passe le temps d'une requête d'une image, en dehors du modèle (pense à ce que fait le JSON de 3 072 entiers, à l'aller et au retour) ? Pourquoi le débit gagné par le batch est-il bien inférieur à 32 ? Avec un objectif « 95 % des requêtes en moins de 100 ms » (un SLO), lequel des deux usages le tiendrait ici, et que ferais-tu du regroupement dynamique (*dynamic batching*, 207.7) ? Compare avec les nombres synthétiques de ✏️ B7.3.

### Ex B7.20 — Accélérer l'inférence sur CPU : batches, threads, inference_mode, bfloat16, quantification 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer, sur la machine qui sert, ce que chaque levier change à la latence et aux décisions du modèle.  
**Prérequis :** Ex B7.19 · ✏️ B7.1 (la part de la couche dense) · ch. 24 (bfloat16) · fiche §207.8 (« Accélérer sur CPU ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

**Le contrat** des quatre fonctions de la cellule suivante (leurs docstrings le donnent) :
- `latency_20(fn, x, n=50, warmup=5)` rend `(p50, p95)` en millisecondes, deux floats Python, de `n` appels `fn(x)` mesurés par `time.perf_counter`, après `warmup` appels à blanc non mesurés ;
- `bfloat16_20(model)` rend une fonction `x -> logits` : une **copie** du modèle convertie en `torch.bfloat16`, appliquée à `x` converti, qui rend des logits `float32` ; le modèle donné ne change pas ;
- `quantized_20(model)` rend une **copie** du modèle dont les `nn.Linear` sont quantifiées en int8 par `torch.ao.quantization.quantize_dynamic` (`dtype=torch.qint8`) ; dans PyTorch 2.11, elle affiche un avertissement de dépréciation (207.8), que la cellule tait ;
- `agreement_20(a, b)` rend `(part, écart)` pour deux tenseurs de logits `(N, C)` : la part des lignes dont les argmax sont les mêmes, et le plus grand écart absolu, deux floats Python.

La cellule de vérification essaie chaque fonction sur un petit cas (une fonction lente pendant ses appels à blanc, puis rapide ; des logits dont trois argmax sur quatre s'accordent ; le type et la structure des modèles rendus). L'expérience mesure ensuite le VGG du MP5 sur une grille : batch de 1 et de 32 ; `inference_mode`, `no_grad` ou l'autograd actif ; le nombre de threads par défaut ou un seul ; float32, bfloat16 ou int8 dynamique. Pour chaque ligne, le p50, le p95 et les images par seconde ; puis, sur 1 000 images de test, l'accuracy de chaque précision et l'accord de ses décisions avec le float32. **Aucune mesure n'est enregistrée** : elles dépendent de la machine et varient d'une exécution à l'autre ; réexécute la cellule deux fois pour voir ce qui tient.

Dans tes notes : quels leviers accélèrent vraiment ici, lesquels ne changent rien, lesquels **ralentissent**, et pourquoi (pense au processeur, à ce qu'il sait calculer en 16 bits, et à la part des paramètres que la quantification dynamique touche, ✏️ B7.1 j) ? Les décisions changent-elles, et sur combien d'images sur 1 000 ? Lequel de ces leviers mettrais-tu en production pour un service qui reçoit une image à la fois, et lequel pour une prédiction par batches de nuit ? Que retiens-tu de l'écart entre deux exécutions de la même ligne ?

In [ ]:
import copy


def latency_20(fn, x, n=50, warmup=5):
    """(p50, p95), two Python floats in ms, of n calls fn(x) measured with time.perf_counter, after `warmup` calls of
    fn(x) that are not measured."""
    for _ in range(warmup):
        fn(x)
    times = []
    for _ in range(n):
        start = time.perf_counter()
        fn(x)
        times.append(1000 * (time.perf_counter() - start))
    p50, p95 = np.percentile(times, [50, 95])
    return float(p50), float(p95)


def bfloat16_20(model):
    """A function x -> logits: a COPY of the model converted to torch.bfloat16, applied to x converted to bfloat16; the
    logits are returned as float32. The model given is not changed."""
    low = copy.deepcopy(model).to(torch.bfloat16)
    return lambda x: low(x.to(torch.bfloat16)).float()


def quantized_20(model):
    """A COPY of the model whose nn.Linear layers are quantized to int8 by torch.ao.quantization.quantize_dynamic
    (dtype=torch.qint8); the model given is not changed."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)      # « torch.ao.quantization is deprecated » (2.11)
        return torch.ao.quantization.quantize_dynamic(copy.deepcopy(model), {torch.nn.Linear}, dtype=torch.qint8)


def agreement_20(a, b):
    """(share, gap) for two tensors of logits (N, C): the share of the rows whose argmax agree and the largest absolute
    difference between them, two Python floats."""
    a, b = torch.as_tensor(a).float(), torch.as_tensor(b).float()
    return float((a.argmax(dim=1) == b.argmax(dim=1)).float().mean()), float((a - b).abs().max())


tools_ok_20 = False
warnings.filterwarnings("ignore", message=r".*torch\.ao\.quantization is deprecated.*")
with checked("B7.20"):
    calls_20 = []

    def probe_20(x):                                      # slow while warming up, then fast: what is measured?
        calls_20.append(time.perf_counter())
        time.sleep(0.03 if len(calls_20) <= 3 else 0.001)

    got_20 = latency_20(probe_20, None, n=8, warmup=3)
    ok_lat_20 = (isinstance(got_20, tuple) and len(got_20) == 2 and all(type(v) is float for v in got_20)
                 and len(calls_20) == 11 and 0.5 < got_20[0] < 15 and got_20[0] <= got_20[1] < 25)
    verdict("B7.20", ok_lat_20, f"latency_20 : (p50, p95) en ms, les 3 appels à blanc écartés ({fr(got_20[0], 1)} et "
            f"{fr(got_20[1], 1)} ms pour un appel d'environ 1 ms).",
            f"latency_20(fn, x, n=8, warmup=3) sur une fonction qui dort 30 ms pendant les 3 premiers appels puis 1 ms : "
            f"attendu 11 appels et (p50, p95) en ms, deux floats d'environ 1 ms (les 3 appels à blanc, lents, ne sont pas "
            f"mesurés) ; reçu {len(calls_20)} appels et {got_20!r}.")
    model_20 = wbp.reference_model()[0]
    x_20 = model_input(X_B[:16], REF_INFO["mean"], REF_INFO["std"])
    with torch.inference_mode():
        ref_20 = model_20(x_20)
        low_20 = bfloat16_20(model_20)(x_20)
    ok_bf_20 = (isinstance(low_20, torch.Tensor) and low_20.dtype == torch.float32 and tuple(low_20.shape) == (16, 10)
                and next(model_20.parameters()).dtype == torch.float32 and 0 < float((low_20 - ref_20).abs().max()) < 1)
    verdict("B7.20", ok_bf_20, "bfloat16_20 : des logits float32 calculés en bfloat16 (proches, pas égaux), le modèle "
            "d'origine intact.", f"bfloat16_20 : attendu une fonction qui rend des logits float32 (16, 10) calculés en "
            f"bfloat16, sans changer le modèle donné ; reçu {getattr(low_20, 'dtype', type(low_20).__name__)}, modèle "
            f"{next(model_20.parameters()).dtype}.")
    model_20 = wbp.reference_model()[0]                     # a new copy: bfloat16_20 may have changed the first one
    q_20 = quantized_20(model_20)
    kinds_20 = {type(m).__module__ + "." + type(m).__name__ for m in q_20.modules()}
    linear_q_20 = any("quantized" in k and k.endswith("Linear") for k in kinds_20)
    intact_20 = type(model_20[-1]) is torch.nn.Linear and q_20 is not model_20
    convs_20 = any(type(m) is torch.nn.Conv2d for m in q_20.modules())
    ok_q_20 = linear_q_20 and intact_20 and convs_20
    verdict("B7.20", ok_q_20, "quantized_20 : une copie dont la couche dense est en int8 dynamique, les convolutions en "
            "float32, le modèle d'origine intact.", "quantized_20 : attendu une COPIE dont les nn.Linear sont "
            "quantifiées (quantize_dynamic, qint8) et les convolutions inchangées ; "
            + "; ".join(part for part in ["aucune couche dense quantifiée" if not linear_q_20 else "",
                                          "le modèle donné a changé (inplace=True, ou pas de copie ?)" if not intact_20 else "",
                                          "les convolutions ont changé" if not convs_20 else ""] if part) + ".")
    a_20 = torch.tensor([[3.0, 0, 0], [0, 2.0, 0], [0, 0, 1.0], [1.0, 0, 0]])
    b_20 = torch.tensor([[2.5, 0, 0], [0, 2.0, 0], [0, 0, 1.0], [0, 1.5, 0]])
    got_ag_20 = agreement_20(a_20, b_20)
    ok_ag_20 = (isinstance(got_ag_20, tuple) and len(got_ag_20) == 2 and all(type(v) is float for v in got_ag_20)
                and abs(got_ag_20[0] - 0.75) < 1e-9 and abs(got_ag_20[1] - 1.5) < 1e-6)
    verdict("B7.20", ok_ag_20, "agreement_20 : 0,75 et 1,5 sur un petit cas (trois argmax sur quatre en accord).",
            f"agreement_20 sur quatre lignes dont trois argmax en accord et un écart maximal de 1,5 : attendu (0.75, 1.5), "
            f"deux floats ; reçu {got_ag_20!r}.")
    tools_ok_20 = bool(ok_lat_20 and ok_bf_20 and ok_q_20 and ok_ag_20)

In [ ]:
if not tools_ok_20:
    print("⏳ Ex B7.20 : l'expérience attend tes quatre fonctions justes (cellule précédente).")
else:
    with checked("B7.20"), wbp.torch_state_kept(), warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        start_20 = time.perf_counter()
        fp32_20 = wbp.reference_model()[0]
        bf16_20, int8_20 = bfloat16_20(fp32_20), quantized_20(fp32_20)
        X1000_20, Y1000_20 = wbp.test_images(100)
        x_all_20 = model_input(X1000_20, REF_INFO["mean"], REF_INFO["std"])
        x1_20, x32_20 = x_all_20[:1], x_all_20[:32]
        threads_20 = torch.get_num_threads()

        def timed_20(fn, x, mode="inference", threads=None):
            torch.set_num_threads(threads or threads_20)
            try:
                if mode == "inference":
                    with torch.inference_mode():
                        return latency_20(fn, x)
                if mode == "no_grad":
                    with torch.no_grad():
                        return latency_20(fn, x)
                return latency_20(fn, x)                # autograd on: a graph is built at each call
            finally:
                torch.set_num_threads(threads_20)

        grid_20 = [("batch 1, float32, inference_mode", fp32_20, x1_20, "inference", None),
                   ("batch 1, float32, no_grad", fp32_20, x1_20, "no_grad", None),
                   ("batch 1, float32, autograd actif", fp32_20, x1_20, "autograd", None),
                   ("batch 1, float32, 1 thread", fp32_20, x1_20, "inference", 1),
                   ("batch 1, bfloat16", bf16_20, x1_20, "inference", None),
                   ("batch 1, int8 dynamique", int8_20, x1_20, "inference", None),
                   ("batch 32, float32, inference_mode", fp32_20, x32_20, "inference", None),
                   ("batch 32, float32, 1 thread", fp32_20, x32_20, "inference", 1),
                   ("batch 32, bfloat16", bf16_20, x32_20, "inference", None),
                   ("batch 32, int8 dynamique", int8_20, x32_20, "inference", None)]
        print(f"{threads_20} threads par défaut ici. {'':10s} {'p50 (ms)':>9s} {'p95 (ms)':>9s} {'images/s':>9s}")
        for name_20, fn_20, x_20, mode_20, th_20 in grid_20:
            p50_20, p95_20 = timed_20(fn_20, x_20, mode_20, th_20)
            print(f"{name_20:36s} {p50_20:9.2f} {p95_20:9.2f} {len(x_20) * 1000 / p50_20:9.0f}")
        with torch.inference_mode():
            logits_20 = {"float32": fp32_20(x_all_20), "bfloat16": bf16_20(x_all_20), "int8 dynamique": int8_20(x_all_20)}
        print("\nSur 1 000 images de test de CIFAR-10 :")
        for name_20, out_20 in logits_20.items():
            share_20, gap_20 = agreement_20(logits_20["float32"], out_20)
            acc_20 = float((out_20.argmax(dim=1).numpy() == Y1000_20).mean())
            print(f"  {name_20:15s} accuracy {fr(acc_20, 3)} ; accord des argmax avec float32 {fr(100 * share_20, 1)} % ; "
                  f"écart maximal des logits {gap_20:.3g}")
        print(f"(mesures en {fr(time.perf_counter() - start_20, 0)} s ; aucune n'est enregistrée : elles dépendent de la "
              "machine, et d'une exécution à l'autre)")

### Ex B7.21 — Exporter avec torch.export et vérifier l'équivalence numérique (ONNX en option) 📦 ★★ ⏱️ 25 min
**Objectif :** exporter le modèle servi en un graphe qui se recharge sans ton code, et vérifier qu'il calcule la même chose.  
**Prérequis :** Ex B7.17 · 🧠 Q10 · fiche §207.8 (« torch.export et ONNX ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

> **Mode d'emploi du paquet** : chaque fichier de ton paquet est une chaîne de la cellule à compléter (`DATA_PY = r'''...'''`) ; la cellule de vérification l'**écrit** dans ton projet (`mon_travail/b7_production/cifar-prod/`, à côté de ce notebook), puis l'importe ou le lance dans un autre processus. Pour corriger un fichier, corrige sa chaîne et réexécute les deux cellules. Si tu préfères un éditeur, modifie le fichier sur le disque : la cellule le voit et ne l'écrase plus (elle le dit).

**Le contrat** de `src/cifarprod/export.py` (les docstrings dans la cellule à compléter) :
- `export_model(model, path, max_batch=1024)` met le modèle en mode `eval()`, l'exporte par `torch.export.export` avec une taille de batch **libre** de 1 à `max_batch` (à partir d'un batch d'exemple de 2 images `(2, 3, 32, 32)`), enregistre l'`ExportedProgram` dans `path` (un `.pt2`, `torch.export.save`) et rend ce chemin ;
- `export_artifact(folder, max_batch=1024)` exporte le modèle de l'artefact `folder` (`load_artifact`) dans `folder / "model.pt2"` et rend ce chemin ;
- `max_gap(model, exported, x)` rend le plus grand écart absolu entre les logits du modèle et ceux d'`exported` sur le batch `x`, calculés sans graphe, un float Python.

La cellule de vérification exporte `ARTIFACT_B` avec ton `export_artifact`, puis lance un **processus sans ton paquet** (`cifarprod` n'y est pas importable) qui recharge `model.pt2` par `torch.export.load` et calcule des batches de 1, 7 et 64 images de test : l'écart maximal avec `MODEL_B` doit rester **au plus** $10^{-6}$ (un verdict, puis un `wb.check` de ce verdict : rien à taper). Ce processus essaie aussi un batch de 1 025 images et une image de 28 × 28, que les gardes doivent refuser. Puis elle exporte avec ton `export_model` un modèle **laissé en mode entraînement** par son appelant (l'export doit être celui du mode `eval()`), et essaie `max_gap` sur un modèle dont on a décalé les biais de sortie de 0,01. L'export ONNX n'est essayé que si `onnxscript` et `onnxruntime` sont installés (extra `prod`, ni dans le workbook ni sur Colab).

Dans tes notes : que vaut l'écart, et pourquoi n'est-ce pas surprenant (que fait l'export des opérations du modèle) ? Que donnerait un export fait en mode entraînement sur ce VGG, et pourquoi la batchnorm en est-elle la cause ? Pourquoi le batch d'exemple a-t-il 2 images et pas 1 ? Combien pèse `model.pt2` à côté de `model.safetensors`, et que contient-il de plus ? Quand choisirais-tu ONNX plutôt que `torch.export` ?

In [ ]:
# Ex B7.21: src/cifarprod/export.py
EXPORT_PY = r'''"""Export a model with torch.export, and check that the export computes what the model computes.

    path = export_artifact("artifact")                 # artifact/model.pt2, a batch of 1 to 1024 images
    exported = torch.export.load(path).module()        # a callable, without the code of cifarprod
    gap = max_gap(model, exported, x)                  # of the order of the float32 rounding, or 0

An ExportedProgram holds the graph of the operations, the weights and the guards (the shapes it accepts): it is
served without the Python code that built the model (another program, C++, a phone) and refuses an input of another
shape instead of computing in silence.
"""

from __future__ import annotations

from pathlib import Path

from .artifact import load_artifact


def export_model(model, path, max_batch: int = 1024) -> Path:
    """Export ``model`` in eval() mode (it is switched to it) with torch.export, the batch size free from 1 to
    ``max_batch`` (dynamic_shapes on the dimension 0 of the argument "input"), on an example batch of 2 images of
    3 x 32 x 32, and save the ExportedProgram in ``path`` (.pt2). Returns the path."""
    import torch

    model.eval()
    batch = torch.export.Dim("batch", min=1, max=int(max_batch))
    example = (torch.zeros(2, 3, 32, 32),)
    program = torch.export.export(model, example, dynamic_shapes={"input": {0: batch}})
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.export.save(program, str(path))
    return path


def export_artifact(folder, max_batch: int = 1024) -> Path:
    """The model of the artifact ``folder`` (load_artifact) exported into ``folder / "model.pt2"``."""
    model, _ = load_artifact(folder)
    return export_model(model, Path(folder) / "model.pt2", max_batch)


def max_gap(model, exported, x) -> float:
    """The largest absolute difference between the logits of ``model`` and of ``exported`` on the batch ``x``
    (float32 (N, 3, 32, 32)), computed without a graph, as a Python float."""
    import torch

    with torch.inference_mode():
        return float((model(x).double() - exported(x).double()).abs().max())
'''


ok_21 = {}
GAP_21 = None                                            # the largest gap over the batches 1, 7 and 64
with checked("B7.21"), warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r".*given buffer is not writable.*")
    if ARTIFACT_B is None:
        print("⏳ Ex B7.21 : l'export part de l'artefact de B7.17 : fais d'abord passer B7.17.")
        raise StopCheck
    wbp.write_files(PROJECT, {"src/cifarprod/export.py": EXPORT_PY})
    export_21 = wbp.import_module(PROJECT, "export")
    with wbp.torch_state_kept():
        path_21 = export_21.export_artifact(ARTIFACT_B)
    ok_21["file"] = path_21 is not None and Path(path_21).resolve() == (ARTIFACT_B / "model.pt2").resolve() \
        and Path(path_21).is_file()
    if not ok_21["file"]:
        verdict("B7.21", False, "", f"export_artifact doit écrire et rendre {ARTIFACT_B.name}/model.pt2 ; rendu {path_21!r}.")
        raise StopCheck
    sizes_21 = (Path(path_21).stat().st_size, (ARTIFACT_B / "model.safetensors").stat().st_size)
    bytes_21 = [f"{n:_}".replace("_", " ") for n in sizes_21]
    verdict("B7.21", True, f"export_artifact écrit model.pt2 dans l'artefact ({bytes_21[0]} octets, contre "
            f"{bytes_21[1]} pour les poids seuls).", "")
    io_21 = scratch("export")
    x_21 = model_input(X_B[:64], INFO_B["mean"], INFO_B["std"]).numpy()
    np.save(io_21 / "x.npy", x_21)
    probe_21 = ("import json, sys, warnings\nsys.modules['cifarprod'] = None   # not importable here, even if installed\n"
                "import numpy as np\nimport torch\nwarnings.simplefilter('ignore')\n"
                "exported = torch.export.load(sys.argv[1]).module()\nx = torch.from_numpy(np.load(sys.argv[2]))\n"
                "with torch.inference_mode():\n    for b in (1, 7, 64):\n        np.save(f'{sys.argv[3]}/out{b}.npy', exported(x[:b]).numpy())\n"
                "refused = {}\nfor name, bad in {'1025': torch.zeros(1025, 3, 32, 32), '28': torch.zeros(2, 3, 28, 28)}.items():\n"
                "    try:\n        exported(bad)\n        refused[name] = None\n    except Exception as err:\n"
                "        refused[name] = str(err).splitlines()[0][:120]\n"
                "try:\n    import cifarprod\n    reachable = True\nexcept ImportError:\n    reachable = False\n"
                "print(json.dumps({'refused': refused, 'cifarprod': reachable}))\n")
    (io_21 / "reload_pt2.py").write_text(probe_21, encoding="utf-8")
    r_21 = wbp.run([sys.executable, io_21 / "reload_pt2.py", path_21, io_21 / "x.npy", io_21], scratch("elsewhere"))
    seen_21 = r_21.last_json() if r_21.ok else None
    if not run_line("B7.21", r_21, r_21.ok and isinstance(seen_21, dict) and not seen_21["cifarprod"],
                    "un processus SANS ton paquet recharge model.pt2 et calcule.",
                    "attendu un processus sans cifarprod qui recharge model.pt2 (torch.export.load) et calcule les "
                    "batches de 1, 7 et 64 images"):
        raise StopCheck
    with torch.inference_mode():                             # the model on the SAME batches as the export
        want_21 = {b: MODEL_B(torch.from_numpy(x_21[:b])).double().numpy() for b in (1, 7, 64)}
    gaps_21 = {b: float(np.abs(np.load(io_21 / f"out{b}.npy").astype(np.float64) - want_21[b]).max()) for b in (1, 7, 64)}
    GAP_21 = max(gaps_21.values())
    ok_21["equal"] = GAP_21 <= 1e-6
    verdict("B7.21", ok_21["equal"], "l'export calcule ce que calcule le modèle, à 1e-6 près, pour 1, 7 et 64 images "
            f"(écarts : {', '.join(f'{v:.2g}' for v in gaps_21.values())}).",
            f"écarts entre le modèle et model.pt2 : {', '.join(f'batch {b} : {v:.3g}' for b, v in gaps_21.items())} ; "
            "attendu au plus 1e-6 : la taille de batch est-elle libre (dynamic_shapes), le modèle en mode eval() ?")
    ok_21["guards"] = all(seen_21["refused"].values())
    verdict("B7.21", ok_21["guards"], f"les gardes refusent un batch de 1 025 images et une image de 28 × 28 "
            f"(« {seen_21['refused'].get('28')} »).",
            f"un modèle exporté doit refuser ce qui sort de ses gardes : batch de 1 025 images "
            f"({'refusé' if seen_21['refused'].get('1025') else 'accepté'}), image de 28 × 28 "
            f"({'refusée' if seen_21['refused'].get('28') else 'acceptée'}) : max_batch=1024 dans Dim.")
    trained_21 = wbp.reference_model()[0].train()           # a model left in training mode by its caller
    with wbp.torch_state_kept():
        path_t_21 = export_21.export_model(trained_21, io_21 / "train.pt2")
        exported_t_21 = torch.export.load(str(path_t_21)).module()
        with torch.inference_mode():
            gap_t_21 = float((wbp.reference_model()[0](torch.from_numpy(x_21)) - exported_t_21(torch.from_numpy(x_21))).abs().max())
    ok_21["eval"] = gap_t_21 <= 1e-6 and not trained_21.training
    verdict("B7.21", ok_21["eval"], "export_model met le modèle en mode eval() avant d'exporter (un modèle reçu en mode "
            "entraînement donne le même export).", f"export_model d'un modèle reçu en mode train() : écart {gap_t_21:.3g} "
            "avec le modèle en mode eval() (attendu au plus 1e-6) : en mode entraînement, la batchnorm de l'export "
            "normalise par les statistiques de chaque batch.")
    shifted_21 = wbp.reference_model()[0]
    with torch.no_grad():
        shifted_21[-1].bias += 0.01
    got_gap_21 = export_21.max_gap(wbp.reference_model()[0], shifted_21, torch.from_numpy(x_21[:8]))
    ok_21["max_gap"] = type(got_gap_21) is float and abs(got_gap_21 - 0.01) < 1e-5
    verdict("B7.21", ok_21["max_gap"], "max_gap : 0,01 pour un modèle dont on a décalé les biais de sortie de 0,01.",
            f"max_gap(modèle, modèle aux biais de sortie décalés de 0.01, x) : attendu le float 0.01, reçu {got_gap_21!r}.")
    if importlib.util.find_spec("onnxscript") and importlib.util.find_spec("onnxruntime"):
        import onnxruntime
        onnx_path_21 = io_21 / "model.onnx"
        torch.onnx.export(wbp.reference_model()[0], (torch.from_numpy(x_21[:2]),), str(onnx_path_21),
                          dynamic_shapes={"input": {0: torch.export.Dim("batch", min=1, max=1024)}})
        session_21 = onnxruntime.InferenceSession(str(onnx_path_21))
        out_onnx_21 = session_21.run(None, {session_21.get_inputs()[0].name: x_21})[0]
        print(f"ℹ️ ONNX (option) : écart maximal avec le modèle {float(np.abs(out_onnx_21 - want_21[64]).max()):.2g} sur 64 images.")
    else:
        print("ℹ️ ONNX (option) : onnxscript et onnxruntime ne sont pas installés ici (extra prod) : l'export ONNX n'est "
              "pas essayé.")
if ok_21 and all(ok_21.values()) and len(ok_21) == 5:
    print("✅ Ex B7.21 : ton modèle exporté se recharge sans ton code et calcule exactement ce que calcule le modèle.")

In [ ]:
wb.record("B7.21", True, mistakes={"ton export s'écarte du modèle de plus de 1e-6 : la taille de batch est-elle libre, "
                                   "et le modèle en mode eval() quand tu l'exportes ?": False})

In [ ]:
# Is the largest gap at most 1e-6? (nothing to type: the value comes from your code)
if GAP_21 is None:
    print("⏳ Ex B7.21 : l'écart attend ton export (cellule précédente).")
else:
    wb.check("B7.21", bool(GAP_21 <= 1e-6), computed=True)

## Partie C · L'exploitation : la dérive, l'intégration continue, le Dockerfile et le défi

Le service tourne ; il faut maintenant le **surveiller** (B7.22 : des images plus sombres ou bruitées arrivent, lesquelles faut-il signaler ?), **protéger** chaque modification par une intégration continue (B7.23), **emballer** le service dans une image Docker (B7.24), et rendre le tout reproductible par n'importe qui, de `git clone` à une prédiction servie (🏆 B7.25). Rien n'est envoyé sur GitHub ni construit par Docker : le workflow et le Dockerfile sont vérifiés par lecture, et le défi se joue sur un clone local de ton dépôt.

### Ex B7.22 — Surveiller la dérive : PSI et test KS sur des images assombries ou bruitées, quand alerter ? 📈 ★★ ⏱️ 25 min
**Objectif :** mesurer une dérive sur les entrées et sur les sorties, et décider de ce qui mérite une alerte.  
**Prérequis :** ✏️ B7.4 · Ex B7.17 · ch. 8 (p-valeur) · fiche §207.10 (PSI, test KS, « Agir ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

Une **référence** (500 images de test, et les prédictions du VGG du MP5 sur elles) et quatre **fenêtres** de production de 500 **autres** images de test, comme un service pourrait les recevoir : **A** telles quelles ; **B** plus sombres (pixels × 0,6 : une caméra au crépuscule) ; **C** bruitées (un bruit gaussien d'écart-type 25 sur l'échelle 0-255 : un capteur moins cher) ; **D** un peu plus sombres (pixels × 0,9). On surveille une variable des **entrées**, la luminosité moyenne de chaque image, et une des **sorties**, la classe prédite.

**Le contrat** des trois fonctions de la cellule suivante (leurs docstrings le donnent) :
- `brightness_22(images)` : la luminosité moyenne de chaque image `uint8` `(N, 32, 32, 3)`, la moyenne de ses 3 072 valeurs divisée par 255, un tableau `float64` `(N,)` ;
- `psi_22(ref, prod, bins=10, eps=1e-4)` : le PSI de 207.10 sur `bins` intervalles coupés aux quantiles $k/\text{bins}$ de la **référence** (`np.quantile`, $k = 1, \dots, \text{bins} - 1$ : des déciles, et non les quartiles de ✏️ B7.4), le premier et le dernier ouverts ; une valeur égale à une coupure va dans l'intervalle **au-dessus** ; $\varepsilon$ ajouté à **toutes** les parts ; un float Python ;
- `psi_classes_22(ref_labels, prod_labels, n_classes=10, eps=1e-4)` : le même PSI sur les parts des dix classes prédites.

La cellule de vérification compare tes fonctions à un calcul écrit autrement, sur quatre cas (la référence contre elle-même, des valeurs tirées ailleurs, des parts nulles, des valeurs égales aux coupures). Puis le tableau : pour chaque fenêtre, le PSI de la luminosité, le test KS sur la luminosité (`scipy.stats.ks_2samp` : $D$ et la p-valeur), le PSI des classes prédites, la confiance moyenne (la probabilité de la classe prédite) et l'accuracy (avec les vrais labels, connus ici, rarement en production) ; et deux figures, les histogrammes de la luminosité et les parts des classes prédites. **Lis-les**, puis réponds dans la cellule qui suit (des lettres de fenêtres, dans n'importe quel ordre, entre guillemets) :
- a) `inputs_22` : les fenêtres dont le PSI de la luminosité dépasse 0,25 (« changement important ») ;
- b) `ks_22` : les fenêtres où le test KS sur la luminosité rejette à 5 % ($p < 0{,}05$) ;
- c) `outputs_22` : les fenêtres dont le PSI des classes prédites dépasse 0,25 ;
- d) `harm_22` : les fenêtres où l'accuracy perd plus de 10 points par rapport à la référence ;
- e) `psi_d_22` : le PSI de la luminosité de la fenêtre D, à 2 décimales.

Dans tes notes : pour chaque fenêtre, que ferais-tu de l'alerte (rien, surveiller de plus près, vérifier la chaîne d'acquisition, revenir en arrière ou réentraîner, 207.10) ? Quelle fenêtre une surveillance des seules entrées laisserait-elle passer, et pourquoi ? Le test KS et l'accuracy disent-ils la même chose pour chaque fenêtre, et que deviendrait la p-valeur du test KS de D avec 50 000 images par fenêtre ? Que changerait une référence prise sur les images d'entraînement plutôt que sur le test ?

In [ ]:
# A reference (500 test images, the model's predictions on them) and four production windows of 500 OTHER test images,
# as a service could receive them: A as they are; B darker (pixels x 0.6, a camera at dusk); C noisy (Gaussian noise
# of standard deviation 25 on the 0-255 scale, a cheaper sensor); D a little darker (pixels x 0.9)
import matplotlib.pyplot as plt
from scipy import stats

X_22, Y_22 = wbp.test_images(100)                        # 1 000 test images, 100 per class
REF_22, REF_Y_22 = X_22[::2], Y_22[::2]
PROD_22, PROD_Y_22 = X_22[1::2], Y_22[1::2]
WINDOWS_22 = {"A": PROD_22,
              "B": np.round(PROD_22 * 0.6).astype(np.uint8),
              "C": np.clip(PROD_22 + np.random.default_rng(22).normal(0, 25, PROD_22.shape), 0, 255).round().astype(np.uint8),
              "D": np.round(PROD_22 * 0.9).astype(np.uint8)}
MODEL_22 = wbp.reference_model()[0]                      # the served model (the reference VGG of the MP5)
PROBA_22 = {name: offline_b(MODEL_22, images, REF_INFO["mean"], REF_INFO["std"])[0]
            for name, images in {"ref": REF_22, **WINDOWS_22}.items()}
print(f"ℹ️ référence : {len(REF_22)} images ; fenêtres A à D : {len(PROD_22)} images chacune (d'autres images du test).")

In [ ]:
def brightness_22(images):
    """The mean brightness of each uint8 image (N, 32, 32, 3): the mean of its 3 072 values divided by 255, a float64
    NumPy array (N,)."""
    images = np.asarray(images)
    return images.reshape(len(images), -1).mean(axis=1, dtype=np.float64) / 255


def psi_22(ref, prod, bins=10, eps=1e-4):
    """The PSI of the values `prod` against the reference `ref` (1-D arrays): `bins` intervals cut at the quantiles
    k / bins of ref (np.quantile, k = 1 .. bins - 1), the first and the last open; a value equal to a cut goes into
    the interval above it; the shares q_k (ref) and p_k (prod), eps added to every share; the sum of
    (p_k - q_k) ln(p_k / q_k), a Python float."""
    cuts = np.quantile(ref, np.arange(1, bins) / bins)
    q = np.bincount(np.searchsorted(cuts, ref, side="right"), minlength=bins) / len(ref) + eps
    p = np.bincount(np.searchsorted(cuts, prod, side="right"), minlength=bins) / len(prod) + eps
    return float(np.sum((p - q) * np.log(p / q)))


def psi_classes_22(ref_labels, prod_labels, n_classes=10, eps=1e-4):
    """The PSI of the shares of the classes 0 .. n_classes - 1 of prod_labels against those of ref_labels (integer
    arrays), eps added to every share, a Python float."""
    q = np.bincount(ref_labels, minlength=n_classes) / len(ref_labels) + eps
    p = np.bincount(prod_labels, minlength=n_classes) / len(prod_labels) + eps
    return float(np.sum((p - q) * np.log(p / q)))


tools_ok_22 = False


def oracle_psi_22(ref, prod, bins=10, eps=1e-4):
    """The PSI written another way (np.digitize, right=False: a value equal to a cut goes above it)."""
    cuts = np.quantile(np.asarray(ref, float), [k / bins for k in range(1, bins)])
    share = lambda v: np.array([(np.digitize(v, cuts) == k).mean() for k in range(bins)]) + eps
    q, p = share(np.asarray(ref, float)), share(np.asarray(prod, float))
    return float(((p - q) * np.log(p / q)).sum())


with checked("B7.22"):
    light_22 = brightness_22(REF_22[:5])
    want_light_22 = REF_22[:5].astype(np.float64).mean(axis=(1, 2, 3)) / 255
    ok_b_22 = isinstance(light_22, np.ndarray) and light_22.shape == (5,) and np.allclose(light_22, want_light_22, rtol=0, atol=1e-12)
    verdict("B7.22", ok_b_22, f"brightness_22 : la luminosité moyenne de chaque image ({fr(float(want_light_22[0]), 4)} pour "
            "la première).", f"brightness_22 de 5 images : attendu un tableau (5,) de moyennes dans [0, 1], reçu "
            f"{np.asarray(light_22)!r:.120}.")
    cases_22 = {"la référence contre elle-même": (np.linspace(0, 1, 101), np.linspace(0, 1, 101)),
                "des valeurs tirées ailleurs": (np.random.default_rng(0).normal(0, 1, 400), np.random.default_rng(1).normal(0.5, 1, 300)),
                "tout dans un intervalle (des parts nulles)": (np.random.default_rng(2).uniform(0, 1, 200), np.full(50, 0.99)),
                "des valeurs égales aux coupures": (np.array([0, 0, 0, 1, 2, 2, 4, 4, 4, 5, 5, 6, 7, 7], float),
                                                    np.array([5, 5, 4, 4, 7, 2, 6, 5, 0, 3, 6, 4], float))}
    ok_p_22 = True
    for label_22, (r_22, p_22) in cases_22.items():
        with np.errstate(divide="ignore", invalid="ignore"):         # a log(0) of your code: the value says it
            got_22, want_22 = psi_22(r_22, p_22), oracle_psi_22(r_22, p_22)
        good_22 = type(got_22) is float and abs(got_22 - want_22) <= 1e-9 * max(1, abs(want_22))
        ok_p_22 &= good_22
        verdict("B7.22", good_22, f"psi_22, {label_22} : {fr(want_22, 4)}.",
                f"psi_22, {label_22} : attendu le float {want_22:.6f}, reçu {got_22!r} (coupures aux déciles de la "
                "référence, une valeur égale à une coupure va dans l'intervalle au-dessus, eps ajouté à chaque part).")
    with np.errstate(divide="ignore", invalid="ignore"):
        got_c_22 = psi_classes_22(np.arange(100) % 10, np.r_[np.zeros(40, int), np.arange(60) % 6])
    q_c_22 = np.full(10, 0.1) + 1e-4
    p_c_22 = np.bincount(np.r_[np.zeros(40, int), np.arange(60) % 6], minlength=10) / 100 + 1e-4
    want_c_22 = float(((p_c_22 - q_c_22) * np.log(p_c_22 / q_c_22)).sum())
    ok_c_22 = type(got_c_22) is float and abs(got_c_22 - want_c_22) < 1e-9
    verdict("B7.22", ok_c_22, f"psi_classes_22 : {fr(want_c_22, 4)} quand quatre classes disparaissent.",
            f"psi_classes_22 : attendu {want_c_22:.6f} (dix classes, quatre absentes de la production, eps ajouté à "
            f"chaque part), reçu {got_c_22!r}.")
    tools_ok_22 = bool(ok_b_22 and ok_p_22 and ok_c_22)

In [ ]:
if not tools_ok_22:
    print("⏳ Ex B7.22 : le tableau attend tes trois fonctions justes (cellule précédente).")
else:
    LIGHT_22 = {name: brightness_22(images) for name, images in {"ref": REF_22, **WINDOWS_22}.items()}
    PRED_22 = {name: proba.argmax(axis=1) for name, proba in PROBA_22.items()}
    acc_ref_22 = float((PRED_22["ref"] == REF_Y_22).mean())
    print(f"{'fenêtre':8s} {'PSI lum.':>9s} {'KS D':>6s} {'KS p':>9s} {'PSI classes':>12s} {'confiance':>10s} "
          f"{'accuracy':>9s}")
    print(f"{'réf.':8s} {'':>9s} {'':>6s} {'':>9s} {'':>12s} {PROBA_22['ref'].max(axis=1).mean():10.3f} {acc_ref_22:9.3f}")
    TABLE_22 = {}
    for name_22 in WINDOWS_22:
        ks_22 = stats.ks_2samp(LIGHT_22["ref"], LIGHT_22[name_22])
        TABLE_22[name_22] = {"psi": psi_22(LIGHT_22["ref"], LIGHT_22[name_22]), "D": float(ks_22.statistic),
                             "p": float(ks_22.pvalue), "psi_classes": psi_classes_22(PRED_22["ref"], PRED_22[name_22]),
                             "confidence": float(PROBA_22[name_22].max(axis=1).mean()),
                             "accuracy": float((PRED_22[name_22] == PROD_Y_22).mean())}
        row_22 = TABLE_22[name_22]
        print(f"{name_22:8s} {row_22['psi']:9.3f} {row_22['D']:6.3f} {row_22['p']:9.2g} {row_22['psi_classes']:12.3f} "
              f"{row_22['confidence']:10.3f} {row_22['accuracy']:9.3f}")
    print("PSI lum. : la luminosité moyenne de chaque image, 10 intervalles aux déciles de la référence ; KS : le test de "
          "Kolmogorov-Smirnov sur cette luminosité ; PSI classes : la part de chaque classe prédite ; confiance : la "
          "probabilité moyenne de la classe prédite ; accuracy : avec les vrais labels, connus ici, rarement en production.")
    fig_22, axes_22 = plt.subplots(1, 2, figsize=(12, 3.8))
    edges_22 = np.linspace(0, 1, 41)
    axes_22[0].hist(LIGHT_22["ref"], bins=edges_22, color="0.6", label="référence")
    for name_22, color_22 in zip(WINDOWS_22, ("tab:blue", "tab:orange", "tab:green", "tab:red")):
        axes_22[0].hist(LIGHT_22[name_22], bins=edges_22, histtype="step", lw=1.8, color=color_22, label=f"fenêtre {name_22}")
    axes_22[0].set(xlabel="luminosité moyenne d'une image", ylabel="images", title="Les entrées")
    axes_22[0].legend(fontsize=8)
    width_22 = 0.16
    for j_22, (name_22, color_22) in enumerate(zip(["ref", *WINDOWS_22], ("0.6", "tab:blue", "tab:orange", "tab:green", "tab:red"))):
        shares_22 = np.bincount(PRED_22[name_22], minlength=10) / len(PRED_22[name_22])
        axes_22[1].bar(np.arange(10) + (j_22 - 2) * width_22, shares_22, width_22, color=color_22,
                       label="référence" if name_22 == "ref" else f"fenêtre {name_22}")
    axes_22[1].set_xticks(np.arange(10), [FRENCH_16[c] if "FRENCH_16" in globals() else c for c in CLASSES], rotation=45,
                          fontsize=8)
    axes_22[1].set(ylabel="part des images", title="Les sorties : classes prédites")
    axes_22[1].legend(fontsize=8)
    fig_22.tight_layout()
    plt.show()

In [ ]:
inputs_22, ks_22, outputs_22, harm_22 = "B", "BD", "C", "C"
psi_d_22 = round(TABLE_22["D"]["psi"], 2)

In [ ]:
wb.record("B7.22a", "B", mistakes={"au moins une des fenêtres choisies n'a pas sa place : relis la colonne PSI lum. et le seuil 0,25": {'BCD', 'AB', 'ABC', 'ABCD', 'BD', 'ABD', 'BC'}, "il manque au moins une fenêtre, et une autre n'a pas sa place : relis la colonne PSI lum. et le seuil 0,25": {'C', 'AD', 'A', 'ACD', 'AC', 'D', 'CD'}})
wb.record("B7.22b", "BD", mistakes={"il manque au moins une fenêtre : relis la colonne KS p et le seuil 0,05": {'B', 'D'}, "au moins une des fenêtres choisies n'a pas sa place : relis la colonne KS p et le seuil 0,05": {'ABCD', 'BCD', 'ABD'}, "il manque au moins une fenêtre, et une autre n'a pas sa place : relis la colonne KS p et le seuil 0,05": {'C', 'AD', 'AB', 'A', 'ABC', 'ACD', 'AC', 'BC', 'CD'}})
wb.record("B7.22c", "C", mistakes={"au moins une des fenêtres choisies n'a pas sa place : relis la colonne PSI classes et le seuil 0,25": {'BCD', 'ABCD', 'ABC', 'ACD', 'AC', 'BC', 'CD'}, "il manque au moins une fenêtre, et une autre n'a pas sa place : relis la colonne PSI classes et le seuil 0,25": {'AD', 'AB', 'A', 'BD', 'ABD', 'B', 'D'}})
wb.record("B7.22d", "C", mistakes={"au moins une des fenêtres choisies n'a pas sa place : relis la colonne accuracy, comparée à celle de la référence": {'BCD', 'ABCD', 'ABC', 'ACD', 'AC', 'BC', 'CD'}, "il manque au moins une fenêtre, et une autre n'a pas sa place : relis la colonne accuracy, comparée à celle de la référence": {'AD', 'AB', 'A', 'BD', 'ABD', 'B', 'D'}})
wb.record("B7.22e", psi_d_22, decimals=2, mistakes={"relis la question : la fenêtre D, colonne PSI lum.": {round(TABLE_22[k]["psi"], 2) for k in "ABC"}, "relis la question : le PSI de la luminosité, pas celui des classes": round(TABLE_22["D"]["psi_classes"], 2), "relis la question : le PSI, pas la statistique D du test KS": round(TABLE_22["D"]["D"], 2)})

### Ex B7.23 — Intégration continue : un workflow GitHub Actions qui installe, teste et lance un entraînement fumée 🛠️ ★★ ⏱️ 25 min
**Objectif :** écrire le workflow qui rejoue l'installation, les tests et un entraînement fumée à chaque push.  
**Prérequis :** Ex B7.15 · fiche §207.6 (« L'intégration continue »), §207.9 (secrets) · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

**Le contrat** de `.github/workflows/ci.yml` (la fiche en donne le début) : déclenché à chaque `push` et `pull_request` ; un job sur `ubuntu-latest` ; les étapes, dans cet ordre : `actions/checkout@v7` ; `actions/setup-python@v7` avec `python-version: "3.13"` (**entre guillemets**) et `cache: pip` ; l'installation de torch depuis l'**index CPU** de PyTorch, **puis** celle de ton paquet en mode éditable avec son extra `dev` (`pip install -e ".[dev]"`, ou `".[dev,serve]"` pour les tests de l'API) ; `pytest` (les tests `slow` compris) ; un **entraînement fumée** : `scripts/make_synthetic.py` écrit un fichier de données synthétiques (les données du projet ne sont pas dans git), puis `cifarprod train --config configs/smoke.yaml --set data=<ce fichier>`. Aucune étape n'avale son échec (`|| true`, `continue-on-error`), aucun secret n'est écrit en clair (un secret passerait par `${{ secrets.NOM }}`).

La cellule suivante donne `scripts/make_synthetic.py` (256 images d'entraînement et 64 de validation tirées par `synthetic` de B7.9, au schéma du fichier du projet). La cellule de vérification écrit les deux fichiers dans ton projet et relit ton workflow **sans le lancer** (ni GitHub ni Docker ne sont là) : PyYAML d'abord (attention, il lit la clé `on:` comme le booléen `True` : le vérificateur le sait, pas forcément ton prochain script), puis chaque point du contrat, l'ordre des étapes, la syntaxe de chaque bloc `run:` par `bash -n` (qui lit sans exécuter), et la présence dans le projet des fichiers que le workflow cite.

Dans tes notes : que se passe-t-il, sur GitHub, si `python-version: 3.10` est écrit sans guillemets ? Pourquoi installer torch **avant** le paquet ? Pourquoi un entraînement fumée **en plus** des tests, et pourquoi sur des données synthétiques ? Combien de temps prendrait ce workflow (pense au cache de pip et au téléchargement de torch CPU), et que verrait un recruteur sur la page de ton dépôt ?

In [ ]:
# scripts/make_synthetic.py (given): the data file of the smoke training in the continuous integration, which has no CIFAR-10
MAKE_SYNTHETIC_PY = r'''"""Write a small data file with the schema of the project (random images): python scripts/make_synthetic.py PATH.

The continuous integration has no CIFAR-10 (the data are not in git): its smoke training runs on this file.
"""

import sys
from pathlib import Path

import numpy as np

from cifarprod.data import synthetic


def main(argv=None) -> int:
    args = sys.argv[1:] if argv is None else argv
    path = Path(args[0] if args else "data/synthetic.npz")
    path.parent.mkdir(parents=True, exist_ok=True)
    X, y = synthetic(256, seed=0)
    X_val, y_val = synthetic(64, seed=1)
    np.savez(path, x_train=X, y_train=y, x_val=X_val, y_val=y_val)
    print(path)
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''

In [ ]:
# Ex B7.23: .github/workflows/ci.yml
CI_YML = r'''# Continuous integration: on every push and pull request, install the package on a clean machine, run the tests,
# then a smoke training on synthetic data (the CIFAR-10 subset is not in git). A step that fails stops the job.
name: ci
on:
  push:
  pull_request:
jobs:
  test:
    runs-on: ubuntu-latest
    timeout-minutes: 20
    steps:
      - uses: actions/checkout@v7
      - uses: actions/setup-python@v7
        with:
          python-version: "3.13"
          cache: pip
      - name: Install PyTorch (CPU index) and the package
        run: |
          python -m pip install --upgrade pip
          pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cpu
          pip install -e ".[dev,serve]"
      - name: Tests
        run: pytest -q
      - name: Smoke training on synthetic data
        run: |
          python scripts/make_synthetic.py data/synthetic.npz
          cifarprod train --config configs/smoke.yaml --set data=data/synthetic.npz --out runs/ci
'''


ok_23 = {}
with checked("B7.23"):
    wbp.write_files(PROJECT, {"scripts/make_synthetic.py": MAKE_SYNTHETIC_PY, ".github/workflows/ci.yml": CI_YML})
    if CI_YML is ...:
        print("⏳ Ex B7.23 : écris d'abord CI_YML.")
        raise StopCheck
    for item_23 in wbp.review_workflow(CI_YML, PROJECT):
        ok_23[item_23.key] = item_23.ok
        verdict("B7.23", item_23.ok, item_23.success, item_23.failure)
        if item_23.key == "yaml" and not item_23.ok:
            raise StopCheck
if ok_23 and all(ok_23.values()):
    print("✅ Ex B7.23 : ton workflow installe, teste et entraîne à chaque push (vérifié par lecture : rien n'est lancé ici). "
          "Sur GitHub, l'onglet Actions montre son premier run après ton prochain push.")

### Ex B7.24 — Lire et compléter un Dockerfile : image de base, dépendances CPU, utilisateur non root, variables 📦 ★★ ⏱️ 20 min
**Objectif :** lire un Dockerfile, en repérer les erreurs, et le compléter selon les bonnes pratiques d'une image de service.  
**Prérequis :** Ex B7.18 · fiche §207.9 (« Image, conteneur et Dockerfile », « Variables d'environnement et secrets ») · **Parcours :** C

Un collègue a commencé le Dockerfile de ton API (`DOCKERFILE`, dans la cellule suivante) : plusieurs lignes sont fausses, d'autres manquent (les `TODO`). **Lis-le, corrige-le et complète-le** pour qu'il construise l'image du service sur CPU. **Le contrat** (le tableau de 207.9) :
- une image de base **slim** de Python 3.13 à une **version précise** (pas un tag qui bouge) ;
- des réglages en `ENV`, **jamais un secret** : `PYTHONUNBUFFERED=1` et `MODEL_DIR`, le dossier de l'artefact dans l'image ;
- les dépendances **avant** le code : `COPY requirements-serve.txt`, `RUN` torch `2.11.0` depuis l'index CPU, puis `pip install -r requirements-serve.txt` ; ensuite seulement `pyproject.toml` et `src/`, le paquet lui-même (`pip install --no-deps .`), et le dossier `artifact/` (`MODEL_DIR` le désigne) ;
- un utilisateur **sans privilèges**, créé (`useradd`) puis choisi (`USER`) avant la commande ;
- `EXPOSE 8000` et une commande en **forme exec** (une liste JSON) qui lance ton API (`python -m cifarprod.serve`, B7.18) sur `0.0.0.0` et le port exposé.

Écris aussi `DOCKERIGNORE` (ce qui ne doit jamais entrer dans l'image : au moins `.git`, `data/`, `runs/`, les caches `__pycache__` et un éventuel `.env`) et `REQUIREMENTS_SERVE`, les dépendances de l'API (`numpy`, `pyyaml`, `safetensors`, `fastapi`, `uvicorn`, `pydantic`), chacune **figée** avec `==` à la version de ton environnement (`importlib.metadata.version(...)`), sans torch. La cellule de vérification écrit les trois fichiers dans ton projet, relit le Dockerfile **sans le construire** (Docker n'est ni sur Colab ni dans le workbook), point par point, et lance `python -m cifarprod.serve --help` pour vérifier que la commande de `CMD` existe dans ton paquet.

Dans tes notes : pour chaque ligne fausse du collègue, que se serait-il passé (au build, au lancement, ou six mois plus tard) ? Pourquoi `COPY . .` avant `pip install` rend-il chaque build lent ? Pourquoi `--host 0.0.0.0` dans un conteneur, et pourquoi pas sur ton PC ? Où mettrais-tu la clé d'API qu'il avait écrite, et que doit-il faire maintenant qu'elle a été écrite dans un fichier ?

In [ ]:
# Ex B7.24: the Dockerfile of the reference package, its .dockerignore and its requirements

# Dockerfile
DOCKERFILE = r'''# The prediction API of cifarprod, on CPU. Build: docker build -t cifar-api .  Run: docker run -p 8000:8000 cifar-api
# (an artifact folder, written by cifarprod train --out artifact, must be next to this file)
FROM python:3.13.18-slim-trixie

# settings, never secrets: unbuffered logs, no .pyc, no pip cache, the folder of the model
ENV PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1 \
    PIP_NO_CACHE_DIR=1 \
    MODEL_DIR=/app/artifact

WORKDIR /app

# the dependencies first: this layer stays in the cache while only the code changes; torch from the CPU index
COPY requirements-serve.txt .
RUN pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cpu \
    && pip install -r requirements-serve.txt

# then the code and the artifact of the model
COPY pyproject.toml .
COPY src/ src/
RUN pip install --no-deps .
COPY artifact/ artifact/

# a user without privileges runs the server
RUN groupadd --system app && useradd --system --no-log-init --gid app app
USER app

EXPOSE 8000
CMD ["python", "-m", "cifarprod.serve", "--host", "0.0.0.0", "--port", "8000"]
'''

# .dockerignore
DOCKERIGNORE = r'''.git
.github
.venv
.env
data/
runs/
tests/
**/__pycache__
*.egg-info
.pytest_cache
'''

# requirements-serve.txt
REQUIREMENTS_SERVE = r'''# The dependencies of the API, pinned (the versions of the workbook, which are those of Colab); torch is installed
# apart, from the CPU index of PyTorch (see the Dockerfile)
numpy==2.1.3
pyyaml==6.0.3
safetensors==0.8.0
fastapi==0.141.1
uvicorn==0.53.0
pydantic==2.13.5
'''


ok_24 = {}
with checked("B7.24"):
    if DOCKERIGNORE is ... or REQUIREMENTS_SERVE is ...:
        print("⏳ Ex B7.24 : écris aussi DOCKERIGNORE et REQUIREMENTS_SERVE.")
        raise StopCheck
    wbp.write_files(PROJECT, {"Dockerfile": DOCKERFILE, ".dockerignore": DOCKERIGNORE,
                              "requirements-serve.txt": REQUIREMENTS_SERVE})
    for item_24 in wbp.review_dockerfile(DOCKERFILE, DOCKERIGNORE, REQUIREMENTS_SERVE, PROJECT):
        ok_24[item_24.key] = item_24.ok
        verdict("B7.24", item_24.ok, item_24.success, item_24.failure)
if ok_24 and all(ok_24.values()):
    print("✅ Ex B7.24 : ton Dockerfile suit les bonnes pratiques (vérifié par lecture : Docker n'est pas lancé). Pour le "
          "construire sur ta machine : cifarprod train --out artifact, puis docker build -t cifar-api . et "
          "docker run -p 8000:8000 cifar-api.")

### Ex B7.25 — Défi : un dépôt prêt pour la production, de git clone à une prédiction servie en moins de 10 min de CPU 🏆 ★★★★ ⏱️ 180 min
**Objectif :** livrer un dépôt que n'importe qui clone, installe, teste et fait prédire en une commande.  
**Prérequis :** Ex B7.9 à B7.24 · fiche §207.1 à §207.9 · le ML Test Score (§207.6) · **Fil rouge :** CIFAR-10 · **Parcours :** C

**Le défi.** Ton dépôt `cifar-prod`, tel que B7.9 à B7.24 l'ont construit, doit aller **de `git clone` à une prédiction servie en moins de 10 minutes de CPU** sur une machine neuve, et se juger sur une **grille de 10 critères**, notée **par toi**, chaque critère à 0, ½ ou 1 point (la grille détaillée, avec ce qui vaut ½ et 1, est dans `05_solutions.md`) :

| # | Critère | Ce qu'on regarde |
|---|---|---|
| 1 | installation sur un clone neuf | `git clone`, puis `pip install -e ".[dev,serve]"` (après torch CPU), sans autre geste |
| 2 | `make` ou script unique | une commande, `python scripts/demo.py` (ou `make demo`), de l'entraînement à la prédiction servie |
| 3 | tests verts | `pytest -q` passe sur le clone, sans le fichier de données |
| 4 | intégration continue | `.github/workflows/ci.yml` (B7.23) |
| 5 | configuration | `configs/`, validée (B7.11) |
| 6 | artefact versionné | version, empreintes des données et des poids (B7.17) |
| 7 | API qui répond | `/health` et `/predict` (B7.18) |
| 8 | latence mesurée | un p50 et un p95, en ms, dans le README |
| 9 | README | ce que c'est, installer, tester, entraîner, servir, les résultats |
| 10 | fiche modèle | `MODEL_CARD.md` : modèle, usage prévu, données, performances, limites (ch. 23, B6) |

**À écrire** dans la cellule suivante : `README_MD`, `MODEL_CARD_MD` et `DEMO_PY` (`scripts/demo.py`, dont la docstring est le contrat : dans un clone neuf, sans `data/` ni `runs/`, des données synthétiques si le fichier manque, un entraînement fumée par ta ligne de commande, l'artefact, l'API sur un port libre et toujours arrêtée, `/health`, des requêtes `/predict` pour la latence, et pour dernière ligne de stdout un résumé JSON `{"status": 200, "label", "probability", "p50_ms", "p95_ms", "artifact"}`). Puis **commite** ton projet (`git init`, `git add -A`, `git commit`, dans un terminal, dans son dossier) : l'auto-contrôle clone ce que tu as commité, comme le ferait une autre machine. Ton projet devient ainsi un dépôt git **dans** celui du workbook : `git add mon_travail` (00_setup) n'en enregistre plus les fichiers, seulement une référence (git avertit : *embedded git repository*). C'est voulu, ton projet a son propre historique : publie-le comme un dépôt à part (c'est le but du défi), et note son adresse dans ta copie de `06_mes_reponses.md`.

**L'auto-contrôle** (la cellule de vérification) clone ton dépôt dans un dossier temporaire (ou le copie, s'il n'est pas un dépôt git), l'installe dans un environnement virtuel neuf, y lance `pytest`, puis ta démo, et chronomètre chaque étape. Il ne télécharge rien (il réutilise les bibliothèques du notebook, et vérifie donc à part que tes tests et tes scripts n'importent que des dépendances déclarées dans `pyproject.toml`) : il compte ≈ 3 min pour PyTorch CPU sur une machine neuve, et ramène la durée mesurée à la vitesse de la machine de référence du chapitre (`wbp.machine_speed()`, un petit calcul fixe). Il affiche ensuite, pour chaque critère, un indice (✅ ou ❌) tiré de ce qu'il a vu : **une aide pour ta grille, pas une note** ; aucune note n'est calculée, ni avant ni après.

Dans tes notes : ta note sur 10, critère par critère, et ce qui manque pour chaque demi-point perdu. Qu'est-ce que l'auto-contrôle ne peut pas voir (la qualité de ton README, la justesse de ta fiche modèle, ce qui n'est pas commité) ? Si tu publies le dépôt sur GitHub, que montre l'onglet Actions après ton premier push ? Quel test du ML Test Score (207.6) ton dépôt passe-t-il, et lequel ajouterais-tu en premier ?

In [ ]:
# Ex B7.25: the README, the model card and the demo of the reference package; then the project is committed with git

# README.md
README_MD = r'''# cifar-prod · un classifieur CIFAR-10, du notebook à la production

Un classifieur d'images 32 × 32 (les dix classes de CIFAR-10) livré comme un projet : un paquet Python installable
(`cifarprod`), une ligne de commande d'entraînement pilotée par une configuration validée, des runs reproductibles et
journalisés, des tests, un artefact de modèle qui se suffit, une API HTTP, un export `torch.export`, une intégration
continue et un Dockerfile. Projet du chapitre bonus B7 du workbook ; le modèle servi par défaut est le VGG 16-32-64 du
mini-projet 5 (85,0 % d'accuracy sur les 10 000 images de test de CIFAR-10, voir `MODEL_CARD.md`).

## Installer

```bash
git clone <ce dépôt> cifar-prod && cd cifar-prod
pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cpu   # PyTorch CPU d'abord (Linux, Windows)
pip install -e ".[dev,serve]"                                                 # le paquet, pytest, FastAPI et uvicorn
```

Python 3.11 ou plus (testé avec 3.13 et PyTorch 2.11). Sans l'index CPU, `pip install torch` télécharge sous Linux la
version CUDA (plus de 2 Go).

## De git clone à une prédiction servie, en une commande

```bash
python scripts/demo.py        # ou : make demo
```

La démo prend `data/cifar10_2000.npz` s'il est là (2 000 images de CIFAR-10 ; les données ne sont pas dans git), sinon
un fichier synthétique ; elle lance un entraînement fumée (`configs/smoke.yaml`), écrit l'artefact dans `runs/demo/`,
sert l'API sur un port libre, l'interroge, puis l'arrête. Sa dernière ligne est un résumé JSON (classe, probabilité,
p50 et p95 de la latence). Sur un CPU à deux cœurs : ≈ 7 s, installation de PyTorch non comprise.

## Tester, entraîner, servir

```bash
pytest -q                                                    # 34 tests (les tests de données sautent sans le fichier)
cifarprod train --config configs/fast.yaml --out artifact    # 2 000 images, 1 epoch, ≈ 4 s ; --set lr=0.01 pour surcharger
python -m cifarprod.serve --artifact artifact --port 8000    # GET /health, POST /predict, POST /predict_batch
curl -s localhost:8000/health
```

Chaque run ajoute une ligne à `runs/runs.jsonl` (configuration, environnement, empreinte des données, métriques,
artefacts) ; codes de sortie : 0 succès, 2 configuration invalide, 1 données illisibles ou entraînement en échec.
L'artefact (`model.safetensors` et `meta.json`) porte l'architecture, la normalisation, les classes dans l'ordre des
sorties, la version du paquet, l'empreinte SHA-256 des données d'entraînement et celle des poids, que le service
vérifie au chargement. `python -c "from cifarprod.export import export_artifact; export_artifact('artifact')"` écrit
`artifact/model.pt2`, rechargeable sans ce code (écart avec le modèle : 0,0 sur 1, 7 et 64 images).

## Résultats et latence

| Modèle | Données | Accuracy |
|---|---|---|
| VGG 16-32-64 (MP5), servi par l'API | 45 000 images, 30 epochs | 85,0 % (test, 10 000 images) |
| CNN 16-32, `configs/fast.yaml` | 2 000 images, 1 epoch | 30,2 % (validation, 500 images) ; 31,6 % ± 2,6 sur 5 graines |

Latence du VGG servi par FastAPI et uvicorn sur un CPU à deux cœurs (200 requêtes après 20 appels à blanc, client
`requests` sur la même machine) : `POST /predict` d'une image, p50 ≈ 5 à 8 ms, **p95 ≈ 8 à 10 ms** ; `POST
/predict_batch` de 32 images, p50 ≈ 70 à 90 ms, p95 ≈ 105 à 120 ms, soit 2 à 3,5 fois le débit des requêtes d'une
image. L'appel direct du modèle à une image prend ≈ 1 ms : l'essentiel du temps d'une requête est le HTTP et le JSON.

## Déployer

```bash
cifarprod train --config configs/fast.yaml --out artifact
docker build -t cifar-api . && docker run -p 8000:8000 cifar-api
```

L'image part de `python:3.13.18-slim-trixie`, installe PyTorch depuis l'index CPU et les versions figées de
`requirements-serve.txt`, et fait tourner le serveur sans privilèges. L'intégration continue
(`.github/workflows/ci.yml`) installe, teste et lance un entraînement fumée à chaque push.

## Organisation

```text
pyproject.toml   configs/   scripts/ (demo.py, make_synthetic.py)   Dockerfile   Makefile   MODEL_CARD.md
src/cifarprod/   data.py model.py train.py config.py cli.py tracking.py artifact.py serve.py export.py
tests/           test_data.py test_model.py test_cli.py test_serve.py
```
'''

# MODEL_CARD.md
MODEL_CARD_MD = r'''# Fiche modèle · cifar-prod (VGG 16-32-64 servi par une API)

## Le modèle

Le VGG du mini-projet 5 : trois étages de deux convolutions 3 × 3 (16, 32 puis 64 canaux), chacune suivie d'une
batchnorm et d'une ReLU, un max pooling après les deux premiers étages, un pooling global moyen et une couche dense
vers les dix logits ; **72 954 paramètres**. Entraîné le 2026-10-29 sur 45 000 images de CIFAR-10 (30 epochs, SGD de
Nesterov, `OneCycleLR`, label smoothing 0,1, augmentation), 28 minutes sur un CPU à deux cœurs. L'artefact
(`model.safetensors`, `meta.json`, version du paquet 0.1.0) contient la normalisation apprise sur l'entraînement
(moyenne 0,4916 ; 0,4823 ; 0,4466 ; écart-type 0,2471 ; 0,2435 ; 0,2615, pixels ramenés dans [0, 1]), l'ordre des
classes, l'empreinte SHA-256 des images et des labels d'entraînement (`b76c6f2c…`) et celle des poids.

## Usage prévu

Classer des photos couleur 32 × 32 dans les dix classes de CIFAR-10, par une API HTTP (`POST /predict`), pour
l'enseignement et un portfolio. **Usages exclus** : toute décision qui compte pour une personne, des images d'autres
classes ou d'une autre nature (le modèle choisit toujours une des dix classes), un contexte adversarial.

## Les données

CIFAR-10 (A. Krizhevsky, 2009) : 60 000 photos 32 × 32 étiquetées à la main, 6 000 par classe, tirées du web des
années 2000 (base *80 Million Tiny Images*, retirée en 2020). Entraînement : 45 000 des 50 000 images d'entraînement
(découpage stratifié, graine 0) ; validation : les 5 000 autres ; test : 10 000 images, ouvert une fois.

## Les performances

Accuracy de test **85,0 %** (± 0,7 point à 95 %) ; de 71,2 % (chat) à 93,4 % (bateau) selon la classe. Servi par
FastAPI sur un CPU à deux cœurs : p95 ≈ 8 à 10 ms par requête d'une image. L'export `torch.export` donne exactement les
mêmes logits que le modèle.

## Les limites, la surveillance et les risques

- **Sensible au bruit** : un bruit gaussien d'écart-type 25 (sur 255) sur les pixels fait tomber l'accuracy de 83 % à
  36 % sur 500 images de test, sans changer la luminosité moyenne : la surveillance des entrées par la seule
  luminosité ne le voit pas, celle des classes prédites (PSI) le voit.
- **Peu sensible à l'éclairage** : des images assombries (pixels × 0,6) changent beaucoup la distribution des entrées
  (PSI de la luminosité au-dessus de 3) pour une perte d'accuracy de 3 points.
- Des probabilités non calibrées, aucune robustesse adversariale (FGSM à ε = 1/255 : 39 % d'accuracy), une seule graine.
- **Surveiller** : la distribution des classes prédites et la confiance moyenne par fenêtre (PSI à 10 intervalles,
  seuils 0,1 et 0,25), la luminosité et le contraste des entrées, la latence p95 et le taux de réponses 4xx et 5xx ;
  garder l'artefact précédent pour revenir en arrière.
- **Données personnelles** : une photo envoyée peut montrer une personne ; les réponses 422 ne renvoient pas l'entrée,
  les journaux ne gardent que des statistiques d'images, pour une durée écrite.
'''

# scripts/demo.py
DEMO_PY = r'''"""From a fresh clone to a served prediction, in one command: python scripts/demo.py

1. the data: data/cifar10_2000.npz if it is there, else a synthetic file (scripts/make_synthetic.py);
2. a smoke training with configs/smoke.yaml, through the command line (cifarprod train): the artifact of the model
   (model.safetensors, meta.json) in runs/demo/;
3. the API on a free port, in a thread (cifarprod.serve.running, always stopped at the end): GET /health, then
   POST /predict with one image, 50 times, for the latency.
The last line of stdout is a JSON summary: {"status", "label", "probability", "p50_ms", "p95_ms", "artifact",
"seconds"}; the exit code is 0 only if every step worked.
"""

from __future__ import annotations

import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[1]


def step(message: str) -> None:
    print(f"[demo] {message}", file=sys.stderr, flush=True)


def main() -> int:
    import requests

    from cifarprod.data import load_npz
    from cifarprod.serve import running

    start = time.perf_counter()
    data = ROOT / "data" / "cifar10_2000.npz"
    if not data.is_file():
        data = ROOT / "data" / "synthetic.npz"
        subprocess.run([sys.executable, str(ROOT / "scripts" / "make_synthetic.py"), str(data)], check=True,
                       stdout=subprocess.DEVNULL)
    step(f"data: {data.relative_to(ROOT)}")
    out = ROOT / "runs" / "demo"
    train = subprocess.run([sys.executable, "-m", "cifarprod", "train", "--config", str(ROOT / "configs" / "smoke.yaml"),
                            "--set", f"data={data}", "--out", str(out), "--runs", str(ROOT / "runs" / "runs.jsonl")],
                           cwd=ROOT, capture_output=True, text=True)
    if train.returncode != 0:
        step(f"training failed (exit code {train.returncode}): {train.stderr.strip()[-500:]}")
        return 1
    step(f"trained: {train.stdout.strip()}")
    image = load_npz(data, "val")[0][0]
    with running(out) as url:
        session = requests.Session()
        health = session.get(f"{url}/health", timeout=10)
        step(f"health: {health.status_code} {health.json().get('package_version')}")
        times, answer = [], None
        for _ in range(55):
            t0 = time.perf_counter()
            answer = session.post(f"{url}/predict", json={"pixels": image.tolist()}, timeout=10)
            times.append(time.perf_counter() - t0)
    if health.status_code != 200 or answer.status_code != 200:
        step(f"the API answered {health.status_code} and {answer.status_code}")
        return 1
    p50, p95 = np.percentile(1000 * np.array(times[5:]), [50, 95])
    body = answer.json()
    print(json.dumps({"status": answer.status_code, "label": body["label"], "probability": body["probability"],
                      "p50_ms": round(float(p50), 2), "p95_ms": round(float(p95), 2), "artifact": str(out),
                      "seconds": round(time.perf_counter() - start, 1)}))
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''

# Makefile
MAKEFILE = r'''# make install · make test · make demo (from a fresh clone to a served prediction)
PYTHON ?= python

install:
	$(PYTHON) -m pip install -e ".[dev,serve]"

test:
	$(PYTHON) -m pytest -q

demo:
	$(PYTHON) scripts/demo.py

.PHONY: install test demo
'''


# The solutions notebook commits its temporary project (your copy: do it yourself, in a terminal)
wbp.write_files(PROJECT, {"README.md": README_MD, "MODEL_CARD.md": MODEL_CARD_MD, "scripts/demo.py": DEMO_PY,
                          "Makefile": MAKEFILE})
for args_25 in (["git", "init", "-q"], ["git", "add", "-A"],
                ["git", "-c", "user.name=workbook", "-c", "user.email=workbook@example.org", "commit", "-q", "-m",
                 "cifar-prod: from the notebook to production"]):
    print(f"$ {' '.join(args_25[:2])} … → code {wbp.run(args_25, PROJECT, timeout=60).returncode}")


# The self-check of the challenge: your repository cloned afresh, installed, tested, then your demo, timed. No grade is
# computed here: grade your repository yourself with the grid of 05_solutions.md (10 criteria)
FILES_25 = {"README.md": README_MD, "MODEL_CARD.md": MODEL_CARD_MD, "scripts/demo.py": DEMO_PY}
if all(text is ... for text in (README_MD, MODEL_CARD_MD)):
    print("⏳ Ex B7.25 : écris d'abord README_MD, MODEL_CARD_MD et DEMO_PY (et fais B7.9 à B7.24).")
else:
    with checked("B7.25"):
        wbp.write_files(PROJECT, FILES_25)
        steps_25, start_25 = {}, time.perf_counter()
        speed_25 = wbp.machine_speed()
        t_25 = time.perf_counter()
        clone_25, how_25, uncommitted_25 = wbp.clone_project(PROJECT, scratch("clone") / "cifar-prod")
        steps_25["clone"] = (True, time.perf_counter() - t_25, how_25)
        if how_25 != "git clone":
            print("ℹ️ ton projet n'est pas un dépôt git : la cellule le copie (sans data/, runs/ ni caches) au lieu de le "
                  "cloner. Fais git init, git add et git commit dans son dossier pour que le clone ne voie que ce que tu as "
                  "commité, comme une autre machine.")
        elif uncommitted_25:
            print(f"ℹ️ {len(uncommitted_25)} fichier(s) modifié(s) ou nouveau(x) pas encore commité(s), absent(s) du clone : "
                  f"{', '.join(uncommitted_25[:6])}.")
        install_25 = wbp.install_editable(clone_25)
        try:
            steps_25["install"] = (bool(install_25.ok), install_25.seconds, install_25.method)
            tests_25 = demo_25 = None
            if install_25.ok:
                tests_25 = install_25.run(["python", "-m", "pytest", "-q", "-p", "no:cacheprovider", "--strict-markers",
                                           "-c", str(clone_25 / "pyproject.toml"), "--rootdir", str(clone_25)], clone_25,
                                          timeout=300)
                summary_25 = (tests_25.stdout.strip().splitlines() or ["(rien)"])[-1]
                steps_25["tests"] = (tests_25.ok and not tests_25.leftover, tests_25.seconds, summary_25)
                demo_25 = install_25.run(["python", "scripts/demo.py"], clone_25, timeout=300)
                steps_25["demo"] = (demo_25.ok and not demo_25.leftover, demo_25.seconds,
                                    "délai dépassé" if demo_25.timed_out else f"code {demo_25.returncode}")
        finally:
            install_25.cleanup()
        total_25 = time.perf_counter() - t_25
        equivalent_25 = total_25 * 0.2628 / speed_25
        for name_25, (good_25, seconds_25, what_25) in steps_25.items():
            print(f"{'✅' if good_25 else '❌'} {name_25:8s} {seconds_25:6.1f} s  ({what_25})")
        verdict("B7.25", equivalent_25 + 180 <= 600 and all(g for g, _, _ in steps_25.values()),
                f"de la copie à la prédiction servie : {fr(total_25, 0)} s ici, soit {fr(equivalent_25, 0)} s sur la "
                f"machine de référence du chapitre (cette machine va {fr(0.2628 / speed_25, 2)} fois sa vitesse) ; "
                f"avec ≈ {180 // 60} min pour télécharger PyTorch CPU sur une machine neuve, moins de "
                f"{600 // 60} min.",
                f"de la copie à la prédiction servie : {fr(total_25, 0)} s ici ({fr(equivalent_25, 0)} s sur la machine de "
                f"référence) ; attendu chaque étape réussie, en moins de {600 // 60} min avec ≈ {180 // 60} "
                "min de téléchargement de PyTorch CPU.")
        for what_25, done_25 in (("la démo", demo_25), ("les tests", tests_25)):
            if done_25 is not None and done_25.leftover:
                print(f"❌ Ex B7.25 : {what_25} laisse un processus en vie après sa fin (un serveur jamais arrêté ?), "
                      "arrêté ici : arrête-le dans un finally (with running(...) as url:).")
            elif done_25 is not None and done_25.timed_out:
                print(f"❌ Ex B7.25 : {what_25} ne rend pas la main en {fr(done_25.seconds, 0)} s : un serveur jamais "
                      "arrêté (un thread qui n'est pas daemon, serve_forever sans shutdown) ? Arrête-le dans un finally.")
        if demo_25 is not None and not demo_25.ok:
            print(f"   la démo : {demo_25.tail(4)[:400]}")
        if tests_25 is not None and not tests_25.ok:
            print("   les tests : " + " ⏎ ".join(l for l in tests_25.stdout.splitlines() if l.startswith(("FAILED", "ERROR")))[:400])
        summary_demo_25 = demo_25.last_json() if demo_25 is not None and demo_25.ok else None
        meta_25, fresh_25 = {}, False
        if isinstance(summary_demo_25, dict) and summary_demo_25.get("artifact"):
            art_25 = Path(str(summary_demo_25["artifact"]))
            art_25 = (clone_25 / art_25 if not art_25.is_absolute() else art_25).resolve()
            try:
                meta_25 = json.loads((art_25 / "meta.json").read_text(encoding="utf-8"))
                fresh_25 = (art_25.is_relative_to(clone_25.resolve())
                            and (art_25 / "meta.json").stat().st_mtime >= time.time() - demo_25.seconds - 5)
            except (OSError, ValueError):
                meta_25 = {}
            if not fresh_25:
                print("❌ Ex B7.25 : l'artefact du résumé de ta démo n'a pas été écrit par elle dans le clone : il doit "
                      "sortir de ton entraînement fumée, dans le dépôt cloné, pas d'un autre dossier.")
        if isinstance(summary_demo_25, dict) and "direct1_19" in globals():
            try:
                fast_25 = float(summary_demo_25.get("p50_ms")) < 0.5 * float(np.percentile(direct1_19, 50))
            except (TypeError, ValueError):
                fast_25 = True
            if fast_25:
                print(f"ℹ️ le p50 de ta démo ({summary_demo_25.get('p50_ms')} ms) est plus rapide que la moitié d'un appel "
                      "direct du modèle (B7.19) : ta latence est-elle vraiment mesurée sur des requêtes HTTP ?")
        undeclared_25 = wbp.undeclared_imports(clone_25)
        if undeclared_25:
            print(f"❌ Ex B7.25 : dépendances non déclarées : {', '.join(undeclared_25[:4])} : sur une machine neuve, "
                  "pip install -e \".[dev,serve]\" ne les installe pas (ici, l'environnement du notebook les a) ; "
                  "ajoute-les à l'extra dev de pyproject.toml (ou utilise urllib.request).")
        readme_25 = (clone_25 / "README.md").read_text(encoding="utf-8") if (clone_25 / "README.md").is_file() else ""
        card_25 = (clone_25 / "MODEL_CARD.md").read_text(encoding="utf-8") if (clone_25 / "MODEL_CARD.md").is_file() else ""
        ci_25 = (clone_25 / ".github/workflows/ci.yml")
        ci_ok_25 = ci_25.is_file() and all(r.ok for r in wbp.review_workflow(ci_25.read_text(encoding="utf-8"), clone_25))
        configs_25 = [p.name for p in (clone_25 / "configs").glob("*.yaml")] if (clone_25 / "configs").is_dir() else []
        evidence_25 = [
            ("installation sur un clone neuf", steps_25.get("install", (False,))[0] and how_25 == "git clone"
             and not undeclared_25, f"{how_25}, puis pip install -e dans un environnement neuf"
             + (f" ; non déclarées : {', '.join(undeclared_25[:3])}" if undeclared_25 else ", dépendances déclarées")),
            ("make ou script unique", steps_25.get("demo", (False,))[0] and fresh_25, "python scripts/demo.py, code 0"
             + (" ; un Makefile aussi" if (clone_25 / "Makefile").is_file() else "")),
            ("tests verts", steps_25.get("tests", (False,))[0], steps_25.get("tests", (0, 0, "pas lancés"))[2]),
            ("intégration continue", ci_ok_25, ".github/workflows/ci.yml, relu comme en B7.23"),
            ("configuration", {"fast.yaml", "smoke.yaml"} <= set(configs_25), f"configs/ : {', '.join(sorted(configs_25)) or 'vide'}"),
            ("artefact versionné", bool(fresh_25 and meta_25.get("package_version") and meta_25.get("weights_sha256")
                                         and re.fullmatch(r"[0-9a-f]{64}", str((meta_25.get("data") or {}).get("sha256")))),
             "meta.json : version du paquet, SHA-256 des poids et empreinte des données"),
            ("API qui répond", fresh_25 and isinstance(summary_demo_25, dict) and summary_demo_25.get("status") == 200
             and summary_demo_25.get("label") in CLASSES, f"la démo : {str(summary_demo_25)[:120]}"),
            ("latence mesurée", bool(re.search(r"p95\D{0,40}?(?<![\w.,])\d+(?:[.,]\d+)?(?:\s*(?:à|-|–|to)\s*\d+(?:[.,]\d+)?)?\s*ms|p95\s*\(ms\)\D{0,20}?(?<![\w.,])\d+(?:[.,]\d+)?", readme_25)),
             "le README donne un p95 en ms (un nombre mesuré)"),
            ("README", len(readme_25.split()) >= 150 and "pip install" in readme_25 and "demo" in readme_25,
             f"README.md : {len(readme_25.split())} mots, installation et démo citées"),
            ("fiche modèle", len(card_25.split()) >= 150 and all(w in card_25.lower() for w in ("usage", "données", "limite")),
             f"MODEL_CARD.md : {len(card_25.split())} mots, usage prévu, données et limites")]
        print("\nIndices pour ta grille (une aide, pas une note : chaque critère se juge en lisant ton dépôt) :")
        for k_25, (name_25, seen_25, what_25) in enumerate(evidence_25, start=1):
            print(f"  {k_25:2d}. {'✅' if seen_25 else '❌'} {name_25} — {what_25}")
        print(f"(auto-contrôle en {fr(time.perf_counter() - start_25, 0)} s)")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu transformer un notebook en paquet installable, avec une ligne de commande, une configuration validée, des runs journalisés et des tests qui attrapent de vraies erreurs ?
2. Sais-tu rapporter une accuracy sur plusieurs graines avec son intervalle, et dire ce qui rend deux runs identiques au bit près (ou non) ?
3. Sais-tu livrer un modèle dans un artefact qui se suffit, le servir par une API qui valide ses entrées, mesurer son p95 et vérifier un export ?
4. Sais-tu calculer un PSI et lire un test KS, dire ce qu'une alerte mérite, et écrire le workflow et le Dockerfile qui rendent ton dépôt reproductible par n'importe qui ?

**Pour aller plus loin** : l'article de Sculley et coll. (📄 B7.8) et le ML Test Score de Breck et coll., cités dans la fiche. La suite : le bonus B8, l'apprentissage par renforcement moderne.